# [1.2] Mechanistic Interpretability 입문: TransformerLens & induction circuits (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/02_[1.2]_Intro_to_Mech_Interp)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part2_intro_to_mech_interp/1.2_Intro_to_Mech_Interp_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part2_intro_to_mech_interp/1.2_Intro_to_Mech_Interp_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 본 장의 학습 내용에 관한 질문은 해당 전용 채널에서 해주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 연결되는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-12.png" width="350">

# 소개

이 페이지들은 Neel Nanda의 **TransformerLens** 라이브러리를 통해 mechanistic interpretability의 핵심 개념들을 소개하도록 설계되었습니다.

대부분의 섹션은 다음과 같은 방식으로 구성되어 있습니다:

1. TransformerLens의 특정 기능이 소개됩니다.
2. 해당 기능을 적용해야 하는 연습 문제가 제공됩니다.

연습 문제들의 공통 주제는 **induction circuits**입니다. Induction circuits는 transformer 내의 특정한 유형의 circuit으로, 기본적인 in-context learning을 수행할 수 있습니다. 계속 진행하시기 전에 [corresponding section of Neel's glossary](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=_Jzi6YHRHKP1JziwdE02qdYZ)을 읽어보시기 바랍니다. [LessWrong post](https://www.lesswrong.com/posts/TvrfY4c9eaGLeyDkE/induction-heads-illustrated) 또한 도움이 될 수 있으며, 여기에는 induction 메커니즘을 단계별로 설명하는 몇 가지 다이어그램(아래 그림과 같은)이 포함되어 있습니다.

각 연습 문제에는 5점 만점의 난이도와 중요도 등급, 그리고 해당 연습 문제에 소비해야 할 예상 최대 시간과 때로는 짧은 주석이 제공됩니다. 등급과 예상 시간은 상대적으로 해석하시기 바랍니다 (예: 예상 시간보다 약 50% 더 많은 시간을 쓰고 있다고 느껴진다면, 그에 맞춰 조정하십시오). 중요도가 낮아 수행할 가치가 없다고 느껴지거나, 더 핵심적인 내용으로 빠르게 넘어가고 싶다면 연습 문제를 건너뛰거나 솔루션을 확인하셔도 좋습니다!

본격적으로 내용을 학습하기 전, 전반적인 이해를 돕는 오늘의 강의를 듣고 싶으시다면 아래 영상을 시청하십시오:

<iframe width="540" height="304" src="https://www.youtube.com/embed/lfwT79Hgytc" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram.png" width="1000">

## 내용 및 학습 목표

### 1️⃣ TransformerLens: 소개

이 섹션은 TransformerLens 라이브러리에 빠르게 익숙해지도록 설계되었습니다. 모델을 로드하고 실행하는 방법을 배우며, 모든 모델에 공통적으로 적용되는 아키텍처 템플릿에 대해 학습합니다 (이전 연습 문제들을 이미 수행하셨다면, 많은 설계 원칙이 동일하게 적용되므로 익숙하실 것입니다).

> ##### 학습 목표
>
> - `HookedTransformer` 모델을 로드하고 실행합니다.
> - 이 모델들의 기본적인 아키텍처를 이해합니다.
> - 모델의 tokenizer를 사용하여 텍스트를 token으로 변환하고, 그 반대로 변환하는 방법을 익힙니다.
> - activation을 cache하는 방법과 cache에서 activation에 접근하는 방법을 익힙니다.
> - `circuitsvis`을 사용하여 attention head를 시각화합니다.

### 2️⃣ induction head 찾기

여기서는 induction head가 무엇인지, 어떻게 작동하며 왜 중요한지에 대해 배웁니다. 또한 모델 입력이 반복되는 시퀀스일 때, attention pattern에 나타나는 특징적인 induction head stripe를 통해 이를 식별하는 방법을 배웁니다.

> ##### 학습 목표
>
> - induction head가 무엇인지, 그리고 그것이 구현하는 알고리즘을 이해합니다.
> - activation pattern을 조사하여 기본적인 attention head pattern을 식별하고, attention head를 자동으로 감지하는 함수를 직접 작성합니다.
> - 반복되는 랜덤 시퀀스에서 생성된 attention pattern을 살펴보고 induction head를 식별합니다.

### 3️⃣ TransformerLens: Hooks

다음으로, 모델 내부의 activation에 접근하고 개입할 수 있게 해주는 TransformerLens의 강력한 기능인 hook에 대해 배웁니다. 주로 hook의 기초와 이를 이용해 activation에 접근하는 방법에 집중할 것입니다 (인과적 개입은 이후의 IOI 연습 문제에서 주로 다룹니다). 또한 모델 내에서 logit attribution을 수행하는 도구를 구축하여, 특정 작업에 대한 모델 성능에 어떤 컴포넌트가 기여하는지 식별해 봅니다.

> ##### 학습 목표
>
> - hook이 무엇인지, 그리고 TransformerLens에서 어떻게 사용되는지 이해합니다.
> - hook을 사용하여 activation에 접근하고, 결과를 처리하며, 이를 외부 tensor에 기록합니다.
> - attribution을 수행하는 도구를 구축합니다. 즉, 주어진 작업의 성능에 모델의 어떤 컴포넌트가 책임이 있는지 감지합니다.
> - **ablation**과 같은 기본적인 개입을 수행하기 위해 hook을 어떻게 사용할 수 있는지 이해합니다.

### 4️⃣ induction circuit 역공학 (Reverse-engineering)

마지막으로, transformer의 weight를 직접 살펴봄으로써 circuit을 역공학하는 방법을 배웁니다 (이는 해석 가능성의 "gold standard"라고 할 수 있으며, 모든 상황에서 가능한 것은 아닙니다). 행렬 곱셈을 통해 QK 및 OV circuit을 조사하고, `FactoredMatrix` 클래스가 이러한 행렬 분석을 얼마나 쉽게 만드는지 배웁니다. 또한 두 induction head 사이의 composition 증거를 찾고, 이를 통해 형성된 전체 circuit의 기능을 조사합니다.

> ##### 학습 목표
>
> - activation pattern을 통해 circuit을 조사하는 것과 weight를 직접 살펴보고 circuit을 역공학하는 것의 차이를 이해합니다.
> - factored matrix 클래스를 사용하여 induction circuit 내의 QK 및 OV circuit을 조사합니다.
> - induction circuit에 대해 추가 탐색을 수행합니다: composition score 및 targeted ablation.

## 설정 코드

In [ ]:
import os
import sys
from pathlib import Path

from importlib.metadata import packages_distributions

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
if "transformer-lens" not in packages_distributions():
    %pip install transformer_lens==2.17.0 einops eindex-callum jaxtyping git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/ARENA_3.0-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import functools
import sys
from pathlib import Path
from typing import Callable

import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
from eindex import eindex
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utils,
)
from transformer_lens.hook_points import HookPoint

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part2_intro_to_mech_interp"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part2_intro_to_mech_interp.tests as tests
from plotly_utils import (
    hist,
    imshow,
    plot_comp_scores,
    plot_logit_attribution,
    plot_loss_difference,
)

# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

# 1️⃣ TransformerLens: 소개

> ##### 학습 목표
>
> - `HookedTransformer` 모델을 로드하고 실행합니다
> - 이러한 모델들의 기본 architecture를 이해합니다
> - 모델의 tokenizer를 사용하여 텍스트를 token으로 변환하고, 그 반대로 변환하는 방법을 익힙니다
> - activation을 cache하는 방법과 cache에서 activation에 접근하는 방법을 알아봅니다
> - attention head를 시각화하기 위해 `circuitsvis`을 사용합니다

## Introduction

*참고 - 이 내용의 대부분은 Neel Nanda의 관점에서 작성되었습니다.*

이 노트북은 [TransformerLens](https://github.com/neelnanda-io/TransformerLens)를 위한 데모 노트북입니다. **이 라이브러리는 제가 ([Neel Nanda](neelnanda.io)) GPT-2 스타일의 언어 모델에 대한 [mechanistic interpretability](https://distill.pub/2020/circuits/zoom-in/)를 수행하기 위해 작성한 것입니다.** mechanistic interpretability의 목표는 훈련된 모델을 가져와서, 모델이 훈련 중에 가중치로부터 학습한 알고리즘을 역공학(reverse engineer)하는 것입니다. 오늘날 우리는 기본적으로 인간 수준으로 영어를 말할 수 있는 컴퓨터 프로그램(GPT-3, PaLM 등)을 가지고 있지만, 그것들이 어떻게 작동하는지, 혹은 우리가 직접 어떻게 작성해야 하는지에 대해서는 전혀 알지 못한다는 것이 현재의 사실입니다. 저는 이 점이 매우 불만족스러우며, 이를 해결하고 싶습니다! Mechanistic interpretability는 매우 초기 단계의 작은 분야이며, 해결되지 않은 문제가 *매우* 많습니다. 도움을 주고 싶으시다면, 하나를 선택해 도전해 보시기 바랍니다! **어디서부터 시작해야 할지 확인하시려면 저의 [list of concrete open problems](https://docs.google.com/document/d/1WONBzNqfKIxERejrrPlQMyKqg7jSFW92x5UMXNrMdPo/edit#)를 확인해 주세요.**

제가 이 라이브러리를 작성한 이유는 Anthropic의 interpretability 팀을 떠나 독립적인 연구를 시작한 후, 오픈 소스 툴링의 상태에 매우 좌절했기 때문입니다. 모델을 *사용*하거나 *훈련*시키기 위한 HuggingFace나 DeepSpeed 같은 훌륭한 인프라는 많지만, 모델의 내부를 파헤치고 어떻게 작동하는지 역공학하기 위한 도구는 거의 없습니다. **이 라이브러리는 그 문제를 해결하고**, 실제 인프라를 갖춘 산업체 조직에서 일하지 않더라도 이 분야에 쉽게 입문할 수 있도록 만들기 위해 노력했습니다! 핵심 기능들은 [Anthropic's excellent Garcon tool](https://transformer-circuits.pub/2021/garcon/index.html)에서 많은 영감을 받았습니다. Garcon을 구축하고 탐색적 연구를 가속화하는 좋은 인프라의 가치를 보여준 Nelson Elhage와 Chris Olah에게 감사를 표합니다!

제가 따른 핵심 설계 원칙은 탐색적 분석을 가능하게 하는 것입니다. 일반적인 ML과 비교했을 때 mechanistic interpretability의 가장 즐거운 부분 중 하나는 피드백 루프가 매우 짧다는 점입니다! 이 라이브러리의 목적은 실험 아이디어를 떠올리고 결과를 확인하기까지의 간극을 최대한 줄여서, **연구가 놀이처럼 느껴지게** 하고 몰입 상태(flow state)에 쉽게 진입할 수 있도록 만드는 것입니다. 이 노트북은 라이브러리가 어떻게 작동하고 어떻게 사용하는지를 보여주지만, 탐색적 연구에 얼마나 효과적인지 확인하고 싶으시다면 [my notebook analysing Indirect Objection Identification](https://github.com/neelnanda-io/TransformerLens/blob/main/Exploratory_Analysis_Demo.ipynb) 또는 [my recording of myself doing research](https://www.youtube.com/watch?v=yo4QvDn-vsU)를 확인해 보세요!

## 모델 로드 및 실행

TransformerLens에는 40개 이상의 오픈 소스 GPT 스타일 모델이 포함되어 있습니다. `HookedTransformer.from_pretrained(MODEL_NAME)`을 통해 이 모델들 중 어느 것이든 로드할 수 있습니다. 이 데모 노트북에서는 80M 파라미터 모델인 GPT-2 Small을 살펴보겠습니다. 나머지 모델에 대한 정보는 Available Models 섹션을 참조하시기 바랍니다.

In [ ]:
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

### HookedTransformerConfig

또는, config 객체를 정의한 다음 `HookedTransformer.from_config(cfg)`을 호출하여 모델을 정의할 수 있습니다. 이는 모델의 architecture를 더 세밀하게 제어하고 싶을 때 특히 유용합니다. 다음 섹션에서 induction head를 연구하기 위해 attention-only 모델을 정의할 때 이 방식의 예시를 살펴보겠습니다.

모델을 이런 방식으로 정의하지 않더라도, 모델의 `cfg` attribute를 통해 config 객체에 접근할 수 있습니다.

### 연습 문제 - 모델 조사하기

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> ```

GPT-2 Small 모델에 대해 `gpt2_small.cfg`을 사용하여 다음 사항들을 찾아보세요:

* layer 수
* layer당 head 수
* 최대 context window

일부 항목은 문서 페이지를 확인해야 할 수도 있습니다. VSCode를 사용 중이라면 `HookedTransformerConfig`를 우클릭하고 "Go to definition"을 선택하여 확인할 수 있습니다. Colab을 사용 중이라면 [GitHub page](https://github.com/neelnanda-io/TransformerLens)을 읽어보시기 바랍니다.

<details>
<summary>정답</summary>

config 객체의 다음 파라미터들이 정답을 제공합니다:

```
cfg.n_layers == 12
cfg.n_heads == 12
cfg.n_ctx == 1024
```

</details>

### 모델 실행하기

모델은 단일 문자열 또는 token 텐서(shape: `[batch, position]`, 모두 정수)로 실행할 수 있습니다. 가능한 반환 타입은 다음과 같습니다:

* `"logits"` (shape `[batch, position, d_vocab]`, 실수),
* `"loss"` (다음 token을 예측할 때의 cross-entropy loss),
* `"both"` (`(logits, loss)`의 튜플)
* `None` (모델을 실행하지만 logit은 계산하지 않습니다 - 중간 activation만 사용하려는 경우 더 빠릅니다)

In [ ]:
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model out, let's find the loss on this paragraph!"""

loss = gpt2_small(model_description_text, return_type="loss")
print("Model loss:", loss)

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Model loss: tensor(4.3443, device='cuda:0')</pre>

## Transformer architecture

HookedTransformer는 약간 수정된 GPT-2 architecture이지만, 계산적으로는 동일합니다. 가장 중요한 변경 사항은 attention head의 내부 구조입니다:

* residual stream을 query, key, value로 매핑하는 가중치 `W_K`, `W_Q`, `W_V`는 하나의 큰 결합된 행렬이 아니라 3개의 별도 행렬입니다.
* 가중치 행렬 `W_K`, `W_Q`, `W_V`, `W_O` 및 activation은 하나의 큰 축으로 평탄화하는 대신, 별도의 `head_index` 및 `d_head` 축을 가집니다.
    * 모든 activation은 `[batch, position, head_index, d_head]` 형태를 가집니다.
    * `W_K`, `W_Q`, `W_V`는 `[head_index, d_model, d_head]` 형태이며, `W_O`는 `[head_index, d_head, d_model]` 형태입니다.
* **중요 - 우리는 일반적으로 가중치 행렬이 왼쪽이 아닌 오른쪽에서 곱해지는 관례를 따릅니다.** 다시 말해, 행렬의 형태는 `[input, output]`이며, `new_activation = old_activation @ weights + bias`와 같이 계산합니다.
    * 이 방식이 직관적이지 않게 느껴진다면, 아래 드롭다운을 클릭하여 예시를 확인하십시오.

<details>
<summary>우리 모델의 행렬 곱셈 예시</summary>

* **Query matrices**
    * 특정 layer와 head에 대한 각 query matrix `W_Q`는 `[d_model, d_head]` 형태를 가집니다.
    * 따라서 residual stream의 벡터 `x`의 길이가 `d_model`라면, 그에 대응하는 query 벡터는 `x @ W_Q`이며, 길이는 `d_head`입니다.
* **Embedding matrix**
    * embedding matrix `W_E`는 `[d_vocab, d_model]` 형태를 가집니다.
    * 따라서 `A`가 특정 token에 대응하는 길이 `d_vocab`의 one-hot-encoded 벡터라면, 이 token의 embedding 벡터는 `A @ W_E`이며, 길이는 `d_model`입니다.

</details>

실제 코드는 TransformerLens의 다양한 모델 제품군과 일관성을 유지하기 위한 여러 Boolean flag들이 있어 다소 복잡합니다. 코드와 내부 구조를 이해하시려면 대신 [CleanTransformerDemo](https://colab.research.google.com/github/neelnanda-io/TransformerLens/blob/clean-transformer-demo/Clean_Transformer_Demo.ipynb)에 있는 코드를 읽으시는 것을 추천합니다.

### Parameters와 Activations

모델에서 parameters와 activations를 구분하는 것이 중요합니다.

* **Parameters**는 학습 과정에서 학습되는 weights와 biases입니다.
    * 모델의 input이 변경되어도 이 값들은 변하지 않습니다.
    * 모델에서 직접 접근할 수 있으며, 예를 들어 embedding matrix의 경우 `model.W_E` 입니다.
* **Activations**는 forward pass 동안 계산되는 일시적인 숫자들로, input의 함수입니다.
    * 이러한 값들은 단 한 번의 forward pass 동안만 존재하고 그 이후에는 사라진다고 생각할 수 있습니다.
    * hook을 사용하여 forward pass 중에 이 값들에 접근할 수 있지만(hook에 대해서는 나중에 더 자세히 다룹니다), 특정 input의 맥락 밖에서 모델의 activations에 대해 이야기하는 것은 의미가 없습니다.
    * Attention scores와 patterns는 activations입니다 (이들은 다른 activation과 matrix multiplication에 사용되기 때문에 약간 직관적이지 않을 수 있습니다).

아래 링크는 bias가 없는 attention-only 모델의 단일 layer(`TransformerBlock`라고 불립니다) 다이어그램을 보여줍니다. 각 박스는 **activation**에 해당하며, 해당 activation에 접근하기 위해 나중에 사용할 hook point의 이름도 함께 알려줍니다. 각 박스 아래의 빨간색 텍스트는 activation의 shape를 나타냅니다 (batch dimension은 제외합니다). 각 화살표는 activation에 대한 연산에 해당하며, **parameters**가 포함된 경우 화살표 위에 표시되어 있습니다.

[Link to diagram](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/small-merm.svg)

다음 링크는 모든 기능(biases, layernorms, MLPs 포함)이 포함된 `TransformerBlock`의 다이어그램입니다. 처음에는 이 모든 내용이 이해되지 않더라도 걱정하지 마십시오. 나중에 일부 세부 사항으로 다시 돌아올 것입니다. 이러한 transformer들을 다루다 보면 그 architecture에 더 익숙해질 것입니다.

[Link to diagram](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/full-merm.svg)

이 모델들을 사용할 때 작업을 더 편리하게 만들어 줄 몇 가지 단축 방법입니다:

* `model.blocks[0].attn.W_Q` 등을 통해 모델에서 직접 `W_Q`와 같이 weight를 인덱싱할 수 있습니다 (예를 들어, 이는 layer 0에 있는 모든 head의 `[nheads, d_model, d_head]` query weight를 제공합니다).
    * 하지만 더 쉬운 방법은 `model.W_Q`으로 인덱싱하는 것이며, 이는 모델의 **모든** query weight를 포함하는 `[nlayers, nheads, d_model, d_head]` tensor를 제공합니다.
* 마찬가지로 embedding, unembedding, positional embedding에 대해서는 각각 `model.W_E`, `model.W_U`, `model.W_pos`라는 단축 방법이 존재합니다.
* MLP layer가 포함된 모델의 경우, linear layer를 위한 `model.W_in`와 `model.W_out`도 사용할 수 있습니다.
* 모든 bias에 대해서도 동일하게 적용됩니다 (예: 모든 query bias를 위한 `model.b_Q`).

## Tokenization

tokenizer는 모델 내부에 저장되어 있으며, `model.tokenizer`을 사용하여 접근할 수 있습니다. 또한 내부적으로 tokenizer를 호출하는 몇 가지 헬퍼 메서드들이 있습니다. 예를 들어:

* `model.to_str_tokens(text)`은 문자열을 token-as-strings 리스트로 변환합니다 (또는 문자열 리스트를 token-as-strings의 리스트의 리스트로 변환합니다).
* `model.to_tokens(text)`는 문자열을 token 텐서로 변환합니다.
* `model.to_string(tokens)`은 token 텐서를 문자열로 변환합니다.

사용 예시:

In [ ]:
print(gpt2_small.to_str_tokens("gpt2"))
print(gpt2_small.to_str_tokens(["gpt2", "gpt2"]))
print(gpt2_small.to_tokens("gpt2"))
print(gpt2_small.to_string([50256, 70, 457, 17]))

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">['<|endoftext|>', 'g', 'pt', '2']
[['<|endoftext|>', 'g', 'pt', '2'], ['<|endoftext|>', 'g', 'pt', '2']]
tensor([[50256,    70,   457,    17]], device='cuda:0')
<|endoftext|>gpt2</pre>

<details>
<summary>참고 - <code><|endoftext|></code></summary>

위에서 눈치채셨을 수도 있는 이상한 점은 `to_tokens`와 `to_str_tokens`가 각 prompt의 시작 부분에 이상한 `<|endoftext|>`를 추가했다는 것입니다. 우리는 이전 연습 문제 세트에서 이를 접했으며, 이것이 **Beginning of Sequence (BOS)** token(GPT-2의 경우 EOS 및 PAD token과 동일하며 인덱스는 `50256`입니다)이라는 점을 확인했습니다.

TransformerLens는 기본적으로 이 token을 추가하며, 이는 새로운 사용자들을 쉽게 혼란스럽게 만들 수 있습니다. 특히, **여기에는** `model.forward`(예를 들어 `model("Hello World")`를 수행할 때 암시적으로 사용되는 것)이 **포함됩니다**. `to_tokens`, `to_str_tokens`, `model.forward` 및 문자열을 multi-token tensor로 변환하는 다른 모든 함수에서 `prepend_bos=False` flag를 설정하여 이 동작을 비활성화할 수 있습니다.

`prepend_bos`는 약간의 편법이며, 여기서 올바른 기본값이 무엇인지에 대해 저도 고민이 많았습니다. 제가 이렇게 하는 이유는 transformer가 첫 번째 token을 이상하게 처리하는 경향이 있기 때문입니다. 이는 (모든 입력이 1000개 이상의 token인) 학습 단계에서는 크게 중요하지 않지만, 짧은 prompt를 조사할 때는 큰 문제가 될 수 있습니다! 그 이유는 attention pattern이 확률 분포이므로 합계가 1이 되어야 하며, 따라서 "꺼짐" 상태를 시뮬레이션하기 위해 보통 첫 번째 token을 바라보기 때문입니다. BOS token을 제공하면 head들이 그것을 바라봄으로써 쉴 수 있게 되어, 첫 번째 "실제" token의 정보가 보존됩니다.

더 나아가, *일부* 모델들은 BOS token이 필요하도록 학습되었습니다 (OPT와 제가 만든 interpretability-friendly 모델들이 그렇고, GPT-2와 GPT-Neo는 그렇지 않습니다). 하지만 GPT-2가 이렇게 학습되지 않았음에도 불구하고, 경험적으로는 이것이 interpretability를 더 쉽게 만드는 것으로 보입니다.

</details>

### 연습 문제 - 모델이 몇 개의 token을 정확하게 예측했나요?

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

위에서 모델에 입력한 `model_description_text`을 생각해 보십시오. 모델이 몇 개의 token을 정확하게 예측했습니까? 어떤 token들이 정답이었습니까?

In [ ]:
logits: Tensor = gpt2_small(model_description_text, return_type="logits")
prediction = logits.argmax(dim=-1).squeeze()[:-1]

# YOUR CODE HERE - get the model's prediction on the text

<details>
<summary>힌트</summary>

`return_type="logits"`를 사용하여 모델의 예측값을 얻은 다음, vocab 차원에 대해 argmax를 취하십시오. 그 후, 이 예측값들을 `model_description_text`에서 유도된 실제 token들과 비교하십시오.

이 예측 텐서의 `[:-1]`번째 요소들을 입력 token들의 `[1:]`번째 요소들과 비교해야 한다는 점을 기억하십시오 (모델의 출력은 현재 token이 아니라 *다음* token에 대한 확률 분포를 나타내기 때문입니다).

또한, batch 차원을 처리해야 한다는 점을 기억하십시오 (`logits`와 `to_tokens`의 출력 모두 기본적으로 batch 차원을 가지기 때문입니다).

</details>

<details>
<summary>정답 - 확인해야 할 내용</summary>

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Model accuracy: 33/111
Correct tokens: ['\n', '\n', 'former', ' with', ' models', '.', ' can', ' of', 'ooked', 'Trans', 'former', '_', 'NAME', '`.', ' model', ' the', 'Trans', 'former', ' to', ' be', ' and', '-', '.', '\n', '\n', ' at', 'PT', '-', ',', ' model', ',', "'s", ' the']
</pre>

따라서 모델이 111개의 token 중 33개를 맞혔습니다. 나쁘지 않은 결과입니다!

</details>


<details><summary>풀이</summary>

```python
logits: Tensor = gpt2_small(model_description_text, return_type="logits")
prediction = logits.argmax(dim=-1).squeeze()[:-1]

true_tokens = gpt2_small.to_tokens(model_description_text).squeeze()[1:]
is_correct = prediction == true_tokens

print(f"Model accuracy: {is_correct.sum()}/{len(true_tokens)}")
print(f"Correct tokens: {gpt2_small.to_str_tokens(prediction[is_correct])}")
```
</details>

**Induction heads**는 앞으로의 실습에서 더 자세히 살펴볼 특별한 종류의 attention head입니다. 이를 통해 모델은 특정 형태의 in-context learning을 수행할 수 있습니다. 즉, 토큰 `B`이 토큰 `A` 다음에 온다는 하나의 관찰 결과로부터 일반화하여, 설령 이 두 토큰이 모델의 학습 데이터에서 함께 나타난 적이 없더라도, 향후 `A`가 나타날 때 `A` 다음에 토큰 `B`가 올 것이라고 예측하는 것입니다.

**이 텍스트에서 induction heads가 작동하고 있다는 증거를 찾을 수 있습니까?**

<details>
<summary>induction heads의 증거</summary>

induction heads의 증거는 모델이 토큰 `'H'` 다음에 `'ooked', 'Trans', 'former'`을 성공적으로 예측했다는 사실에서 알 수 있습니다. 이는 `HookedTransformer`이 이 텍스트 문자열에서 두 번째로 나타난 시점이었으며, 모델이 첫 번째가 아닌 두 번째에 이를 예측했기 때문입니다. (모델이 첫 번째에 `former`을 예측하기는 했지만, `Transformer`는 모델이 학습 과정에서 이미 접했던 단어라고 합리적으로 가정할 수 있으므로, `HookedTransformer`과 달리 이 예측에는 induction 능력이 필요하지 않았을 것입니다.)

```python
print(gpt2_small.to_str_tokens("HookedTransformer", prepend_bos=False))     # --> ['H', 'ooked', 'Trans', 'former']
```
</details>

## 모든 Activation 캐싱하기

mechanistic interpretability를 수행할 때의 첫 번째 기본 작업은 모델이라는 블랙박스를 열어 모델의 모든 내부 activation을 살펴보는 것입니다. 이는 `logits, cache = model.run_with_cache(tokens)`을 통해 가능합니다. GPT-2 논문의 첫 번째 문장을 사용하여 이를 시도해 보겠습니다.

<details>
<summary>여담 - <code>remove_batch_dim</code>에 관한 참고 사항</summary>

모델 내부의 모든 activation은 batch 차원으로 시작합니다. 여기서는 단일 batch 차원만 입력했기 때문에, 해당 차원의 길이는 항상 1이며 다소 번거롭습니다. 따라서 `remove_batch_dim=True` 키워드를 전달하면 이 차원이 제거됩니다.

`gpt2_cache_no_batch_dim = gpt2_cache.remove_batch_dim()`을 사용해도 동일한 효과를 얻을 수 있습니다.
</details>

In [ ]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_tokens, remove_batch_dim=True)

print(type(gpt2_logits), type(gpt2_cache))

`gpt2_cache` 객체를 살펴보면, 모델의 서로 다른 activation에 각각 대응하는 매우 많은 수의 key들이 포함되어 있음을 확인할 수 있습니다. cache를 직접 인덱싱하거나, 더 편리한 인덱싱 단축 표기법을 사용하여 key에 접근할 수 있습니다. 예를 들어, layer 0의 attention pattern을 추출하는 두 가지 방법은 다음과 같습니다:

In [ ]:
attn_patterns_from_shorthand = gpt2_cache["pattern", 0]
attn_patterns_from_full_name = gpt2_cache["blocks.0.attn.hook_pattern"]

t.testing.assert_close(attn_patterns_from_shorthand, attn_patterns_from_full_name)

<details>
<summary>참고: <code>utils.get_act_name</code></summary>

이 둘이 동일한 이유는 내부적으로 첫 번째 예시가 실제로 `utils.get_act_name("pattern", 0)`로 인덱싱하며, 이는 `"blocks.0.attn.hook_pattern"`으로 평가되기 때문입니다.

일반적으로 `utils.get_act_name`은 activation의 짧은 이름과 레이어 번호가 주어졌을 때, 해당 activation의 전체 이름을 가져오는 데 유용한 함수입니다.

**Transformer Architecture** 섹션의 다이어그램을 사용하여 activation 이름을 찾는 데 도움을 받을 수 있습니다.
</details>

### 연습 문제 - activation 검증하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> 
> If you're already comfortable implementing things like attention calculations (e.g. having gone through Neel's transformer walkthrough) you can skip this exercise. However, it might serve as a useful refresher.
> ```

`hook_q`, `hook_k` 그리고 `hook_pattern`이 다이어그램에서 암시된 방식으로 서로 연관되어 있는지 검증하십시오. 이를 위해 `layer0_pattern_from_cache` (cache에서 직접 가져온 layer 0의 attention pattern)와 `layer0_pattern_from_q_and_k` (`hook_q` 및 `hook_k`로부터 계산된 layer 0의 attention pattern)를 계산하여 확인하십시오. attention pattern은 확률값이므로, 적절하게 scale 및 softmax를 적용해야 함을 기억하십시오.

In [ ]:
layer0_pattern_from_cache = gpt2_cache["pattern", 0]

# YOUR CODE HERE - define `layer0_pattern_from_q_and_k` manually, by manually performing the
# steps of the attention calculation (dot product, masking, scaling, softmax)
t.testing.assert_close(layer0_pattern_from_cache, layer0_pattern_from_q_and_k)
print("Tests passed!")

<details>
<summary>힌트</summary>

총 세 가지의 서로 다른 cache index를 사용해야 합니다:

* attention pattern을 가져오기 위한 `gpt2_cache["pattern", 0]` (shape는 `[nhead, seqQ, seqK]` 입니다)
* query 벡터를 가져오기 위한 `gpt2_cache["q", 0]` (shape는 `[seqQ, nhead, headsize]` 입니다)
* key 벡터를 가져오기 위한 `gpt2_cache["k", 0]` (shape는 `[seqK, nhead, headsize]` 입니다)

</details>


<details><summary>솔루션</summary>

```python
layer0_pattern_from_cache = gpt2_cache["pattern", 0]

q, k = gpt2_cache["q", 0], gpt2_cache["k", 0]
seq, nhead, headsize = q.shape
layer0_attn_scores = einops.einsum(q, k, "seqQ n h, seqK n h -> n seqQ seqK")
mask = t.triu(t.ones((seq, seq), dtype=t.bool), diagonal=1).to(device)
layer0_attn_scores.masked_fill_(mask, -1e9)
layer0_pattern_from_q_and_k = (layer0_attn_scores / headsize**0.5).softmax(-1)
```
</details>

## Attention Head 시각화하기

Mathematical Frameworks 논문의 핵심 통찰은 본질적으로 해석 가능한 모델의 부분들, 즉 input token, output logit, 그리고 attention pattern을 해석하는 데 집중해야 한다는 것입니다. 그 외의 모든 것들(residual stream, key, query, value 등)은 의미 있는 결과물을 계산하는 과정에서 생성되는 압축된 중간 상태들입니다. 따라서 자연스러운 시작점은 다양한 텍스트에 대한 attention pattern을 통해 head를 분류하는 것입니다.

interpretability 연구를 할 때는 요약 통계량을 내는 것보다 데이터를 시각화하는 것부터 시작하는 것이 항상 좋습니다. 요약 통계량은 매우 오해의 소지가 있을 수 있기 때문입니다! 하지만 이제 attention pattern을 시각화했으므로, 기본적인 요약 통계량을 생성하고 시각화 결과를 통해 이를 검증할 수 있습니다! (따라서 웹 개발이나 데이터 시각화 능력이 뛰어난 것은 놀라울 정도로 유용한 기술 스택이 됩니다! Neural network는 매우 고차원적인 객체이기 때문입니다.)

[Alan Cooney's CircuitsVis library](https://github.com/alan-cooney/CircuitsVis) (Anthropic의 PySvelte 라이브러리 기반)를 사용하여 layer 0에 있는 모든 head의 attention pattern을 시각화해 보겠습니다. 이전 연습 문제들을 수행하셨다면 이 라이브러리를 이미 보셨을 것입니다.

우리는 두 가지 중요한 인자를 받는 `cv.attention.attention_patterns` 함수를 사용할 것입니다:

* `attention`: shape이 `[n_heads, seq_len, seq_len]`인 attention head pattern입니다. 이는 각 head에 대한 attention 확률의 스택 그리드로 구성됩니다. 즉, `attention[head, d, s]`는 attention head `head`에서 destination position `d`가 source position `s`를 바라보는 attention 확률입니다.
* `tokens`: token 리스트이며, `attention`의 `seq_len` 차원과 길이가 같아야 합니다. 실수로 dummy dimension이 포함된 리스트를 전달하거나, BOS token으로 인해 `seq_len`과 길이가 달라지지 않도록 주의하십시오!

이 시각화는 인터랙티브합니다! token이나 head 위에 마우스를 올리거나, 클릭하여 고정해 보십시오. 왼쪽 상단과 각 head에 표시되는 그리드는 destination position과 source position으로 구성된 attention pattern 그리드입니다. GPT-2는 **causal attention**을 사용하므로 하삼각 행렬(lower triangular) 형태를 띱니다. 즉, attention은 뒤쪽만 바라볼 수 있으며, 정보는 네트워크에서 앞으로만 이동할 수 있습니다.

> 참고 - 데이터를 다른 방식으로 보여주는 `cv.attention.attention_heads` 함수를 사용할 수도 있습니다 (구문은 `attention_patterns`과 완전히 동일합니다). VSCode에서 이를 표시할 경우 메인 플롯의 크기가 계속해서 줄어드는 버그가 발생할 수 있습니다. 이 경우 HTML로 저장한 뒤(즉, `html = cv.attention.attention_heads(...); with open("attn_heads.html", "w") as f: f.write(str(html))` 사용) 브라우저에서 플롯을 여시기 바랍니다.

In [ ]:
print(type(gpt2_cache))
attention_pattern = gpt2_cache["pattern", 0]
print(attention_pattern.shape)
gpt2_str_tokens = gpt2_small.to_str_tokens(gpt2_text)

print("Layer 0 Head Attention Patterns:")
display(
    cv.attention.attention_patterns(
        tokens=gpt2_str_tokens,
        attention=attention_pattern,
        attention_head_names=[f"L0H{i}" for i in range(12)],
    )
)

head 위에 마우스를 올리면 attention pattern을 볼 수 있으며, head를 클릭하면 고정할 수 있습니다. 각 token 위에 마우스를 올리면 해당 token이 어떤 다른 token들에 attention 하는지(또는 어떤 다른 token들이 해당 token에 attention 하는지 - 이는 드롭다운을 `Destination <- Source`(으)로 변경하여 확인할 수 있습니다) 볼 수 있습니다.

<details>
<summary>Other circuitsvis functions - neuron activations</summary>

`circuitsvis` 라이브러리에는 **neuron activations**를 위한 몇 가지 멋진 시각화 도구들이 있습니다. 여기에 몇 가지 더 소개합니다 (지금 모두 이해하실 필요는 없으며, 나중에 다시 확인하셔도 됩니다).

아래 함수는 neuron activations를 시각화합니다. 예시에서는 하나의 sequence만 보여주지만, 여러 개의 sequence를 보여줄 수도 있습니다 (`tokens`가 문자열 리스트의 리스트이고, `activations`가 tensor 리스트인 경우).

```python
neuron_activations_for_all_layers = t.stack([
    gpt2_cache["post", layer] for layer in range(gpt2_small.cfg.n_layers)
], dim=1)
# shape = (seq_pos, layers, neurons)

cv.activations.text_neuron_activations(
    tokens=gpt2_str_tokens,
    activations=neuron_activations_for_all_layers
)
```

다음 함수는 각 neuron이 어떤 단어에서 가장 많이/적게 활성화되는지를 보여줍니다 (정상적으로 작동하려면 다소 특이한 indexing이 필요하다는 점에 유의하십시오).

```python
neuron_activations_for_all_layers_rearranged = utils.to_numpy(einops.rearrange(neuron_activations_for_all_layers, "seq layers neurons -> 1 layers seq neurons"))

cv.topk_tokens.topk_tokens(
    # Some weird indexing required here ¯\_(ツ)_/¯
    tokens=[gpt2_str_tokens],
    activations=neuron_activations_for_all_layers_rearranged,
    max_k=7,
    first_dimension_name="Layer",
    third_dimension_name="Neuron",
    first_dimension_labels=list(range(12))
)
```
</details>

# 2️⃣ induction heads 찾기

> ##### 학습 목표
>
> - induction heads가 무엇인지, 그리고 그것들이 구현하는 알고리즘이 무엇인지 이해합니다.
> - activation 패턴을 조사하여 기본적인 attention head 패턴을 식별하고, attention heads를 자동으로 감지하는 함수를 직접 작성합니다.
> - 반복되는 랜덤 시퀀스에서 생성된 attention 패턴을 살펴봄으로써 induction heads를 식별합니다.

## Toy Attention-Only 모델 소개

여기서는 오늘을 위해 특별히 학습된 toy 2L attention-only transformer를 소개합니다. 해석을 더 쉽게 만들기 위해 몇 가지 변경 사항을 적용했습니다:
- attention 블록만 가지고 있습니다.
- positional embedding은 token embedding과 달리, attention 레이어에서 각 key와 query 벡터를 계산하기 직전에만 residual stream에 추가됩니다. 즉, query는 `Q = (resid + pos_embed) @ W_Q + b_Q`로 계산하고 key도 마찬가지로 계산하지만, value는 `V = resid @ W_V + b_V`로 계산합니다. 이는 **residual stream이 positional 정보를 직접적으로 인코딩할 수 없음**을 의미합니다.
    - 이렇게 하면 induction head가 형성되기가 *훨씬* 쉬워지며, 2-3배 더 빠르게 형성됩니다 [see the comparison of two training runs](https://wandb.ai/mechanistic-interpretability/attn-only/reports/loss_ewma-22-08-24-11-08-83---VmlldzoyNTI0MDMz?accessToken=8ap8ir6y072uqa4f9uinotdtrwmoa8d8k2je4ec0lyasf1jcm3mtdh37ouijgdbm). (각 곡선의 솟아오른 부분은 induction head의 형성을 나타냅니다.)
    - 이를 수행하는 아래의 인자는 `positional_embedding_type="shortformer"` 입니다.
- MLP 레이어, LayerNorm, bias가 없습니다.
- 별도의 embed 및 unembed 행렬이 존재합니다 (즉, weight가 tied 되어 있지 않습니다).

이제 `HookedTransformerConfig` 객체로 모델을 정의합니다. 이는 이전 연습 세트에서 사용했던 `Config` 객체와 유사하지만, 훨씬 더 많은 기능을 가지고 있습니다. 각 인자가 어떤 역할을 하는지 확인하려면 문서 페이지를 참조하시기 바랍니다 (VSCode에서 우클릭 후 "Go to Definition" 선택).

In [ ]:
cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,  # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer",
)

지난 섹션에서는 tokenizer를 명시적으로 정의하고 이를 모델에 전달해야 했습니다. 하지만 여기서는 tokenizer 이름만 전달하면 모델이 자동으로 tokenizer를 생성합니다 (내부적으로 `AutoTokenizer.from_pretrained(tokenizer_name)`을 호출합니다).

아래에서는 HuggingFace에서 state dict를 다운로드하는 일부 상용구 코드를 사용하여 weight를 로드합니다 (HuggingFace에 직접 업로드한 모든 모델에 대해 이 작업을 수행할 수 있습니다):

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

마지막으로, 모델을 생성하고 weight를 로드하겠습니다:

In [ ]:
model = HookedTransformer(cfg)
pretrained_weights = t.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

관련 hook 이름을 기억하기 위해 [diagram at this link](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/small-merm.svg)을(를) 사용하십시오.

### 연습 문제 - attention pattern 시각화 및 조사

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> 
> It's important to be comfortable using circuitsvis, and the cache object.
> ```

*이 연습 문제는 매우 빠르게 끝낼 수 있습니다. 이전 섹션의 코드를 재사용하면 됩니다. 5~10분 후에도 여전히 막혀 있다면 솔루션을 확인하시기 바랍니다.*

다음 prompt에 대해 모델의 두 layer 모두에 대한 attention pattern을 시각화하십시오:

In [ ]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

*(참고로, 이전의 cache를 생성할 때처럼 token 단위가 아니라 문자열 `text`에 대해 모델을 실행했습니다. 이는 `HookedTransformer` 덕분에 가능한 작업입니다.)*

attention pattern을 살펴보십시오. attention head들에 대해 무엇을 발견하셨습니까?

여러 head에서 공통적으로 나타나는 세 가지의 비교적 뚜렷한 기본 패턴을 발견하실 수 있을 것입니다. 이 패턴들은 무엇이며, 왜 이러한 패턴들이 존재하는지 추측해 보시겠습니까?

In [ ]:
# YOUR CODE HERE - visualize attention

<details>
<summary>참고 - 플롯이 표시되지 않을 때 해결 방법</summary>

흔히 하는 실수는 token을 인자로 전달하지 않는 것입니다. 이렇게 하면 attention pattern이 렌더링되지 않습니다.

만약 이것이 문제가 아니라면, Circuitsvis 라이브러리의 문제일 수 있습니다. 인라인으로 플롯하는 대신, 출력을 `str(...)`으로 변환한 다음 HTML 파일로 저장하여 브라우저에서 다운로드하고 열 수 있습니다.

</details>

<details>
<summary>결과 논의 </summary>

매우 빈번하게 반복되는 세 가지 기본 패턴이 있음을 알 수 있습니다:

* 주로 이전 token에 attention을 기울이는 `prev_token_heads` (예: head `0.7`)
* 주로 현재 token에 attention을 기울이는 `current_token_heads` (예: head `1.6`)
* 주로 첫 번째 token에 attention을 기울이는 `first_token_heads` (예: head `0.3` 또는 `1.4`, 다만 이들은 다른 두 패턴보다 약간 덜 명확합니다)

`prev_token_heads`와 `current_token_heads`은 아마 놀랍지 않을 것입니다. 시퀀스에서 서로 가까이 있는 단어들은 아마 훨씬 더 많은 상호 정보량을 가지고 있기 때문입니다 (즉, bigram 또는 trigram 예측만으로도 꽤 많은 것을 얻을 수 있습니다).

`first_token_heads`은 조금 더 놀랍습니다. 여기서 기본적인 직관은 시퀀스의 첫 번째 token이 가끔만 활성화되는 head들을 위한 휴식 또는 null 위치로 자주 사용된다는 점입니다 (attention 확률의 합은 항상 1이 되어야 하기 때문입니다).
</details>


<details><summary>솔루션</summary>

```python
str_tokens = model.to_str_tokens(text)
for layer in range(model.cfg.n_layers):
    attention_pattern = cache["pattern", layer]
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern))
```
</details>

이제 세 가지 기본적인 attention 패턴을 관찰했으므로, 해당 패턴들을 위한 detector를 만들 차례입니다!

### 연습 문제 - 직접 detector 작성하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than 10-25 minutes on these exercises.
> These exercises aren't meant to be too challenging, just to get you thinking about how to characterize head behaviour. 
> Use the hints if you're stuck.
> ```

특정 유형의 head를 찾아내는 detector 역할을 하는 아래 함수들을 완성해야 합니다. 이 결과들을 위의 시각적 attention pattern과 비교하여 detector를 검증하십시오. 요약 통계치만으로는 부정확할 수 있지만, 데이터를 직접 다루며 검증한다면 훨씬 더 신뢰할 수 있습니다.

이러한 작업은 모델이 무엇을 하고 있는지에 대한 관찰이나 직관을 정량적 측정치로 변환할 수 있어야 하기 때문에 유용합니다. 연습이 진행됨에 따라, 훨씬 더 흥미로운 도구와 detector들을 만들어 볼 것입니다!

참고 - 어떤 head가 어떤 작업을 수행하는지, 그리고 어떤 detector가 이를 포착할 수 있는지에 대해 객관적으로 정답이 정해져 있지는 않습니다. 여러분이 찾고자 하는 동작 유형을 식별할 수 있는, 그럴듯해 보이는 방법을 고안해 보십시오. **완벽한 해결책을 찾으려 너무 많은 시간을 소비하지 마시고, attention pattern을 시각적으로 검토한 결과와 대략적으로 일치하는 해결책을 찾으십시오.**

In [ ]:
def current_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
    """
    raise NotImplementedError()


def prev_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
    """
    raise NotImplementedError()


def first_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    """
    raise NotImplementedError()


print("Heads attending to current token  = ", ", ".join(current_attn_detector(cache)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(cache)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(cache)))

<details>
<summary>힌트</summary>

관련 token들에 대한 평균 attention 확률을 계산해 보십시오. 예를 들어, 적절한 `offset` 파라미터와 함께 `t.diagonal`를 사용하여 대각선 바로 아래의 token들을 가져올 수 있습니다:

```python
>>> arr = t.arange(9).reshape(3, 3)
>>> arr
tensor([[0, 1, 2],
        [3, 4, 5],
        [6, 7, 8]])

>>> arr.diagonal()
tensor([0, 4, 8])

>>> arr.diagonal(-1)
tensor([3, 7])
```

특정 layer의 모든 attention 확률을 가져오기 위해 `cache["pattern", layer]`를 사용해야 하며, 그 후 0번째 차원을 인덱싱하여 올바른 head를 가져와야 함을 기억하십시오.
</details>

<details>
<summary>예상 출력 (방법에 따라 약간 다를 수 있습니다)</summary>

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Heads attending to current token  =  0.9
Heads attending to previous token =  0.7
Heads attending to first token    =  0.3, 1.4, 1.10
</pre>

</details>

<details>
<summary>솔루션 (가능한 한 가지 방법)</summary>

참고 - 아래 코드에서 임계값으로 `score=0.4`를 선택한 것은 다소 임의적이지만, 충분히 잘 작동하는 것으로 보입니다. 이 특정 사례에서 `0.5`의 임계값을 사용하면 어떤 head도 current-token head로 분류되지 않습니다.

```python
def current_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of diagonal elements
            score = attention_pattern.diagonal().mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def prev_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of sub-diagonal elements
            score = attention_pattern.diagonal(-1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def first_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of 0th elements
            score = attention_pattern[:, 0].mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads
```

</details>

위의 attention 시각화 결과와 출력값을 비교해 보십시오. 결과가 타당해 보입니까? 보너스 연습 문제로, 다른 텍스트를 입력하여 결과가 얼마나 안정적인지 확인해 보십시오. 특정 head들이 항상 동일하게 분류됩니까?

이제 induction head에 주목해 볼 시간입니다.

## induction head란 무엇인가요?

(참고: 저는 두 번째 layer에서 '현재 token의 복사본 바로 다음에 오는 token'에 attention을 주는 head를 induction **head**라고 부르며, layer 0의 **previous token head**와 layer 1의 **induction head**의 조합으로 구성된 circuit을 induction **circuit**이라고 부릅니다.)

[Induction heads](https://transformer-circuits.pub/2021/framework/index.html#induction-heads)는 transformer에서 볼 수 있는 첫 번째 정교한 circuit입니다! 그리고 이는 [another paper just about them](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html)을 작성했을 정도로 충분히 흥미로운 주제입니다.

<details>
<summary>induction head가 왜 중요한지에 대한 여담</summary>

induction head에는 특히 놀라운 몇 가지 점이 있습니다:

* 이들은 phase change를 통해 상당히 갑작스럽게 발달합니다. 약 2B에서 4B token 사이에서 induction head가 없는 상태에서 상당히 잘 발달된 상태로 변합니다. 이는 1L 모델 [see the comparison of in context learning performance curves for models with different layers](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html#:~:text=Our%20first%20observation)과는 확연히 다른 모습이며, 훨씬 더 큰 모델(예: 13B 모델)에서도 관찰될 수 있습니다.
    * phase change는 alignment 관점에서 특히 흥미로우면서도 (우울한) 부분입니다. 급격한 방향 전환이나 기만(deception), 상황 인식(situational awareness)과 같은 emergent capabilities의 가능성은 alignment가 더 어려워질 수 있는 세상처럼 보이며, 우리는 경고 신호나 우리의 기술을 테스트할 수 있는 더 단순하지만 유사한 모델 없이 갑작스럽게 당할 수 있기 때문입니다.
* 이들은 상당한 loss 감소를 책임집니다. 이들이 발달할 때 loss curve에 눈에 띄는 굴곡이 생길 정도입니다 (이 loss의 변화는 모델 크기의 대폭적인 증가로 인한 loss 증가와 꽤 비슷할 수 있지만, 정확히 일대일로 비교하기는 어렵습니다).
* 이들은 in-context learning의 대부분을 담당하는 것으로 보입니다. 즉, context의 아주 멀리 있는 token들을 사용하여 다음 token을 예측하는 능력입니다. 이는 transformer가 RNN이나 LSTM 같은 이전 architecture보다 뛰어난 성능을 보이는 중요한 방식이며, induction head가 이 부분에서 큰 역할을 하는 것으로 보입니다.
* 동일한 핵심 circuit이 번역이나 few-shot learning과 같은 더 정교한 설정에서도 사용되는 것으로 보입니다. 이러한 작업들을 명확히 담당하면서 동시에 induction head의 역할도 수행하는 head들이 존재합니다.

</details>

계속 진행하시기 전에(또는 [this LessWrong post](https://www.lesswrong.com/posts/TvrfY4c9eaGLeyDkE/induction-heads-illustrated) 전에) [corresponding section of the glossary](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=_Jzi6YHRHKP1JziwdE02qdYZ)를 읽어보실 것을 강력히 권장합니다. 짧게 요약하자면, induction circuit은 layer 0의 previous token head와 layer 1의 induction head로 구성됩니다. 여기서 induction head는 previous token head와의 K-Composition을 통해 현재 token의 복사본 바로 *다음*에 오는 token에 attention을 주는 법을 배웁니다.

##### 질문 - 왜 1L 모델에서는 induction head가 형성될 수 없었을까요?

<details>
<summary>답변</summary>

왜냐하면 이를 위해서는 *그 앞 token의 값*에 기반하여 key position에 attention을 주는 head가 필요하기 때문입니다. attention score는 단순히 key token과 query token의 함수일 뿐, 다른 token의 함수가 아닙니다.

(softmax 때문에 attention pattern에 다른 token의 영향이 *실제로* 포함되기는 합니다. 만약 다른 key token이 높은 attention score를 가지면, softmax가 이 쌍을 억제합니다. 하지만 이 억제는 모든 position에 대해 대칭적이므로, 관련 token *다음*의 token을 체계적으로 선호할 수는 없습니다.)

중요한 세부 사항은 인접한 token들의 값이 (대략적으로) 서로 무관하다는 점입니다. 만약 모델이 상대적인 *position*에 기반하여 attention을 주고 싶었다면 이는 쉬운 일이었을 것입니다.
</details>

## induction 능력 확인하기

induction head를 가진 모델의 놀라운 점은, 무작위 token이 반복되는 시퀀스가 주어졌을 때 시퀀스의 반복되는 후반부를 예측할 수 있다는 것입니다. 이는 학습 데이터와 전혀 다르기 때문에 매우 놀라운 일입니다! 이러한 분포 외(out of distribution) 일반화 능력을 예측하는 것은 여러분이 circuit을 실제로 이해했다는 강력한 증거가 됩니다.

이 모델에 induction head가 있는지 확인하기 위해, 정확히 해당 테스트를 실행하고 두 부분의 성능을 비교해 보겠습니다. token당 loss에서 뚜렷한 차이를 확인할 수 있을 것입니다.

참고 - 결과가 매우 명확하게 나타나며 시각화가 쉽기 때문에, 여기서는 짧은 시퀀스(그리고 단 하나의 시퀀스)를 사용합니다. 실제로는 더 미묘한 작업에 대해 당연히 더 큰 시퀀스를 사용합니다. 하지만 작은 작업에서 반복하고 디버깅하는 것이 가장 효율적일 때가 많습니다.

### 연습 문제 - 반복되는 시퀀스에 대한 토큰별 loss 플롯하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than 10-15 minutes on these exercises.
> ```

아래 함수들을 완성해야 합니다. 첫 번째 함수의 첫 번째 줄에는 prefix를 정의하는 코드가 제공되어 있습니다 (GPT-2는 BOS token이 있도록 학습되었으므로, BOS token이 필요함을 기억하십시오). 또한 이전 연습 문제 세트에서 작성한 `get_log_probs` 함수도 제공해 드렸습니다.

In [ ]:
def generate_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Generates a sequence of repeated random tokens

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
    """
    t.manual_seed(0)  # for reproducibility
    prefix = (t.ones(batch_size, 1) * model.tokenizer.bos_token_id).long()


def run_and_cache_model_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> tuple[Tensor, Tensor, ActivationCache]:
    """
    Generates a sequence of repeated random tokens, and runs the model on it, returning (tokens,
    logits, cache). This function should use the `generate_repeated_tokens` function above.

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
        rep_logits: [batch_size, 1+2*seq_len, d_vocab]
        rep_cache: The cache of the model run on rep_tokens
    """
    raise NotImplementedError()


def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    logprobs = logits.log_softmax(dim=-1)
    # We want to get logprobs[b, s, tokens[b, s+1]], in eindex syntax this looks like:
    correct_logprobs = eindex(logprobs, tokens, "b s [b s+1]")
    return correct_logprobs


seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_cache) = run_and_cache_model_repeated_tokens(model, seq_len, batch_size)
rep_cache.remove_batch_dim()
rep_str = model.to_str_tokens(rep_tokens)
model.reset_hooks()
log_probs = get_log_probs(rep_logits, rep_tokens).squeeze()

print(f"Performance on the first half: {log_probs[:seq_len].mean():.3f}")
print(f"Performance on the second half: {log_probs[seq_len:].mean():.3f}")

plot_loss_difference(log_probs, rep_str, seq_len)

<details>
<summary>힌트</summary>

`t.randint(low, high, shape)`를 사용하여 반복되는 token의 앞부분을 정의할 수 있습니다. 또한 `dtype=t.long`를 지정하는 것을 잊지 마십시오.

그 다음 `t.concat`를 사용하여 prefix와 두 번 복사된 반복 token들을 서로 연결할 수 있습니다.
</details>


<details><summary>솔루션</summary>

```python
def generate_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Generates a sequence of repeated random tokens

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
    """
    t.manual_seed(0)  # for reproducibility
    prefix = (t.ones(batch_size, 1) * model.tokenizer.bos_token_id).long()
    rep_tokens_half = t.randint(0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.int64)
    rep_tokens = t.cat([prefix, rep_tokens_half, rep_tokens_half], dim=-1).to(device)
    return rep_tokens


def run_and_cache_model_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> tuple[Tensor, Tensor, ActivationCache]:
    """
    Generates a sequence of repeated random tokens, and runs the model on it, returning (tokens,
    logits, cache). This function should use the `generate_repeated_tokens` function above.

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
        rep_logits: [batch_size, 1+2*seq_len, d_vocab]
        rep_cache: The cache of the model run on rep_tokens
    """
    rep_tokens = generate_repeated_tokens(model, seq_len, batch_size)
    rep_logits, rep_cache = model.run_with_cache(rep_tokens)
    return rep_tokens, rep_logits, rep_cache
```
</details>

### Induction Attention 패턴 찾기

다음으로 확인해야 할 자연스러운 대상은 induction attention 패턴입니다.

먼저, 앞서 작성한 attention 패턴 시각화 코드(즉, `cv.attention.attention_heads` 또는 `attention_patterns`)로 돌아가서 두 번째 layer에 있을 법한 head들을 수동으로 확인해 보십시오. 어떤 head가 induction head 역할을 하고 있다고 생각하십니까?

참고 - 위에서 `rep_str` 객체를 정의해 두었으므로, 이를 `circuitsvis` 함수에서 사용할 수 있습니다.

In [ ]:
# YOUR CODE HERE - display the attention patterns stored in `rep_cache`, for each layer

<details>
<summary>몇 가지 관찰 사항</summary>

induction head의 특징적인 패턴은 대각선 줄무늬이며, 대각선 오프셋은 `seq_len-1` 입니다 (대상 token이 대상 token의 이전 출현 *이후*의 token에 attention을 기울이기 때문입니다).

head 4와 10은 강한 induction 성향을 보이고, head 6은 매우 약한 induction 성향을 보이며, 나머지는 그렇지 않다는 것을 알 수 있습니다.

</details>


<details><summary>솔루션</summary>

```python
for layer in range(model.cfg.n_layers):
    attention_pattern = rep_cache["pattern", layer]
    display(cv.attention.attention_patterns(tokens=rep_str, attention=attention_pattern))
```
</details>

### 연습 문제 - induction-head 탐지기 만들기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than 5-15 minutes on this exercise.
> This exercise should be very similar to the earlier detector exercises (the only difference being how you index attention).
> ```

이제 offset diagonal에 가해지는 평균 attention을 찾는 induction pattern score 함수를 만들어야 합니다. 이전의 head scorer들과 동일한 스타일로 작성하되, 특성적인 attention head 패턴을 감지하는 데 적합한 다른 방식의 indexing을 사용하시기 바랍니다.

In [ ]:
def induction_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be induction heads

    Remember - the tokens used to generate rep_cache are (bos_token, *rand_tokens, *rand_tokens)
    """
    raise NotImplementedError()


print("Induction heads = ", ", ".join(induction_attn_detector(rep_cache)))

<details>
<summary>도움말 - 어떤 offset을 사용해야 할지 모르겠습니다.</summary>

diagonal의 offset은 `-(seq_len-1)` (여기서 `seq_len`는 두 번 반복되는 random token의 길이입니다)이어야 합니다. 왜냐하면 두 번째 random token `T`는 첫 번째 `T` **이후**의 token에 attend하기 때문입니다.
</details>


<details><summary>정답</summary>

```python
def induction_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be induction heads

    Remember - the tokens used to generate rep_cache are (bos_token, *rand_tokens, *rand_tokens)
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of (-seq_len+1)-offset elements
            seq_len = (attention_pattern.shape[-1] - 1) // 2
            score = attention_pattern.diagonal(-seq_len + 1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads
```
</details>

이 함수가 예상대로 작동한다면, `circuitsvis`에서 관찰한 내용과 일치하는 출력을 확인하실 수 있습니다 (즉, induction head라고 관찰했던 head들이 여기에서 작성한 함수에 의해 induction head로 분류되어야 합니다).

# 3️⃣ TransformerLens: Hooks

> ##### 학습 목표
>
> - hook이 무엇인지, 그리고 TransformerLens에서 어떻게 사용되는지 이해합니다.
> - hook을 사용하여 activation에 접근하고, 결과를 처리하며, 이를 외부 tensor에 기록하는 방법을 익힙니다.
> - attribution을 수행하는 도구를 구축합니다. 즉, 모델의 어떤 컴포넌트가 주어진 태스크의 성능에 기여하는지 탐지합니다.
> - **ablation**과 같은 기본적인 intervention을 수행하기 위해 hook을 어떻게 사용할 수 있는지 이해합니다.

## hook란 무엇인가요?

신경망을 해석할 때 가장 좋은 점 중 하나는 우리가 시스템을 *완벽하게 제어*할 수 있다는 것입니다. 계산적인 관점에서 우리는 내부에서 어떤 연산이 일어나고 있는지 정확히 알고 있습니다 (비록 그것들이 무엇을 의미하는지는 모를지라도 말입니다!). 그리고 우리는 정밀하고 외과적인 수정을 가하여 모델의 동작과 다른 내부 요소들이 어떻게 변하는지 확인할 수 있습니다. 이는 매우 강력한 도구이며, 예를 들어 세심한 counterfactuals와 causal intervention을 설정하여 모델의 동작을 쉽게 이해할 수 있게 해줍니다.

따라서 이러한 작업을 수행할 수 있는 능력은 매우 핵심적인 연산이며, 이것이 TransformerLens가 지원하는 주요 기능 중 하나입니다! 여기서 핵심 기능은 **hook points**입니다. transformer 내부의 모든 activation은 hook point로 둘러싸여 있으며, 이를 통해 우리는 해당 activation을 수정하거나 개입할 수 있습니다.

우리는 해당 activation에 **hook function**을 추가한 다음 `model.run_with_hooks`를 호출함으로써 이 작업을 수행합니다.

*(용어 참고 - 우리 모델의 거의 모든 activation에는 연관된 hook point가 있기 때문에, 때때로 "hook"과 "activation"이라는 용어를 혼용하여 사용하겠습니다.)*

### Hook 함수

Hook 함수는 `activation_value`와 `hook_point`라는 두 개의 인자를 받습니다. `activation_value`는 `ActivationCache`의 값들과 마찬가지로 모델 내의 특정 activation을 나타내는 tensor입니다. `hook_point`는 함수 내부에서 호출할 때 유용한 `hook.layer()`와 같은 메서드나 `hook.name`과 같은 속성을 제공하는 객체입니다.

만약 activation을 수정하기 위해 hook을 사용한다면, hook 함수는 activation 값과 동일한 shape의 tensor를 반환해야 합니다. 하지만 hook 함수가 activation에 접근하여 일부 처리를 수행하고 그 결과를 외부 변수에 기록하게 할 수도 있습니다 (이 경우 hook 함수는 아무것도 반환하지 않아야 합니다).

특정 layer의 attention pattern을 변경하기 위한 hook 함수의 예시는 다음과 같습니다:

```python
def hook_function(
    attn_pattern: Float[Tensor, "batch heads seq_len seq_len"],
    hook: HookPoint
) -> Float[Tensor, "batch heads seq_len seq_len"]:

    # modify attn_pattern (can be inplace)
    return attn_pattern
```

### hook을 사용하여 실행하기

hook 함수(또는 함수들)를 정의했다면, `model.run_with_hooks`를 호출해야 합니다. 이 함수에 대한 일반적인 호출 방식은 다음과 같습니다:

```python
loss = model.run_with_hooks(
    tokens,
    return_type="loss",
    fwd_hooks=[
        ('blocks.1.attn.hook_pattern', hook_function)
    ]
)
```

이 코드를 자세히 살펴보겠습니다.

* `tokens`은 모델의 입력을 나타냅니다.
* `return_type="loss"`은 activation을 수정하고 이것이 loss에 어떤 영향을 미치는지 확인하기 위해 여기서 사용됩니다.
    * logit을 반환하거나, 출력에는 관심이 없고 중간 activation에만 접근하고 싶은 경우에는 `return_type=None`를 사용할 수도 있습니다.
* `fwd_hooks`은 (hook 이름, hook 함수)로 이루어진 2-튜플의 리스트입니다.
    * hook 이름은 어떤 activation을 hook하고 싶은지 지정하는 문자열입니다.
    * hook 함수는 해당 activation을 첫 번째 인자로 받아 실행됩니다.

### hook에 대한 추가 설명

hook의 기능을 최대한 활용하기 위한 몇 가지 추가 참고 사항입니다. 원하신다면 [jump ahead](#hooks-accessing-activations)를 통해 hook이 실제로 사용되는 예시를 먼저 확인하고, 나중에 이 섹션으로 돌아오셔도 됩니다.

<details>
<summary>hook 초기화하기</summary>

`model.run_with_hooks`은 기본 파라미터로 `reset_hooks_end=True`를 가지며, 이는 실행 종료 시 모든 hook(실행 전과 실행 중에 추가된 것 모두 포함)을 초기화합니다. 그럼에도 불구하고, 예를 들어 hook 중 하나에 오류가 있어 함수가 끝까지 실행되지 않는 경우처럼 hook으로 인해 문제가 발생할 수 있습니다. 이 경우 `model.reset_hooks()`을 사용하여 모든 hook을 초기화할 수 있습니다.

hook을 초기화하고 싶지 않은 경우(즉, forward pass 사이에도 hook을 유지하고 싶은 경우), `run_with_hooks` 함수에서 `reset_hooks_end=False`을 설정하거나, forward pass 전에 `add_hook` 메서드를 사용하여 hook을 직접 추가하면 됩니다(이렇게 하면 자동으로 초기화되지 않습니다).

</details>
<details>
<summary>여러 개의 hook을 한 번에 추가하기</summary>

`fwd_hooks` 리스트에 여러 개의 튜플을 포함하는 것이 여러 hook을 추가하는 한 가지 방법입니다:

```python
loss = model.run_with_hooks(
    tokens,
    return_type="loss",
    fwd_hooks=[
        ('blocks.0.attn.hook_pattern', hook_function),
        ('blocks.1.attn.hook_pattern', hook_function)
    ]
)
```

또 다른 방법은 단일 이름 대신 **이름 필터(name filter)**를 사용하는 것입니다:

```python
loss = model.run_with_hooks(
    tokens,
    return_type="loss",
    fwd_hooks=[
        (lambda name: name.endswith("pattern"), hook_function)
    ]
)
```
</details>
<details>
<summary><code>utils.get_act_name</code></summary>

이전 섹션에서 cache를 인덱싱할 때, `cache['blocks.0.attn.hook_pattern']`과 같은 문자열을 사용하거나 `cache['pattern', 0]`과 같은 약칭을 사용할 수 있음을 확인했습니다. 두 번째 방법이 작동하는 이유는 내부적으로 `utils.get_act_name` 함수를 호출하기 때문입니다. 즉, 다음과 같이 작동합니다:

```python
utils.get_act_name('pattern', 0) == 'blocks.0.attn.hook_pattern'
```

forward hook에서 `utils.get_act_name`을 사용하는 것이 전체 문자열을 사용하는 것보다 훨씬 쉬운 경우가 많습니다. 기억해야 할 것은 activation 이름뿐이기 때문입니다(이를 위해 이전 섹션의 다이어그램을 다시 참조할 수 있습니다).
</details>
<details>
<summary><code>functools.partial</code>을 사용하여 hook의 변형 생성하기</summary>

유용한 팁 중 하나는 필요한 것보다 더 많은 인자를 가진 hook 함수를 정의한 다음, `functools.partial`을 사용하여 추가 인자를 채우는 것입니다. 예를 들어, 특정 head만 수정하는 hook 함수를 원하지만, 이를 모든 head에 대해 개별적으로 실행하고 싶은 경우(모든 hook을 한 번에 추가하여 다음 forward pass에서 모두 실행하게 하는 대신), 다음과 같이 할 수 있습니다:

```python
def hook_all_attention_patterns(
    attn_pattern: Float[Tensor, "batch heads seq_len seq_len"],
    hook: HookPoint,
    head_idx: int
) -> Float[Tensor, "batch heads seq_len seq_len"]:
    # modify attn_pattern inplace, at head_idx
    return attn_pattern

for head_idx in range(12):
    temp_hook_fn = functools.partial(hook_all_attention_patterns, head_idx=head_idx)
    model.run_with_hooks(tokens, fwd_hooks=[('blocks.1.attn.hook_pattern', temp_hook_fn)])
```
</details>

그리고 이해하는 데 필수적이지는 않지만, 흥미로운 몇 가지 점들이 있습니다:

<details>
<summary>PyTorch hook과의 관계</summary>

[PyTorch hooks](https://blog.paperspace.com/pytorch-hooks-gradient-clipping-debugging/)은 훌륭하고 과소평가되었지만, 동시에 매우 조잡한 기능입니다. 이는 layer에 작용하여 해당 layer의 입력이나 출력을 수정하거나, autodiff를 적용할 때 gradient를 수정할 수 있습니다. 핵심적인 차이점은 **Hook points**는 layer가 아니라 *activation*에 작용한다는 점입니다. 이는 layer 내부의 각 activation에 개입할 수 있음을 의미하며, transformer의 정확한 layer 구조에 신경 쓸 필요가 없다는 뜻입니다. 또한 hook의 효과가 정확히 어떻게 적용되는지 즉각적으로 알 수 있습니다. 이 조정은 [Garcon's use of ProbePoints](https://transformer-circuits.pub/2021/garcon/index.html)에서 뻔뻔하게 영감을 받았습니다.

또한 다양한 편의 기능들이 제공됩니다. PyTorch의 hook은 전역 상태(global state)이므로, 실수로 모델에 hook을 남겨두면 매우 골치 아플 수 있습니다. TransformerLens의 hook 역시 전역 상태이지만, `run_with_hooks`은 함수 종료 시 모든 hook을 제거함으로써 이를 지역 상태(local state)처럼 다루는 추상화를 제공하려 노력합니다(또한 모든 hook을 제거하는 유용한 `model.reset_hooks()` 메서드가 함께 제공됩니다).
</details>

<details>
<summary>TransformerLens hook은 실제로 어떻게 구현되어 있습니까?</summary>

이들은 forward 메서드로 identity 함수를 가진 모듈로 구현되어 있습니다:

```python
class HookPoint(nn.Module):
    ...
    def forward(self, x):
        return x
```

동시에 hook 함수를 추가하고 제거하기 위한 특별한 기능들도 갖추고 있습니다. HookedTransformer 모델을 출력할 때 hook이 보이는 이유는 모든 모듈이 재귀적으로 출력되기 때문입니다.

모델을 정상적으로 실행할 때, hook 모듈은 모델의 동작을 변경하지 않습니다(identity 함수를 적용하는 것은 아무런 영향을 주지 않기 때문입니다). hook 모듈에 함수를 추가했을 때만(예: hook 모듈로 들어오는 모든 입력을 제거하는 함수), 모델의 동작이 변경됩니다.

</details>

## Hooks: Activation에 접근하기

이후 섹션에서는 hook에 개입하는 코드를 작성할 예정이며, 이는 interpretability에서 hook을 매우 유용하게 만드는 핵심 기능입니다. 하지만 지금은 값을 변경하지 않고 activation에 접근하는 방법만 살펴보겠습니다. 이는 hook 함수가 전역 변수에 값을 기록하고, (activation을 직접 수정하는 대신) 아무것도 반환하지 않도록 함으로써 구현할 수 있습니다.

왜 이런 방식이 필요할까요? 다음과 같은 작업에 유용합니다:

* 특정 태스크에 대한 activation 추출
* 많은 입력값에 대해 시간이 오래 걸리는 계산 수행 (예: 특정 neuron을 가장 많이 활성화하는 텍스트 찾기)

이론적으로는 이전 섹션에서 사용한 `run_with_cache` 함수와 cache 결과의 후처리를 결합하여 이 모든 것을 수행할 수 있습니다. 하지만 hook을 사용하는 것이 더 직관적이고 메모리 효율적일 수 있습니다.

### 연습 문제 - hook을 사용하여 induction score 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 15-20 minutes on this exercise.
> This is our first exercise with hooks, which are an absolutely vital TransformerLens tool. Use the hints if you're stuck.
> ```

우선, 이전 섹션(cache의 값들에 대해 induction head detector 함수를 실행했던 곳)과 동일한 결과를 얻기 위해 hook을 어떻게 사용할 수 있는지 살펴보겠습니다.

대부분의 코드는 아래에 이미 제공되어 있습니다. 여러분이 해야 할 유일한 일은 **`induction_score_hook` 함수를 구현하는 것**입니다. 언급했듯이, 이 함수는 두 개의 인자를 받습니다: activation 값(이 경우 attention pattern이 됩니다)과 hook 객체(함수 내에서 접근 가능한 유용한 메서드와 속성을 제공합니다. 예를 들어, layer를 반환하는 `hook.layer()`나 cache의 이름과 동일한 이름을 반환하는 `hook.name` 등이 있습니다).

작성할 함수는 다음을 수행해야 합니다:

* 이전 섹션에서 induction head detector를 작성할 때 사용했던 것과 동일한 방법론을 사용하여, attention pattern `pattern`에 대한 induction score를 계산합니다.
    * 이번에는 batch 차원이 1보다 크므로, batch 차원에 대해 평균 attention score를 계산해야 한다는 점에 유의하십시오.
    * 또한, 한 번에 하나씩이 아니라 모든 head에 대한 induction score를 동시에 계산하고 있다는 점에 유의하십시오. `torch.diagonal` 함수의 인자인 `dim1`와 `dim2`이 유용할 수 있습니다.
* 이 score를 제공된 전역 변수인 텐서 `induction_score_store`에 기록합니다. 이 텐서의 `[i, j]`번째 요소는 `i`번째 layer의 `j`번째 head에 대한 induction score여야 합니다.

In [ ]:
seq_len = 50
batch_size = 10
rep_tokens_10 = generate_repeated_tokens(model, seq_len, batch_size)

# We make a tensor to store the induction score for each head.
# We put it on the model's device to avoid needing to move things between the GPU and CPU,
# which can be slow.
induction_score_store = t.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)


def induction_score_hook(pattern: Float[Tensor, "batch head_index dest_pos source_pos"], hook: HookPoint):
    """
    Calculates the induction score, and stores it in the [layer, head] position of the
    `induction_score_store` tensor.
    """
    raise NotImplementedError()


# We make a boolean filter on activation names, that's true only on attention pattern names
pattern_hook_names_filter = lambda name: name.endswith("pattern")

# Run with hooks (this is where we write to the `induction_score_store` tensor`)
model.run_with_hooks(
    rep_tokens_10,
    return_type=None,  # For efficiency, we don't need to calculate the logits
    fwd_hooks=[(pattern_hook_names_filter, induction_score_hook)],
)

# Plot the induction scores for each head in each layer
imshow(
    induction_score_store,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head",
    text_auto=".2f",
    width=900,
    height=350,
)

<details>
<summary>도움말 - 이 함수를 어떻게 구현해야 할지 모르겠습니다.</summary>

induction stripe를 얻으려면 다음을 사용할 수 있습니다:

```python
torch.diagonal(pattern, dim1=-2, dim2=-1, offset=1-seq_len)
```

이는 batch의 모든 요소와 모든 attention head에 대해 각 attention scores 행렬의 대각 성분을 반환하기 때문입니다.

이를 구한 후, batch와 대각 성분 차원에 대해 평균을 내면 길이가 `n_heads`인 tensor를 얻을 수 있습니다. 그런 다음 `hook.layer()` 메서드를 사용하여 정확한 행 번호를 가져와 이를 전역 `induction_score_store` tensor에 기록할 수 있습니다.
</details>

<details>
<summary>솔루션</summary>

```python
def induction_score_hook(pattern: Float[Tensor, "batch head_index dest_pos source_pos"], hook: HookPoint):
    """
    Calculates the induction score, and stores it in the [layer, head] position of the `induction_score_store` tensor.
    """
    # Take the diagonal of attn paid from each dest posn to src posns (seq_len-1) tokens back
    # (This only has entries for tokens with index>=seq_len)
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    # Get an average score per head
    induction_score = einops.reduce(induction_stripe, "batch head_index position -> head_index", "mean")
    # Store the result.
    induction_score_store[hook.layer(), :] = induction_score
```

</details>

이 함수가 올바르게 구현되었다면, 이전 섹션에서의 관찰 결과와 일치하는 결과를 확인하실 수 있습니다. 즉, induction head로 식별한 모든 head에 대해서는 높은 induction score(>0.6)가 나타나고, 그 외의 모든 head에 대해서는 낮은 score(0에 가까운 값)가 나타나야 합니다.

### 연습 문제 - GPT2-small에서 induction heads 찾기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 10-20 minutes on this exercise.
> Here, you mostly just need to use previously defined functions and interpret the results, rather than writing new code.
> ```

*지금까지 사용해 온 단순한 2-layer 모델이 아니라, 더 크고 광범위하게 학습된 모델을 조사할 수 있는 첫 번째 기회입니다. 필요한 코드 중 새로운 내용은 없으므로 (대부분 이전 섹션에서 복사할 수 있습니다), 이 연습 문제는 그리 오래 걸리지 않을 것입니다.*

당신의 `gpt2_small` 에 대해 동일한 분석을 수행하십시오. 특히 몇몇 중간 layer의 일부 head들이 높은 induction score를 가지고 있음을 관찰할 수 있을 것입니다. CircuitsVis를 사용하여 반복되는 token sequence를 실행했을 때 이 head들의 attention pattern을 시각화하고, 그것들이 induction head처럼 보이는지 확인하십시오.

참고 - cache에서 직접 플로팅하는 대신 hook을 사용하여 CircuitsVis 플롯(및 기타 시각화)을 만들 수 있습니다. 예를 들어, `model.run_with_hooks` 호출에 포함시켰을 때 특정 hook에서의 attention pattern을 표시하는 hook 함수를 제공해 드렸습니다.

In [ ]:
def visualize_pattern_hook(
    pattern: Float[Tensor, "batch head_index dest_pos source_pos"],
    hook: HookPoint,
):
    print("Layer: ", hook.layer())
    display(cv.attention.attention_patterns(tokens=gpt2_small.to_str_tokens(rep_tokens[0]), attention=pattern.mean(0)))


# YOUR CODE HERE - find induction heads in gpt2_small

<details><summary>솔루션</summary>

```python
seq_len = 50
batch_size = 10
rep_tokens_batch = generate_repeated_tokens(gpt2_small, seq_len, batch_size)

induction_score_store = t.zeros((gpt2_small.cfg.n_layers, gpt2_small.cfg.n_heads), device=gpt2_small.cfg.device)

gpt2_small.run_with_hooks(
    rep_tokens_batch,
    return_type=None,  # For efficiency, we don't need to calculate the logits
    fwd_hooks=[(pattern_hook_names_filter, induction_score_hook)],
)

imshow(
    induction_score_store,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head",
    text_auto=".1f",
    width=700,
    height=500,
)

# Observation: heads 5.1, 5.5, 6.9, 7.2, 7.10 are all strongly induction-y.
# Confirm observation by visualizing attn patterns for layers 5 through 7:

induction_head_layers = [5, 6, 7]
fwd_hooks = [
    (utils.get_act_name("pattern", induction_head_layer), visualize_pattern_hook)
    for induction_head_layer in induction_head_layers
]
gpt2_small.run_with_hooks(
    rep_tokens,
    return_type=None,
    fwd_hooks=fwd_hooks,
)
```
</details>

## interpretability 도구 구축하기

transformer가 특정 작업을 어떻게 수행하는지에 대한 mechanistic한 이해를 발전시키기 위해서는 다음과 같은 질문에 답할 수 있어야 합니다:

> *특정 작업에 대한 모델 성능 중 얼마만큼이 모델의 각 구성 요소(component) 덕분인가?*

여기서 "구성 요소"란 예를 들어 특정 layer의 특정 head를 의미할 수 있습니다.

이러한 질문에 접근하는 방법은 많습니다. 예를 들어, 한 head가 다른 layer의 다른 head들과 어떻게 상호작용하는지 살펴볼 수도 있고, 해당 head의 효과를 제거했을 때 모델이 얼마나 잘 작동하는지 확인하는 causal intervention을 수행할 수도 있습니다. 하지만 지금은 간단하게 다음과 같은 질문을 던져보겠습니다: **이 head가 output logits에 직접적으로 기여하는 바는 무엇인가?**

### Direct Logit attribution

residual stream의 결과로, output logits는 각 layer의 기여도의 합이며, 따라서 각 head의 결과물의 합이 됩니다. 이는 우리가 output logits를 각 head에서 오는 항으로 분해하여 다음과 같이 직접적으로 attribution을 수행할 수 있음을 의미합니다!

<details>
<summary>구체적인 예시</summary>

모델이 Harry라는 token 다음에 Potter라는 token이 온다는 것을 알고 있고, 모델이 이를 어떻게 수행하는지 알아내고 싶다고 가정해 보겠습니다. Harry에서의 logits는 `residual @ W_U` 입니다. 하지만 이는 linear map이며, residual stream은 이전 모든 layer의 합입니다 `residual = embed + attn_out_0 + attn_out_1`. 따라서 `logits = (embed @ W_U) + (attn_out @ W_U) + (attn_out_1 @ W_U)`

더 구체적으로 Potter token의 logit만 살펴볼 수도 있습니다. 이는 `W_U` 의 한 열에 해당하며, 따라서 residual stream의 한 방향에 해당합니다. 이제 우리의 logit은 `(embed @ potter_U) + (attn_out_0 @ potter_U) + (attn_out_1 @ potter_U)` 의 합인 단일 숫자가 됩니다. 더 나아가, 각 attention layer의 출력을 각 head의 결과의 합으로 분해하여 많은 항을 얻을 수 있습니다.
</details>

여기서 여러분의 미션은 각 구성 요소가 정답 logit에 얼마나 기여하는지 확인하는 함수를 작성하는 것입니다. 구성 요소는 다음과 같습니다:

* Direct path (즉, embedding에서 unembedding으로 이어지는 residual connection)
* 각 layer 0 head (residual connection을 통해 layer 1을 건너뜀)
* 각 layer 1 head

강조하자면, 이것들은 모델의 시작부터 끝까지의 경로가 아니라, 특정 구성 요소의 출력에서 logits로 직접 이어지는 경로입니다. 각 경로가 어떻게 계산되었는지에 대해서는 아무런 가정을 하지 않습니다!

이 실습을 위한 몇 가지 중요한 참고 사항입니다:

* 여기서는 logits에 미치는 직접적인(DIRECT) 효과, 즉 이 구성 요소가 residual stream에 쓰고/embedding하는 내용만을 살펴봅니다. 만약 head가 다른 head와 결합하여 logits에 영향을 주거나, 정답 token을 부각시키기 위해 다른 token의 logits를 억제한다면, 이는 포착되지 않습니다!
* 정답 token에 해당하는 logits만 살펴봄으로써, 정답 다음 token 외의 모든 다른 token을 무시할 수 있기 때문에 데이터의 차원이 훨씬 낮아집니다 (50K 크기의 vocab size를 다루는 것은 매우 고통스럽습니다!). 하지만 이는 head가 다른 그럴듯한 logits를 억제하여 정답의 log prob를 높이는 것과 같은 더 미묘한 효과를 놓치는 대가를 치릅니다.
    * 작업이 더 쉬워지는 다른 상황들도 있습니다. 예를 들어, (곧 논의할) IOI 작업에서는 간접 목적어의 logits와 직접 목적어의 logits를 비교하기만 하면 되므로, **이 logits들 사이의 차이**를 사용하고 다른 모든 logits는 무시할 수 있습니다.
* 정답 output logits를 계산할 때, `(position,)` 이 아니라 `(position - 1,)` 차원의 tensor를 얻게 됩니다. 출력(logits)의 마지막 요소와 레이블(tokens)의 첫 번째 요소를 제거하기 때문입니다. 이는 우리가 *다음* token을 예측하고 있으며, 마지막 token 이후의 token은 알 수 없으므로 이를 무시하는 것입니다.

<details>
<summary>여담 - centering <code>W_U</code></summary>

이 실습에서는 걱정하지 않겠지만, logit attribution은 먼저 `W_U` 을 centering할 때 더 의미 있는 경우가 많습니다. 즉, output logits에 쓰는 각 행의 평균이 0이 되도록 보장하는 것입니다. Log softmax는 모든 logits에 상수를 더해도 변하지 않으므로, 단순히 모든 logits를 동일한 양만큼 증가시키는 head의 영향을 제어하고 싶어 합니다. 테스트의 편의를 위해 여기서는 이를 수행하지 않습니다.
</details>

<details>
<summary>질문 - 왜 log probs에 대해서는 이렇게 하지 않나요?</summary>

log probs는 linear하지 않으며, 비선형 함수인 `log_softmax` 을 거치기 때문입니다.
</details>

### 연습 문제 - logit attribution 도구 구축하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> This exercise is important, but has quite a few messy einsums, so you might get more value from reading the solution than doing the exercises.
> ```

아래의 `logit_attribution` 함수를 구현해야 합니다. 이 함수는 "정답 방향(correct direction)"에 대한 각 구성 요소의 기여도를 반환해야 합니다. 정답 방향에 대한 unembedding 벡터인 `W_U_correct_tokens`는 이미 제공되었습니다 (위에서 논의한 이유로 token의 `[1:]` 슬라이스를 사용한다는 점에 유의하십시오).

이 함수 아래의 코드는 logit attribution의 합계를 구하고 이를 모델 끝의 residual stream에 있는 실제 값과 비교함으로써, 구현한 logit attribution 함수가 올바르게 작동하는지 확인합니다.

In [ ]:
def logit_attribution(
    embed: Float[Tensor, "seq d_model"],
    l1_results: Float[Tensor, "seq nheads d_model"],
    l2_results: Float[Tensor, "seq nheads d_model"],
    W_U: Float[Tensor, "d_model d_vocab"],
    tokens: Int[Tensor, "seq"],
) -> Float[Tensor, "seq-1 n_components"]:
    """
    Inputs:
        embed: the embeddings of the tokens (i.e. token + position embeddings)
        l1_results: the outputs of the attention heads at layer 1 (with head as one of the dims)
        l2_results: the outputs of the attention heads at layer 2 (with head as one of the dims)
        W_U: the unembedding matrix
        tokens: the token ids of the sequence

    Returns:
        Tensor of shape (seq_len-1, n_components)
        represents the concatenation (along dim=-1) of logit attributions from:
            the direct path (seq-1,1)
            layer 0 logits (seq-1, n_heads)
            layer 1 logits (seq-1, n_heads)
        so n_components = 1 + 2*n_heads
    """
    W_U_correct_tokens = W_U[:, tokens[1:]]

    raise NotImplementedError()


text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."
logits, cache = model.run_with_cache(text, remove_batch_dim=True)
str_tokens = model.to_str_tokens(text)
tokens = model.to_tokens(text)

with t.inference_mode():
    embed = cache["embed"]
    l1_results = cache["result", 0]
    l2_results = cache["result", 1]
    logit_attr = logit_attribution(embed, l1_results, l2_results, model.W_U, tokens[0])
    # Uses fancy indexing to get a len(tokens[0])-1 length tensor, where the kth entry is the predicted logit for the correct k+1th token
    correct_token_logits = logits[0, t.arange(len(tokens[0]) - 1), tokens[0, 1:]]
    t.testing.assert_close(logit_attr.sum(1), correct_token_logits, atol=1e-3, rtol=0)
    print("Tests passed!")

<details><summary>솔루션</summary>

```python
def logit_attribution(
    embed: Float[Tensor, "seq d_model"],
    l1_results: Float[Tensor, "seq nheads d_model"],
    l2_results: Float[Tensor, "seq nheads d_model"],
    W_U: Float[Tensor, "d_model d_vocab"],
    tokens: Int[Tensor, "seq"],
) -> Float[Tensor, "seq-1 n_components"]:
    """
    Inputs:
        embed: the embeddings of the tokens (i.e. token + position embeddings)
        l1_results: the outputs of the attention heads at layer 1 (with head as one of the dims)
        l2_results: the outputs of the attention heads at layer 2 (with head as one of the dims)
        W_U: the unembedding matrix
        tokens: the token ids of the sequence

    Returns:
        Tensor of shape (seq_len-1, n_components)
        represents the concatenation (along dim=-1) of logit attributions from:
            the direct path (seq-1,1)
            layer 0 logits (seq-1, n_heads)
            layer 1 logits (seq-1, n_heads)
        so n_components = 1 + 2*n_heads
    """
    W_U_correct_tokens = W_U[:, tokens[1:]]

    direct_attributions = einops.einsum(W_U_correct_tokens, embed[:-1], "emb seq, seq emb -> seq")
    l1_attributions = einops.einsum(W_U_correct_tokens, l1_results[:-1], "emb seq, seq nhead emb -> seq nhead")
    l2_attributions = einops.einsum(W_U_correct_tokens, l2_results[:-1], "emb seq, seq nhead emb -> seq nhead")
    return t.concat([direct_attributions.unsqueeze(-1), l1_attributions, l2_attributions], dim=-1)
```
</details>

테스트가 정상적으로 작동하면, 모델을 통과하는 각 경로에 대한 logit attribution을 시각화할 수 있습니다. 결과를 보기 좋게 표시해 주는 헬퍼 함수 `plot_logit_attribution`를 제공해 드렸습니다.

In [ ]:
embed = cache["embed"]
l1_results = cache["result", 0]
l2_results = cache["result", 1]
logit_attr = logit_attribution(embed, l1_results, l2_results, model.W_U, tokens.squeeze())

plot_logit_attribution(model, logit_attr, tokens, title="Logit attribution (demo prompt)")

#### 질문 - 이 그래프의 해석은 무엇입니까?

logit attribution의 가장 큰 변동은 direct path에서 발생한다는 것을 알 수 있습니다. 특히, direct path의 일부 token들은 매우 높은 logit attribution을 가집니다 (예: token 7, 12, 24, 38, 46, 58). 무엇이 특히 이들에게 그렇게 높은 logit attribution을 부여하는지 추측할 수 있습니까?

<details>
<summary>답변 - 이 token들의 특별한 점은 무엇입니까?</summary>

매우 높은 logit attribution을 가진 token들은 일반적인 bigram의 첫 번째 token들입니다. 예를 들어, direct path에서 가장 높은 기여도는 `| manip|`에서 발생하는데, 이는 이 token 다음에 `|ulative|` (또는 아마도 `| ulation|`와 같은 다른 어근)가 올 가능성이 매우 높기 때문입니다. `| super| -> |human|`은 tokenizer가 하나의 단어를 여러 token으로 분리할 때 형성되는 bigram의 또 다른 예시입니다.

tokenizer에 의해 분리된 단일 단어가 아니라, 두 개의 서로 다른 단어에서 기인한 예시들도 있습니다. 여기에는 다음이 포함됩니다:

* `| more| -> | likely|` (12)
* `| machine| -> | learning|` (24)
* `| by| -> | default|` (38)
* `| how| -> | to|` (58)

tokenization의 모든 ~짜증 나는~ 재미있는 특이점들에 대한 논의는 나중에 확인하시기 바랍니다!
</details>

그래프의 또 다른 특징은 layer 1의 head들이 layer 0의 head들보다 훨씬 더 높은 기여도를 보이는 것처럼 보인다는 점입니다. 왜 그렇다고 생각하십니까?

<details>
<summary>힌트</summary>

이 그래프가 transformer를 통과하는 path의 관점에서 실제로 무엇을 나타내는지 생각해보십시오.
</details>

<details>
<summary>답변 - 왜 layer-1 head들이 더 높은 기여도를 가질 수 있습니까?</summary>

이는 앞서 논의한 점 때문입니다. 이 그래프는 다른 head와의 composition에서 발생하는 head의 효과와 같은 것들을 포착하지 못합니다. 따라서 layer-0 head들의 attribution에는 어떠한 composition도 포함되지 않지만, layer-1 head들의 attribution에는 해당 attention head들을 통과하는 단일 head path뿐만 아니라 layer 0과 layer 1의 head들을 통과하는 2-layer compositional path까지 포함되기 때문입니다.
</details>

### 연습 문제 - induction head의 logit attribution 해석하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

*이 연습 문제는 적절한 인자를 사용하여 `logit_attribution` 및 `plot_logit_attribution`를 호출하는 것만으로 구성됩니다. 중요한 부분은 결과를 해석하는 것입니다. 코드 구현에서 막힌다면 솔루션을 확인하시기 바랍니다. 이 부분은 중요하지 않습니다.*

`rep_cache`에 대해, attention-only 모델 `model`의 logit attribution을 수행합니다. 어떤 결과가 나올 것으로 예상하시나요?

<!-- Remember, you'll need to split the sequence in two, with one overlapping token (since predicting the next token involves removing the final token with no label) - your `logit_attr` should both have shape `[seq_len, 2*n_heads + 1]` (ie `[50, 25]` here). -->
<!-- 
<details>
<summary>참고 - 첫 번째 그래프는 거의 의미가 없을 것입니다. 그 이유를 알 수 있나요?</summary>

첫 번째 그래프는 시퀀스의 전반부, 즉 각 token이 처음으로 나타나는 부분에 대한 logit attribution을 보여주기 때문입니다. 이 시퀀스에는 아무런 구조가 없으므로(순수하게 무작위입니다), head가 의미 있는 계산을 수행할 이유가 없습니다. 구조는 token이 반복되는 시퀀스의 후반부에 존재하며, 높은 logit attribution을 가진 head들이 바로 induction을 수행할 수 있는 head들일 것입니다.
</details> -->

In [ ]:
# YOUR CODE HERE - plot logit attribution for the induction sequence (i.e. using `rep_tokens` and
# `rep_cache`), and interpret the results.

<details><summary>솔루션</summary>

```python
seq_len = 50

embed = rep_cache["embed"]
l1_results = rep_cache["result", 0]
l2_results = rep_cache["result", 1]

logit_attr = logit_attribution(embed, l1_results, l2_results, model.W_U, rep_tokens.squeeze())
plot_logit_attribution(model, logit_attr, rep_tokens.squeeze(), title="Logit attribution (random induction prompt)")
```
</details>

우리의 induction head circuit 맥락에서 이 그래프의 해석은 무엇입니까?

<details>
<summary>정답</summary>

그래프의 전반부는 대부분 의미가 없습니다. 왜냐하면 여기의 sequence들은 무작위이며 예측 가능한 패턴이 없으므로, 예측을 위해 의미 있는 계산을 수행하는 모델의 부분이 존재할 수 없기 때문입니다.

후반부에서는 head `1.4`와 `1.10`가 큰 logit attribution 점수를 가지고 있음을 알 수 있습니다. 이 head들이 induction을 수행하고 있는 것처럼 보였다는 이전 관찰 결과(두 head 모두 특징적인 induction 패턴을 보였음)를 고려하면 이는 타당합니다. 하지만 이 그래프가 attention 패턴을 보는 것과는 다른 종류의 증거를 제공한다는 점을 강조할 필요가 있습니다. 단순히 어떤 head가 특정 token에 attention을 기울이고 있다는 사실이, 그 정보가 반드시 구체적인 예측을 내리는 데 사용되고 있음을 의미하지는 않기 때문입니다. head `1.10`가 `1.4`보다 더 큰 직접적인 효과를 가지고 있음에 주목하십시오. 이는 (`1.10`가 `1.4`보다 더 높은 점수를 기록했던) 우리의 attention score 결과와 일치합니다.

</details>

## Hooks: Activation에 개입하기

모델의 출력을 분해하기 위한 도구들을 구축했으므로, 이제 인과적 개입(causal interventions)을 시작할 차례입니다.

### Ablations

간단한 예시인 **ablation**부터 시작하겠습니다. ablation은 모델에 대한 단순한 인과적 개입으로, 모델의 특정 부분을 선택해 이를 0으로 설정하는 것입니다. 이는 해당 부분이 얼마나 중요한지를 측정하는 투박한 대리 지표가 됩니다. 더 나아가, 모델 내의 특정 circuit이 어떤 능력을 가능하게 한다는 가설이 있을 때, *다른* 부분들을 ablation 해도 아무런 변화가 없음을 보여주는 것은 이에 대한 강력한 증거가 될 수 있습니다.

[the glossary](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=fh-HJyz1CgUVrXuoiban6bYx)에서 언급했듯이, ablation을 수행하는 방법은 많습니다. 여기서는 가장 단순한 방법인 zero-ablation에 집중하겠습니다 (비록 이것이 다소 원칙에 어긋나는 방법일지라도 말입니다).

### 연습 문제 - induction head ablation

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should aim to spend 20-35 mins on this exercise.
> ```

아래 코드는 특정 head의 출력 벡터(즉, attention 확률에 따라 value 벡터들의 가중 합을 구한 후, 이를 projection 하여 residual stream에 더하기 전의 벡터)에 대해 zero-ablation을 수행하기 위한 템플릿을 제공합니다. 서로 다른 activation이 무엇을 의미하는지 헷갈린다면 [the diagram](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/small-merm.svg)을 다시 참고하시기 바랍니다.

다음 두 가지 작업을 수행해야 합니다:

1. `head_index_to_ablate`에 의해 지정된 head에 zero-ablation을 수행하도록 `head_zero_ablation_hook`를 채우십시오.
2. `get_ablation_scores` 함수 내의 누락된 코드(즉, `raise NotImplementedError()` 라인이 있는 곳)를 채워, `layer` 레이어의 `head` head를 ablation한 후의 모델 loss가 `loss_with_ablation`으로 계산되도록 하십시오.

`get_ablation_scores` 함수의 나머지 부분은 이러한 각 head를 ablation했을 때 발생하는 loss 증가량을 포함하는 `(n_layers, n_heads)` shape의 tensor를 반환하도록 설계되었습니다.

이 함수에 대한 몇 가지 참고 사항 및 구현 팁입니다:

- `ablation_function`에 `functools.partial`을 적용하고 head index를 특정 값으로 고정하여 임시 hook 함수를 생성할 수 있습니다.
- `utils.get_act_name("z", layer)`을 사용하여 hook point의 이름을 가져올 수 있습니다 (이름이 지정된 hook point의 전체 다이어그램과 이름을 가져오는 방법은 [homepage](https://arena-chapter1-transformer-interp.streamlit.app/)로 이동한 후 왼쪽 사이드바에서 streamlit 참조 페이지를 통해 확인할 수 있습니다).
- `loss_no_ablation`가 `get_log_probs` 함수로 계산되며, 마지막 `seq_len - 1` token들만 취한다는 점에 유의하십시오. 이는 우리가 `2 * seq_len + 1` 길이의 시퀀스(BOS token 하나와 두 번 반복된 랜덤 시퀀스)를 다루고 있으며, 시퀀스의 후반부 loss에만 관심이 있기 때문입니다.
- 함수 시작 부분에서 `model.reset_hooks()`를 호출한다는 점에 유의하십시오. 이는 모델의 동작을 변경할 수 있는 hook이 실수로 남아있지 않도록 하기 위한 일반적인 유용한 관행입니다.

In [ ]:
def head_zero_ablation_hook(
    z: Float[Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index_to_ablate: int,
) -> None:
    raise NotImplementedError()


def get_ablation_scores(
    model: HookedTransformer,
    tokens: Int[Tensor, "batch seq"],
    ablation_function: Callable = head_zero_ablation_hook,
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Returns a tensor of shape (n_layers, n_heads) containing the increase in cross entropy loss
    from ablating the output of each head.
    """
    # Initialize an object to store the ablation scores
    ablation_scores = t.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)

    # Calculating loss without any ablation, to act as a baseline
    model.reset_hooks()
    seq_len = (tokens.shape[1] - 1) // 2
    logits = model(tokens, return_type="logits")
    loss_no_ablation = -get_log_probs(logits, tokens)[:, -(seq_len - 1) :].mean()

    for layer in tqdm(range(model.cfg.n_layers)):
        for head in range(model.cfg.n_heads):
            raise NotImplementedError()

    return ablation_scores


ablation_scores = get_ablation_scores(model, rep_tokens)
tests.test_get_ablation_scores(ablation_scores, model, rep_tokens)

<details><summary>솔루션</summary>

```python
def head_zero_ablation_hook(
    z: Float[Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index_to_ablate: int,
) -> None:
    z[:, :, head_index_to_ablate, :] = 0.0


def get_ablation_scores(
    model: HookedTransformer,
    tokens: Int[Tensor, "batch seq"],
    ablation_function: Callable = head_zero_ablation_hook,
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Returns a tensor of shape (n_layers, n_heads) containing the increase in cross entropy loss
    from ablating the output of each head.
    """
    # Initialize an object to store the ablation scores
    ablation_scores = t.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)

    # Calculating loss without any ablation, to act as a baseline
    model.reset_hooks()
    seq_len = (tokens.shape[1] - 1) // 2
    logits = model(tokens, return_type="logits")
    loss_no_ablation = -get_log_probs(logits, tokens)[:, -(seq_len - 1) :].mean()

    for layer in tqdm(range(model.cfg.n_layers)):
        for head in range(model.cfg.n_heads):
            # Use functools.partial to create a temporary hook function with the head number fixed
            temp_hook_fn = functools.partial(ablation_function, head_index_to_ablate=head)
            # Run the model with the ablation hook
            ablated_logits = model.run_with_hooks(tokens, fwd_hooks=[(utils.get_act_name("z", layer), temp_hook_fn)])
            # Calculate the loss difference (= neg correct logprobs), only on the last seq_len tokens
            loss = -get_log_probs(ablated_logits, tokens)[:, -(seq_len - 1) :].mean()
            # Store the result, subtracting the clean loss so that a value of 0 means no loss change
            ablation_scores[layer, head] = loss - loss_no_ablation

    return ablation_scores
```
</details>

테스트를 통과했다면, 결과를 plot 할 수 있습니다:

In [ ]:
imshow(
    ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads",
    text_auto=".2f",
    width=900,
    height=350,
)

이 결과들에 대해 어떻게 해석하시나요?

<details>
<summary>해석</summary>

이는 단순히 어떤 head가 정답을 얻기 위해 residual stream에 출력을 쓰는 역할을 하는지를 알려줄 뿐만 아니라, **어떤 head가 induction circuit에서 중요한 역할을 하는지**를 알려줍니다.

이 차트는 반복되는 token 시퀀스에 대해 layer 0에서는 head `0.7`이 단연 가장 중요하며(우리가 이를 가장 강력한 "previous token head"로 관찰했으므로 타당합니다), layer 1에서는 head `1.4`, `1.10`가 가장 중요하다는 것을 보여줍니다(우리가 이들이 가장 induction-y 하다고 관찰했으므로 타당합니다).

이는 ablation을 통해 얻을 수 있는 결과의 좋은 예시입니다. 하지만 이는 인과적 개입(causal intervention)이 아니기 때문에, **direct logit attribution과 같은 방법으로는 얻을 수 없는** 결과입니다.
</details>

### 연습 문제 - mean ablation

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should aim to spend 5-15 mins on this exercise.
> ```

zero-ablation의 대안은 **mean-ablation**입니다. 이는 값을 0으로 설정하는 대신, 적절한 분포(일반적으로 batch 차원에 대한 평균을 사용합니다)에 걸친 평균값으로 설정하는 방식입니다. zero-ablation은 모델을 정상적인 분포에서 벗어나게 하므로, 특정 컴포넌트의 효과를 "껐을 때" 얻게 될 결과를 반드시 대표한다고 볼 수 없습니다. 반면 mean ablation은 약간 더 효과적으로 작동합니다(물론 그에 따른 위험 요소가 있습니다). 더 자세한 내용은 [here](https://www.neelnanda.io/mechanistic-interpretability/glossary#:~:text=Ablation%20aka%20Knockout) 또는 [here](https://arxiv.org/html/2404.15255v1)에서 읽어보실 수 있습니다.

아래의 `head_mean_ablation_hook` 함수를 완성하고 코드를 실행하십시오 (또한 이전의 `get_ablation_scores` 함수에서 zero ablation 함수를 하드코딩하지 않고 실제로 `ablation_function`를 사용했는지 확인하십시오. 그렇지 않으면 여기서 코드가 작동하지 않습니다). 중요하지 않은 head들의 값이 중요한 head들에 비해 0에 훨씬 더 가깝게 나타나며, 결과가 약간 더 깔끔해지는 것을 확인하실 수 있습니다.

In [ ]:
def head_mean_ablation_hook(
    z: Float[Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index_to_ablate: int,
) -> None:
    raise NotImplementedError()


rep_tokens_batch = run_and_cache_model_repeated_tokens(model, seq_len=50, batch_size=10)[0]
mean_ablation_scores = get_ablation_scores(model, rep_tokens_batch, ablation_function=head_mean_ablation_hook)

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads",
    text_auto=".2f",
    width=900,
    height=350,
)

<details><summary>솔루션</summary>

```python
def head_mean_ablation_hook(
    z: Float[Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index_to_ablate: int,
) -> None:
    z[:, :, head_index_to_ablate, :] = z[:, :, head_index_to_ablate, :].mean(0)


rep_tokens_batch = run_and_cache_model_repeated_tokens(model, seq_len=50, batch_size=10)[0]
mean_ablation_scores = get_ablation_scores(model, rep_tokens_batch, ablation_function=head_mean_ablation_hook)

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads",
    text_auto=".2f",
    width=900,
    height=350,
)
```
</details>

## 보너스 - head 0.4 & 0.11 이해하기 (매우 어려움!)

우리의 induction ablation 실험에서 강하게 나타났지만, 이 섹션의 다른 분석 `0.4` 및 `0.11` 에서는 그만큼 두드러지지 않았던 2개의 head가 있습니다. 이 head들이 무엇을 하고 있는지 알아내기 위해 인과적 실험(즉, targeted ablations)을 설계할 수 있습니까?

> 참고 - 다음 섹션에서 어느 정도 진전을 이룬 후에 이 과제에 도전하는 것을 권장합니다. 그렇게 하면 induction circuit에 대해 더 기계론적인 이해를 얻을 수 있기 때문입니다. 그렇게 하더라도 이 보너스 연습 문제는 여전히 어려울 수 있습니다. 왜냐하면 우리가 다루어 온 잘 정의된 induction circuit의 범위를 벗어나 잠재적으로 더 모호한 결과로 진입하기 때문입니다. **다시 말씀드리지만, 이 내용은 매우 어렵습니다!**

<details>
<summary>시작을 위한 힌트는 다음과 같습니다</summary>

head `0.4`와 `0.11`이 attention을 주고 있는 위치를 살펴보십시오. 모델이 잘 작동하기 위해 어떤 source position에 attention을 주는 것이 중요한지 알아낼 수 있습니까?

</details>

<details>
<summary>부분 정답 (및 샘플 코드)</summary>

아래는 몇 가지를 제외한 모든 offset position에서 head `0.4` 및 `0.11`의 input을 ablate 했을 때의 효과를 플롯하는 샘플 코드입니다 (예를 들어, 첫 번째 행은 self-attention에서 오는 것을 제외한 모든 head input을 mean ablating 했을 때 loss에 미치는 영향을 보여주며, 두 번째 행은 시퀀스에서 바로 앞에 오는 token에서 오는 것을 제외한 모든 input을 ablate 했을 때의 효과를 보여줍니다).

```python
def head_z_ablation_hook(
    z: Float[Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index_to_ablate: int,
    seq_posns: list[int],
    cache: ActivationCache,
) -> None:
    """
    We perform ablation at the z vector, by doing the equivalent of mean ablating all the inputs to this attention head
    except for those which come from the tokens `n` positions back, where `n` is in the `seq_posns` list.
    """
    batch, seq = z.shape[:2]
    v = cache["v", hook.layer()][:, :, head_index_to_ablate]  # shape [batch seq_K d_head]
    pattern = cache["pattern", hook.layer()][:, head_index_to_ablate]  # shape [batch seq_Q seq_K]

    # Get a repeated version of v, and mean ablate all but the previous token values
    v_repeated = einops.repeat(v, "b sK h -> b sQ sK h", sQ=seq)
    v_ablated = einops.repeat(v_repeated.mean(0), "sQ sK h -> b sQ sK h", b=batch).clone()
    for offset in seq_posns:
        seqQ_slice = t.arange(offset, seq)
        v_ablated[:, seqQ_slice, seqQ_slice - offset] = v_repeated[:, seqQ_slice, seqQ_slice - offset]

    # Take weighted sum of this new v, and use it to edit `z` inplace.
    z[:, :, head_index_to_ablate] = einops.einsum(v_ablated, pattern, "b sQ sK h, b sQ sK -> b sQ h")


def get_ablation_scores_cache_assisted(
    model: HookedTransformer,
    tokens: Int[Tensor, "batch seq"],
    ablation_function: Callable = head_zero_ablation_hook,
    seq_posns: list[int] = [0],
    layers: list[int] = [0],
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Version of `get_ablation_scores` which can use the cache to assist with the ablation.
    """
    ablation_scores = t.zeros((len(layers), model.cfg.n_heads), device=model.cfg.device)

    model.reset_hooks()
    seq_len = (tokens.shape[1] - 1) // 2
    logits, cache = model.run_with_cache(tokens, return_type="logits")
    loss_no_ablation = -get_log_probs(logits, tokens)[:, -(seq_len - 1) :].mean()

    for layer in layers:
        for head in range(model.cfg.n_heads):
            temp_hook_fn = functools.partial(ablation_function, head_index_to_ablate=head, cache=cache, seq_posns=seq_posns)
            ablated_logits = model.run_with_hooks(tokens, fwd_hooks=[(utils.get_act_name("z", layer), temp_hook_fn)])
            loss = -get_log_probs(ablated_logits, tokens)[:, -(seq_len - 1) :].mean()
            ablation_scores[layer, head] = loss - loss_no_ablation

    return ablation_scores


rep_tokens_batch = run_and_cache_model_repeated_tokens(model, seq_len=50, batch_size=50)[0]

offsets = [[0], [1], [2], [3], [1, 2], [1, 2, 3]]
z_ablation_scores = [
    get_ablation_scores_cache_assisted(model, rep_tokens_batch, head_z_ablation_hook, offset).squeeze()
    for offset in tqdm(offsets)
]

imshow(
    t.stack(z_ablation_scores),
    labels={"x": "Head", "y": "Position offset", "color": "Logit diff"},
    title="Loss Difference (ablating heads everywhere except for certain offset positions)",
    text_auto=".2f",
    y=[str(offset) for offset in offsets],
    width=900,
    height=400,
)
```

이 코드 결과로부터 얻을 수 있는 몇 가지 관찰 결과입니다:

- **Head `0.7`는 진정한 previous token head입니다.** 두 번째 행을 보면 이전 token에서 오는 것을 제외한 모든 input을 mean ablating 해도 loss에 영향이 없으므로, 이것이 해당 head가 사용하는 모든 정보임을 알 수 있습니다.
- **Head `0.11`는 오직 current token head입니다.** 첫 번째 행을 보면 self-attending(즉, 현재 token으로의 attention)에서 오는 것을 제외한 모든 input을 mean ablating 해도 loss에 영향이 없으므로, 이것이 해당 head가 사용하는 모든 정보임을 알 수 있습니다.
- **Head `0.4`는 1, 2 또는 3 token 전의 위치에서 오는 정보만 사용합니다.** 이는 위 플롯의 5번째 행에서 확인할 수 있습니다. 1 또는 2 position 전의 token에서 오는 것을 제외한 모든 input을 ablate 했을 때의 효과가 매우 작습니다. 단순히 attention pattern을 보는 것이 아니라 ablation 실험을 통해 이 결론을 내리는 것이 중요하다는 점에 유의하십시오. 왜냐하면 특정 token에 attention을 준다고 해서, 그 token이 이 특정 분포(induction)의 맥락에서 중요한 방식으로 사용되고 있다는 것을 의미하지는 않기 때문입니다.

`0.11`부터 시작해 보겠습니다. 우리는 layer 1에 token을 복사하는 역할을 하는 head들이 있다는 것을 알고 있습니다. 즉, 시퀀스 `[A][B]...[A][B]`에서 이들은 두 번째 `[A]`에서 첫 번째 `[B]`로 attention을 주고 그 value를 복사하여 예측값으로 사용합니다. 그리고 만약 head `0.11`가 항상 self-attend 한다면, `(embedding of B) + (output of head 0.11 when it attends to token B)`를 "`B`의 진정한 embedding"으로 생각하는 것이 타당합니다. 왜냐하면 이것이 layer 1 head가 복사하도록 학습하게 될 대상이기 때문입니다. 이러한 **extended embedding** 또는 **effective embedding**이라는 개념은 나중에 GPT2-Small을 살펴볼 때 다시 등장할 것입니다. `0.11`의 output이 layer-1 copying head의 QK circuit에서 더 중요한지, 아니면 OV copying head에서 더 중요한지는 독자 여러분의 과제로 남겨두겠습니다!

다음으로 `0.4`를 보겠습니다. 이 head는 1 token 전과 2 token 전의 정보를 모두 사용하고 있습니다. induction circuit에는 previous token head가 포함되어 있으므로 이전 token을 사용하는 것은 타당합니다. 하지만 2 position 전의 정보로는 무엇을 하고 있을까요? 한 가지 가설은 이것이 역시 induction circuit을 만들고 있지만, 2개가 아닌 3개의 token을 사용하고 있다는 것입니다! 다시 말해, 두 번째 `[A]`가 "이 token의 value 바로 다음에 온 token"으로 attention을 주는 `[A][B]...[A][B]`와 같은 시퀀스 대신, 두 번째 `[A]`가 "이전 token의 value에서 2 position 뒤에 온 token"으로 attention을 주는 `[Z][A][B]...[Z][A][B]`와 같은 시퀀스를 가질 수 있다는 것입니다. 이를 테스트하는 한 가지 방법은 최대 2번의 반복이 있는 무작위 induction 시퀀스를 구성하는 것입니다. 즉, 전반부는 무작위 시퀀스로 구성하고, 후반부는 전반부에서 서로 인접하게 나타난 무작위 선택 token 쌍들로 구성합니다. 예를 들어, vocab size가 10이고 half seq len이 10인 경우 다음과 같은 시퀀스가 될 수 있습니다:

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">0 5 8 3 1 8 2 2 4 6 (5 8) (4 6) (3 1) (2 4) (0 5)</pre>

head `0.4`에 대한 우리의 가설에 따르면, 이러한 종류의 시퀀스에서 이 head를 mean ablating 하는 것은 loss에 거의 영향을 주지 않아야 합니다 (최소 3 길이의 induction 시퀀스를 지원하도록 설계되었기 때문입니다). 반면, induction 실험에서 중요하다고 식별된 다른 모든 head들(`0.7`, `0.11`, `1.4`, `1.10`)은 여전히 중요해야 합니다. 실제로 우리가 발견한 결과가 바로 이것이며, 아래 코드를 통해 직접 확인해 보실 수 있습니다.

```python
def generate_repeated_tokens_maxrep(
    model: HookedTransformer,
    seq_len: int,
    batch_size: int = 1,
    maxrep: int = 2,
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Same as previous function, but contains a max number of allowed repetitions. For example, maxrep=2 means we can have
    sequences like `[A][B]...[A][B]`, but not `[A][B][C]...[A][B][C]`.
    """
    prefix = (t.ones(batch_size, 1) * model.tokenizer.bos_token_id).long()
    rep_tokens_half = t.randint(0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.int64)
    rep_tokens = t.cat([prefix, rep_tokens_half], dim=-1)
    for _ in range(seq_len // maxrep + 1):
        random_start_posn = t.randint(0, seq_len - 2, (batch_size,)).tolist()
        rep_tokens_repeated = t.stack([rep_tokens_half[b, s : s + maxrep] for b, s in enumerate(random_start_posn)])
        rep_tokens = t.cat([rep_tokens, rep_tokens_repeated], dim=-1)

    return rep_tokens[:, : 2 * seq_len + 1].to(device)


rep_tokens_max2 = generate_repeated_tokens_maxrep(model, seq_len=50, batch_size=50, maxrep=2)

mean_ablation_scores = get_ablation_scores(model, rep_tokens_max2, ablation_function=head_mean_ablation_hook)

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads",
    text_auto=".2f",
    width=900,
    height=350,
)
```

</details>

# 4️⃣ induction circuit 역공학

> ##### 학습 목표
>
> - activation 패턴을 살펴보고 circuit을 조사하는 것과, weight를 직접 살펴보고 circuit을 역공학(reverse-engineering)하는 것의 차이점을 이해합니다.
> - factored matrix 클래스를 사용하여 induction circuit 내의 QK 및 OV circuit을 조사합니다.
> - induction circuit에 대한 추가 탐색을 수행합니다: composition score 및 targeted ablation.

이전 실습에서는 induction circuit에서 어떤 attention head가 중요한지 식별하기 위해 attention pattern과 attribution을 살펴보았습니다. 이는 circuit에 대한 감을 잡는 좋은 방법일 수 있지만, 이를 이해하는 매우 엄격한 방법은 아닙니다. 이는 특정 head가 특정 클래스의 입력에 대해 어떤 작업을 수행하는 것처럼 *보인다*는 점을 관찰하지만, *왜* 그렇게 하는지는 식별하지 않는 **feature analysis**에 더 가깝습니다.

이제 우리는 더 엄격한 mechanistic analysis를 수행하겠습니다. weight를 깊이 있게 분석하여 induction head 알고리즘을 역공학(reverse engineer)하고, 그것이 실제로 우리가 생각하는 대로 작동하고 있는지 검증하겠습니다.

## 복습 - induction circuit

이 섹션의 핵심 내용으로 들어가기 전에, induction heads를 조사하며 지금까지 얻은 결과들을 복습해 보겠습니다. 우리는 다음과 같은 사실을 발견했습니다:

* 반복되는 token 시퀀스가 입력되었을 때, head `1.4`와 `1.10`은 offset `seq_len - 1`를 가진 대각선 줄무늬 형태의 특징적인 induction head attention 패턴을 보였습니다.
    * 이는 CircuitsVis 결과와, 우리가 선택한 지표로 측정했을 때 이 head들이 높은 induction score를 가졌다는 사실(다른 모든 head들은 훨씬 낮은 score를 가짐)을 통해 확인했습니다.
* 또한 head `0.7`은 (반복되지 않는 시퀀스에서도) 시퀀스의 이전 token에 강하게 attend한다는 것을 확인했습니다.
* 모델에 대해 **logit attribution**을 수행한 결과, head `1.4`와 `1.10`가 residual stream에 쓴 값들이 시퀀스의 후반부에서 정확한 예측을 내는 데 모두 중요하다는 것을 발견했습니다.
* 모델에 대해 **zero-ablation**을 수행한 결과, head `0.7`, `1.4`, `1.10`를 ablation했을 때 반복 시퀀스 태스크에서 정확도가 크게 저하되는 것을 확인했습니다.

이 모든 관찰 결과를 바탕으로, induction circuit이 무엇이며 어떻게 작동하는지 여러분의 언어로 요약해 보십시오. 설명 시 특정 head들의 QK 및 OV circuit과 연결 지어 설명하고, 어떤 유형의 attention head composition이 일어나고 있는지 기술해야 합니다.

아래 드롭다운을 사용하여 이해한 내용을 확인해 볼 수 있습니다.

<details>
<summary>알고리즘 요약</summary>

* Head `0.7`는 previous token head입니다 (QK-circuit이 항상 이전 token에 attend하도록 보장합니다).
* Head `0.7`의 OV circuit은 이전 token의 복사본을 embedding에서 사용하는 것과 *다른* subspace에 씁니다.
* Head `0.7`의 출력은 K-Composition을 통해 head `1.10`의 *key* 입력으로 사용되어, '이전 token이 destination token인 source token'에 attend하게 합니다.
* Head `1.10`의 OV-circuit은 source token의 *value*를 동일한 output logit으로 복사합니다.
    * 이는 `0.7` output subspace가 아니라 embedding subspace에서 복사하는 것임에 유의하십시오. 즉, V-Composition을 전혀 사용하지 않습니다.
* `1.4` 또한 `1.10`와 동일한 역할을 수행합니다 (따라서 함께 작동하여 더 정확해질 수 있으며, 구체적인 방법은 나중에 살펴보겠습니다).

강조하자면, 정교하고 어려운 부분은 induction head의 *attention* 패턴을 계산하는 것이며, 여기에는 세심한 composition이 필요합니다. previous token을 찾는 부분과 복사하는 부분은 비교적 간단합니다. 이는 QK circuit과 OV circuit이 어떻게 반독립적으로 작동하는지, 그리고 왜 종종 별개로 생각하는 것이 좋은지를 보여주는 좋은 예시입니다. 또한 attention 패턴을 계산하는 과정에 실제적이고 정교한 계산이 포함될 수 있음을 보여줍니다!

아래는 induction circuit의 다이어그램이며, weight matrix에 각 head가 표시되어 있습니다.

![kcomp_diagram_3.png](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described_3.png)
</details>

## 복습 - QK 및 OV circuit

시작하기 전에, 용어에 대해 짧게 설명하겠습니다. 특정 layer와 head의 weight matrix를 위첨자 표기법으로 나타내겠습니다. 예를 들어, $W_Q^{1.4}$은 layer 1의 4번째 head를 위한 query matrix이며, shape은 `[d_model, d_head]`입니다 (weight matrix를 오른쪽에 곱한다는 점을 기억하십시오). 마찬가지로, attention pattern은 $A^{1.4}$로 표시됩니다 (이것들은 파라미터가 아니라 **activation**이며, $x$가 residual stream(shape `[seq_len, d_model]`)일 때 공식 $A^h = x W_{QK}^h x^T$에 의해 결정된다는 점을 기억하십시오).

약식으로, $A$을 token `A`의 one-hot encoding(즉, `A`의 인덱스만 1이고 나머지는 모두 0인 벡터)으로 나타내겠습니다. 따라서 $A^T W_E$는 `A`의 embedding 벡터가 됩니다.

마지막으로, 특수한 matrix product를 다음과 같이 지칭하겠습니다:

* $W_{OV}^{h} := W_V^{h}W_O^{h}$은 head $h$의 **OV circuit**이며, $W_E W_{OV}^h W_U$은 **full OV circuit**입니다.
* $W_{QK}^h := W_Q^h (W_K^h)^T$는 head $h$의 **QK circuit**이며, $W_E W_{QK}^h W_E^T$은 **full QK circuit**입니다.

이 matrix들의 순서가 **Mathematical Frameworks** 논문과 약간 다르다는 점에 유의하십시오. 이는 TransformerLens가 weight matrix를 저장하는 방식 때문입니다.

#### 질문 - 다음 각 행렬의 해석은 무엇입니까?

*질문이 꽤 많지만, 개념적으로 매우 중요합니다. 혼란스럽다면 처음 몇 가지 질문의 답을 읽어본 후 나머지 질문들에 도전해 보시기 바랍니다.*

답변에는 해당 행렬이 어떤 입력을 받는지, 그리고 출력이 무엇을 나타내는지 설명해야 합니다.

#### $W_{OV}^{h}$

<details>
<summary>정답</summary>

$W_{OV}^{h}$의 크기는 $(d_\text{model}, d_\text{model})$이며, 이는 **residual stream에서 어떤 정보가 소스(source)에서 목적지(destination)로 이동하는지**를 설명하는 linear map입니다.

다시 말해, $x$이 residual stream의 벡터라면, 목적지 token이 $x$ 위치의 소스 token에만 attention을 기울일 때, 목적지 위치의 residual stream에 기록되는 벡터가 $x^T W_{OV}^{h}$입니다.
</details>

#### $W_E W_{OV}^h W_U$

<details>
<summary>힌트</summary>

$A$가 token `A`에 대한 one-hot encoding(즉, token `A`에 해당하는 위치만 1이고 나머지는 모두 0인 벡터)이라면, $A^T W_E W_{OV}^h W_U$이 무엇을 나타내는지 생각해 보십시오. 이 식을 왼쪽에서 오른쪽으로 순차적으로 계산할 수 있습니다 (예: 먼저 $A^T W_E$이 무엇을 나타내는지 생각한 다음, 나머지 두 행렬을 곱하십시오).
</details>
<details>
<summary>정답</summary>

$W_E W_{OV}^h W_U$의 크기는 $(d_\text{vocab}, d_\text{vocab})$이며, 이는 **시작부터 끝까지의 관점에서 어떤 정보가 소스에서 목적지로 이동하는지**를 설명하는 linear map입니다.

$A$가 token `A`에 대한 one-hot encoding이라면:

* $A^T W_E$은 `A`의 embedding 벡터입니다.
* $A^T W_E W_{OV}^h$는 목적지 token이 `A`에만 attention을 기울일 때, 목적지 위치의 residual stream에 기록될 벡터입니다.
* $A^T W_E W_{OV}^h W_U$은 이 벡터의 unembedding, 즉 최종 logits에 더해지는 값입니다.

</details>

#### $W_{QK}^{h}$

<details>
<summary>정답</summary>

$W_{QK}^{h}$의 크기는 $(d_\text{model}, d_\text{model})$이며, 이는 residual stream에서 **정보가 어디로 이동하고 어디서 오는지** (즉, 어떤 residual stream 벡터가 다른 어떤 벡터에 attention을 기울이는지)를 설명하는 bilinear form입니다.

$x_i^T W_{QK}^h x_j = (x_i^T W_Q^h) (x_j^T W_K^h)^T$는 token $i$이 token $j$에 기울이는 attention score입니다.
</details>

#### $W_E W_{QK}^h W_E^T$

<details>
<summary>정답</summary>

$W_E W_{QK}^h W_E^T$의 크기는 $(d_\text{vocab}, d_\text{vocab})$이며, 이는 vocabulary 내의 단어들 사이에서 **정보가 어디로 이동하고 어디서 오는지** (즉, 어떤 token이 다른 어떤 token에 attention을 기울이는지)를 설명하는 bilinear form입니다.

$A$와 $B$이 token `A`과 `B`에 대한 one-hot encoding이라면, $A^T W_E W_{QK}^h W_E^T B$은 token `A`가 token `B`에 기울이는 attention score입니다:

$$
A^T \, W_E\, W_{QK}^{h}\, W_E^T \, B = \underbrace{(A^T W_E W_Q^{h})}_{\text{query for token } A}  \underbrace{(B^T W_E W_K^{h})^T}_{\text{key for token }B}
$$
</details>

#### $W_{pos} W_{QK}^h W_{pos}^T$

<details>
<summary>정답</summary>

$W_{pos} W_{QK}^h W_{pos}^T$의 크기는 $(n_\text{ctx}, n_\text{ctx})$이며, 이는 context 내의 token들 사이에서 **정보가 어디로 이동하고 어디서 오는지** (즉, 어떤 token 위치가 다른 위치에 attention을 기울이는지)를 설명하는 bilinear form입니다.

$i$와 $j$가 위치 `i`과 `j`에 대한 one-hot encoding(즉, 단순히 i번째와 j번째 basis vector)이라면, $i^T W_{pos} W_{QK}^h W_{pos}^T j$은 위치 `i`의 token이 위치 `j`의 token에 기울이는 attention score입니다:

$$
i^T \, W_{pos}\, W_{QK}^{h}\, W_{pos}^T \, j = \underbrace{(i^T W_{pos} W_Q^{h})}_{\text{query for i-th token}}  \underbrace{(j^T W_{pos} W_K^{h})^T}_{\text{key for j-th token}}
$$

</details>

#### $W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T$

여기서 $h_1$는 $h_2$보다 이전 layer에 있습니다.

<details>
<summary>힌트</summary>

이 행렬은 크기가 $(d_\text{vocab}, d_\text{vocab})$인 bilinear form으로 보는 것이 가장 좋습니다. $(A, B)$번째 요소는 다음과 같습니다:

$$
(A^T W_E W_{OV}^{h_1}) W_{QK}^{h_2} (B^T W_E)^T
$$
</details>

<details>
<summary>정답</summary>

$W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T$의 크기는 $(d_\text{vocab}, d_\text{vocab})$이며, 이는 **query-side 벡터**가 head $h_1$의 출력으로 형성되었을 때, head $h_2$에서 정보가 어디로 이동하고 어디서 오는지 설명하는 bilinear form입니다. 다시 말해, 이는 **Q-composition**의 한 사례입니다.

$A$와 $B$이 token `A`와 `B`에 대한 one-hot encoding이라면, $A^T W_E W_{OV}^{h_1} W_{QK}^{h_2} W_E^T B$는 head $h_1$에서 `A`-token에 강하게 attention을 기울인 모든 token이 token `B`**에** 기울이는 attention score입니다.

---

여전히 혼란스럽다면, 이를 더 자세히 분석해 보겠습니다:

$$
\begin{aligned}
A^T \, W_E\, W_{OV}^{h_1} W_{QK}^{h_2}\, W_E^T \, B &= \underbrace{(A^T W_E W_{OV}^{h_1}W_Q^{h_2})}_{\text{query of token which attended to A}}  \underbrace{(B^T W_E W_K^{h_2})^T}_\text{key of token B} \\
\end{aligned}
$$

---

실제 attention score는 단 하나의 항이 아니라 여러 항의 합이 된다는 점에 유의하십시오 (실제로 query와 key 입력의 모든 조합에 대해 서로 다른 항이 존재합니다). 하지만 이 항은 query와 key 입력의 이 조합으로 인한 **특정한 기여분**을 설명하며, 때로는 이 항만이 중요하고 나머지 항들은 최종 확률에 거의 영향을 주지 않는 경우도 있습니다. 나중에 정확히 이런 사례를 보게 될 것입니다.
</details>

시작하기 전에, 이 모든 행렬을 계산할 때 발생할 수 있는 문제가 있습니다. 일부 행렬은 크기가 매우 커서 GPU에 들어가지 않을 수 있습니다. 예를 들어, 두 full circuit 행렬은 모두 $(d_\text{vocab}, d_\text{vocab})$의 shape을 가지며, 저희의 경우 이는 $50278\times 50278 \approx 2.5\times 10^{9}$개의 요소를 의미합니다. GPU가 이를 처리할 수 있더라도, 여전히 비효율적으로 보입니다. 실제로 행렬을 계산하지 않고도 이 행렬들을 의미 있게 분석할 수 있는 방법이 있을까요?

## Factored Matrix 클래스

transformer interpretability에서는 low rank로 factorize된 행렬, 즉 M이 `[large, large]`이지만 A는 `[large, small]`이고 B는 `[small, large]`인 행렬 $M = AB$를 분석해야 하는 경우가 많습니다. 이는 transformer에서 흔히 볼 수 있는 구조입니다.

예를 들어, 위의 OV circuit을 $W_{OV}^h = W_V^h W_O^h$으로 factorize할 수 있으며, 여기서 $W_V^h$은 `[768, 64]`의 shape을 가지고 $W_O^h$는 `[64, 768]`의 shape을 가집니다. 더 극단적인 예로, full OV circuit은 $(W_E W_V^h) (W_O^h W_U)$로 쓸 수 있으며, 이때 두 행렬은 각각 `[50278, 64]`와 `[64, 50278]`의 shape을 가집니다. 마찬가지로, full QK circuit은 $(W_E W_Q^h) (W_E W_K^h)^T$로 쓸 수 있습니다.

`FactoredMatrix` 클래스는 이러한 행렬들을 다루기에 편리한 방법입니다. 이 클래스는 trace, eigenvalues, Frobenius norm, singular value decomposition 계산 및 다른 행렬과의 곱셈과 같은 다양한 연산을 위한 효율적인 알고리즘을 구현합니다. 이는 (근사적으로) 원래 행렬을 대체하는 drop-in replacement로 작동할 수 있습니다.

이 모든 것이 가능한 이유는 행렬의 factorisation을 알면 중요한 특성들을 훨씬 더 쉽게 계산할 수 있기 때문입니다. 직관적으로, $M=AB$은 매우 작은 subspace에서 작동하는 매우 큰 행렬이므로, 실제 값인 $M_{ij}$을 아는 것이 이를 저장하는 가장 효율적인 방법일 것이라고 기대해서는 안 됩니다!

### 연습 문제 - 분해된 행렬(factored matrix)의 성질 유도하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than 10-25 minutes on this exercise.
> 
> If you're less interested in the maths, you can skip these exercises.
> ```

분해된 행렬이 있을 때 어떤 성질들을 쉽게 계산할 수 있는지 알아보기 위해, 몇 가지를 직접 유도해 보겠습니다.

$M=AB$라고 가정해 봅시다. 여기서 $A$의 shape은 $(m, n)$이고, $B$의 shape은 $(n, m)$이며, $m > n$입니다. 따라서 $M$은 크기가 $(m, m)$이고 rank가 최대 $n$인 행렬입니다.

**질문 - $M$의 trace를 어떻게 쉽게 계산할 수 있을까요?**

<details>
<summary>정답</summary>

다음과 같습니다:

$$
\text{Tr}(M) = \text{Tr}(AB)
= \sum_{i=1}^m \sum_{j=1}^n A_{ij} B_{ji}
$$

따라서 trace의 계산 결과는 $O(mn)$입니다.

trace의 순환성(cyclicity)에 의해 $\text{Tr}(M) = \text{Tr}(BA)$임을 보일 수도 있습니다 (비록 trace를 계산하기 위해 $AB$ 곱셈을 직접 계산할 필요는 없지만 말입니다).
</details>

**질문 - $M$의 eigenvalue를 어떻게 쉽게 계산할 수 있을까요?**

(이후 연습 문제에서 보시겠지만, eigenvalue는 행렬을 평가하는 데 매우 중요합니다. 예를 들어, $W_{OV}$의 eigenvalue를 살펴봄으로써 OV circuit의 [copying scores](https://transformer-circuits.pub/2021/framework/index.html#copying-matrix)을 평가할 수 있습니다.)

<details>
<summary>힌트</summary>

$AB$보다는 $BA$의 eigenvalue를 찾는 것이 계산 비용이 더 저렴합니다.

$AB$과 $BA$의 eigenvalue는 어떤 관계가 있을까요?
</details>
<details>
<summary>정답</summary>

$AB$과 $BA$의 eigenvalue는 다음과 같은 관계가 있습니다: 만약 $\mathbf{v}$가 $AB$의 eigenvector이고 eigenvalue가 $ABv = \lambda \mathbf{v}$이라면, $B\mathbf{v}$은 $BA$의 eigenvector이며 동일한 eigenvalue를 가집니다:

$$
BA(B\mathbf{v}) = B (AB\mathbf{v}) = B (\lambda \mathbf{v}) = \lambda (B\mathbf{v})
$$

이는 $B\mathbf{v} = \mathbf{0}$일 때만 성립하지 않지만, 이 경우 $AB\mathbf{v} = \mathbf{0}$이므로 $\lambda = 0$입니다. 따라서 $AB$의 모든 0이 아닌 eigenvalue는 $BA$의 eigenvalue이기도 하다는 결론을 내릴 수 있습니다.

(행렬의 크기가 훨씬 작기 때문에) $BA$의 eigenvalue를 계산하는 것이 계산 비용이 훨씬 저렴하며, 이를 통해 $AB$의 모든 0이 아닌 eigenvalue를 얻을 수 있습니다.
</details>

**질문 (어려움) - $M$의 SVD를 어떻게 쉽게 계산할 수 있을까요?**

<details>
<summary>힌트</summary>

$m > n$를 가진 크기 $(m, n)$ 행렬의 경우, [algorithmic complexity of finding SVD](https://en.wikipedia.org/wiki/Singular_value_decomposition#Numerical_approach)은 $O(mn^2)$입니다. 따라서 $A$과 $B$의 SVD를 찾는 것은 상대적으로 저렴합니다 (복잡도 $mn^2$ vs $m^3$). 이를 이용하여 $M$의 SVD를 찾을 수 있을까요?
</details>


<details>
<summary>정답</summary>

작은 행렬인 $A$과 $B$의 SVD를 계산하는 것이 훨씬 저렴합니다. 이 SVD들을 다음과 같이 나타내겠습니다:

$$
\begin{aligned}
A &= U_A S_A V_A^T \\
B &= U_B S_B V_B^T
\end{aligned}
$$

여기서 $U_A$와 $V_B$은 $(m, n)$이고, 나머지 행렬들은 $(n, n)$입니다.

그러면 다음과 같습니다:

$$
\begin{aligned}
\quad\quad\quad\quad M &= AB \\
&= U_A (S_A V_A^T U_B S_B) V_B^T
\end{aligned}
$$

가운데 있는 행렬의 크기는 $(n, n)$ (즉, 작음)이므로, 이 행렬의 SVD를 저렴하게 계산할 수 있습니다:

$$
\begin{aligned}
\; S_A V_A^T U_B S_B &= U' S' {V'}^T \quad\quad\quad\quad\quad
\end{aligned}
$$

최종적으로, 이를 통해 $M$의 SVD를 얻을 수 있습니다:

$$
\begin{aligned}
\quad\quad M &= U_A U' S' {V'}^T V_B^T \\
&= U S {V'}^T
\end{aligned}
$$

여기서 $U = U_A U'$, $V = V_B V'$, 그리고 $S = S'$입니다.

모든 SVD 계산과 행렬 곱셈의 복잡도는 최대 $O(mn^2)$이었으며, 이는 $O(m^3)$보다 훨씬 효율적입니다 ($U = U_A U'$의 모든 값을 계산할 필요 없이, 0이 아닌 singular value에 해당하는 값들만 계산하면 된다는 점을 기억하십시오).
</details>

궁금하시다면 `FactoredMatrix` 문서를 방문하여 SVD 계산의 구현 방식과 다른 성질 및 연산들을 확인하실 수 있습니다.

이제 `FactoredMatrix` 클래스를 사용하는 몇 가지 동기에 대해 논의했으므로, 실제로 어떻게 작동하는지 살펴보겠습니다.

### 기본 예제

기본 클래스를 직접 사용할 수 있습니다. factored matrix를 직접 생성하고 기본 연산들을 살펴보겠습니다:

In [ ]:
A = t.randn(5, 2)
B = t.randn(2, 5)
AB = A @ B
AB_factor = FactoredMatrix(A, B)
print("Norms:")
print(AB.norm())
print(AB_factor.norm())

print(f"Right dim: {AB_factor.rdim}, Left dim: {AB_factor.ldim}, Hidden dim: {AB_factor.mdim}")

행렬의 eigenvalue와 singular value를 살펴볼 수도 있습니다. 행렬의 rank는 2이지만 크기는 5 by 5이므로, 마지막 3개의 eigenvalue와 singular value는 0이라는 점에 유의하십시오. factored 클래스는 이 0들을 생략합니다.

In [ ]:
print("Eigenvalues:")
print(t.linalg.eig(AB).eigenvalues)
print(AB_factor.eigenvalues)

print("\nSingular Values:")
print(t.linalg.svd(AB).S)
print(AB_factor.S)

print("\nFull SVD:")
print(AB_factor.svd())

<details>
<summary>참고 - SVD 메서드에 의해 반환되는 객체들의 크기입니다.</summary>

만약 $M = USV^T$이고, `M.shape = (m, n)`이며 rank가 `r`라면, SVD 메서드는 행렬 $U, S, V$을 반환합니다. 이 행렬들의 shape는 각각 `(m, r)`, `(r,)`, `(n, r)`인데, 그 이유는 다음과 같습니다:

* $S$의 off-diagonal 성분들은 모두 0이므로 굳이 저장하지 않습니다.
* $U$과 $V$의 열 중에서 0인 singular value에 해당하는 열들은 $USV^T$의 값에 영향을 주지 않으므로 굳이 저장하지 않습니다.
</details>

인수분해된(factored) 행렬을 인수분해되지 않은 행렬과 곱하여 또 다른 인수분해된 행렬을 얻을 수 있습니다 (아래 예시와 같습니다). 또한 두 개의 인수분해된 행렬을 서로 곱하여 또 다른 인수분해된 행렬을 얻을 수도 있습니다.

In [ ]:
C = t.randn(5, 300)
ABC = AB @ C
ABC_factor = AB_factor @ C

print(f"Unfactored: shape={ABC.shape}, norm={ABC.norm()}")
print(f"Factored: shape={ABC_factor.shape}, norm={ABC_factor.norm()}")
print(f"\nRight dim: {ABC_factor.rdim}, Left dim: {ABC_factor.ldim}, Hidden dim: {ABC_factor.mdim}")

이를 다시 분해되지 않은 행렬로 합치고 싶다면, `AB` 성질을 사용하여 곱을 구할 수 있습니다:

In [ ]:
AB_unfactored = AB_factor.AB
t.testing.assert_close(AB_unfactored, AB)

## 회로 역공학 (Reverse-engineering circuits)

우리의 induction circuit 내에는 네 개의 개별 회로가 있습니다: 이전 token head의 OV 및 QK circuit, 그리고 induction head의 OV 및 QK circuit입니다. 이 실습의 다음 섹션들에서, 우리는 이러한 각 회로를 차례대로 역공학해 보겠습니다.

* **OV copying circuit** 섹션에서는 layer-1 OV circuit을 살펴봅니다.
* **QK prev-token circuit** 섹션에서는 layer-0 QK circuit을 살펴봅니다.
* 세 번째 섹션(**K-composition**)은 조금 더 까다롭습니다. layer-0 OV circuit**과** layer-1 QK circuit의 composition을 살펴보아야 하기 때문입니다. 우리는 다음 두 가지를 수행해야 합니다:
    1. 이 두 회로가 composing하고 있음을 보여줍니다 (즉, layer-0 OV circuit의 출력이 layer-1 QK circuit의 key 벡터를 결정하는 주요 요인임을 보여줍니다).
    2. 이 두 회로의 결합 동작이 "token의 두 번째 인스턴스가 이전 인스턴스 *다음*에 오는 token에 attend하게 만드는 것"임을 보여줍니다.

아래 드롭다운에는 세 섹션이 induction circuit의 서로 다른 구성 요소들과 어떻게 연관되는지 설명하는 다이어그램이 포함되어 있습니다. 명확하게 보기 위해 새 탭에서 열어야 할 수도 있습니다.

<details>
<summary>Diagram</summary>

![kcomp](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described_2_new.png)
</details>

그 후, 우리는 composition score를 살펴보겠습니다. 이는 두 attention head가 composing하고 있음을 보여주는 더 수학적으로 정당화된 방법입니다 (특정 입력 클래스에 대한 동작을 살펴볼 필요 없이, 실제 모델 weight의 속성이기 때문입니다).

## [1] OV copying circuit

회로의 쉬운 부분부터 시작하겠습니다. 바로 `1.4`와 `1.10`의 copying OV circuit입니다. head 4부터 시작하겠습니다. 여기서 해석 가능한(즉, **privileged basis**인) 요소는 input token과 output logit뿐이므로, 우리는 다음 행렬을 연구하고자 합니다:

$$
W_E W_{OV}^{1.4} W_U
$$

(`1.10`의 경우도 마찬가지입니다). 이것은 attention pattern과 결합하여 input에서 output으로 연결해 주는 $(d_\text{vocab}, d_\text{vocab})$ 모양의 행렬입니다.

우리는 이 행렬을 계산하고 조사하고자 합니다. 대각 성분(diagonal values)은 매우 높고, 비대각 성분(non-diagonal values)은 훨씬 낮다는 것을 발견하게 될 것입니다.

**질문 - 왜 이러한 결과가 나올 것이라고 예상할 수 있을까요?** (이전 섹션에서 서로 다른 행렬들의 해석이 무엇이었는지 설명했던 내용을 다시 참고하는 것이 도움이 될 것입니다.)

<details>
<summary>힌트</summary>

반복되는 시퀀스가 `A B ... A B`라고 가정해 보겠습니다. $A$, $B$를 그에 대응하는 one-hot encoded token이라고 하겠습니다. 이 행렬의 `B`번째 행은 다음과 같습니다:

$$
B^T W_E W_{OV}^{1.4} W_U
$$

우리의 attention head 맥락에서 이 식의 해석은 무엇일까요?
</details>

<details>
<summary>정답</summary>

반복되는 시퀀스가 `A B ... A B`라면:

$$
B^T W_E W_{OV}^{1.4} W_U
$$

은 **첫 번째 `B` token에서 두 번째 `A` token으로 이동하여, 두 번째 `A` token 다음에 올 token의 예측값으로 사용되는 logit 벡터**입니다. 이는 `B`에 대해서는 높은 예측값을, 그 외의 모든 것에 대해서는 낮은 예측값을 가져야 합니다. 다시 말해, 이 행렬의 `(B, X)`번째 요소는 `X=B`에서 가장 높아야 하며, 이것이 바로 우리가 주장한 내용입니다.

여전히 혼란스럽다면, 아래 다이어그램이 도움이 될 수 있습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described-OV-v3.png" width="750">

</details>

### 연습 문제 - `1.4`를 위한 OV circuit 계산하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

*이 문제는 행렬 곱셈을 통해 circuit을 계산하는 여러 유사한 연습 문제 중 첫 번째입니다. 이 연습 문제는 매우 중요합니다 (특히, 이 행렬이 무엇을 나타내는지와 왜 우리가 여기에 관심을 갖는지 반드시 이해해야 합니다). 하지만 실제 계산 자체는 그리 오래 걸리지 않을 것입니다.*

이를 `FactoredMatrix` 객체로 계산해야 합니다.

모델의 weight에 직접 접근할 수 있다는 점을 기억하십시오. 예를 들어 `model.W_E` 또는 `model.W_Q`를 사용할 수 있습니다 (후자는 layer와 head로 인덱싱된 모든 `W_Q` 행렬을 제공합니다).

In [ ]:
head_index = 4
layer = 1

# YOUR CODE HERE - complete the `full_OV_circuit` object

tests.test_full_OV_circuit(full_OV_circuit, model, layer, head_index)

<details>
<summary>도움이 필요합니다 - 이 클래스를 사용하여 3개 이상의 행렬 곱을 계산하는 방법을 잘 모르겠습니다.</summary>

다음과 같이 직접 계산할 수 있습니다:

```python
full_OV_circuit = FactoredMatrix(W_E @ W_V, W_O @ W_U)
```

또는, `FactoredMatrix` 클래스의 또 다른 유용한 기능은 행렬 곱셈을 체이닝(chain)할 수 있다는 점입니다. 다음 코드는 정확히 동일한 `FactoredMatrix` 객체를 정의합니다:

```python
OV_circuit = FactoredMatrix(W_V, W_O)
full_OV_circuit = W_E @ OV_circuit @ W_U
```
</details>


<details><summary>정답</summary>

```python
head_index = 4
layer = 1

W_O = model.W_O[layer, head_index]
W_V = model.W_V[layer, head_index]
W_E = model.W_E
W_U = model.W_U

OV_circuit = FactoredMatrix(W_V, W_O)
full_OV_circuit = W_E @ OV_circuit @ W_U
```
</details>

이제 이 행렬이 identity 행렬인지 확인해 보겠습니다. 이 행렬은 factored matrix 형태이므로 확인이 다소 까다롭지만, 여전히 시도해 볼 수 있는 방법들이 있습니다.

먼저, 이 행렬이 대각 행렬과 유사한지 검증하기 위해 200개의 무작위 행과 열을 선택하여 시각화해 보겠습니다. 최소한 여기서는 identity 행렬과 비슷하게 보여야 합니다! 우리는 `FactoredMatrix` 클래스의 indexing 메서드를 사용하고 있습니다. 전체를 계산하는 것을 피하기 위해 실제 `.AB` 값을 반환하기 전에 인덱싱을 수행할 수 있습니다 (우리는 `A[left_indices, :] @ B[:, right_indices]`가 `(A @ B)[left_indices, right_indices]`과 같다는 점을 이용합니다).

In [ ]:
indices = t.randint(0, model.cfg.d_vocab, (200,))
full_OV_circuit_sample = full_OV_circuit[indices, indices].AB

imshow(
    full_OV_circuit_sample,
    labels={"x": "Logits on output token", "y": "Input token"},
    title="Full OV circuit for copying head",
    width=700,
    height=600,
)

<details>
<summary>사이드 노트 - factored matrices의 인덱싱</summary>

factored matrices의 또 다른 장점은 전체 행렬을 계산하지 않고도 작은 submatrices를 평가할 수 있다는 점입니다. 이는 행렬 `AB`의 `[i, j]`번째 요소가 `A[i, :] @ B[:, j]`라는 사실에 기반합니다.
</details>

### 연습 문제 - circuit 정확도 계산하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend approximately 10-15 minutes on this exercise.
> ```

factored matrix를 인덱싱하면 또 다른 factored matrix를 얻게 됩니다. 따라서 `A[left_indices, :] @ B[:, left_indices]`을 명시적으로 계산하는 대신, 단순히 `AB[left_indices, left_indices]`라고 작성할 수 있습니다.

여기서 꽤 뚜렷한 대각선 패턴이 관찰되어야 하며, 이는 좋은 신호입니다. 하지만 matrix에 노이즈가 상당히 많기 때문에 정확히 identity가 되지는 않을 것입니다. 대신, "identity에 얼마나 가까운지"에 대한 대략적인 감을 잡기 위해 요약 통계량을 고안해야 합니다.

**Accuracy**는 좋은 요약 통계량입니다. 즉, 가장 큰 logit이 대각선에 위치하는 비율이 얼마나 되는지를 확인하는 것입니다. 노이즈가 많더라도, 상당수의 경우 가장 큰 logit은 여전히 대각선에 위치할 것으로 기대할 수 있습니다.

Colab을 사용 중이거나 강력한 GPU를 가지고 있다면, 전체 matrix를 계산하고 이 테스트를 수행할 수 있을 것입니다. 하지만 CUDA 문제를 피하기 위해 가능하면 이 matrix를 반복문으로 처리하는 것이 더 좋은 관행입니다. 아래 함수에 `batch_size` 인자를 제공했으므로, `d_vocab * d_vocab`와 같은 거대한 matrix 대신 `batch_size * d_vocab` 크기의 matrix만 명시적으로 계산하도록 시도해야 합니다.

In [ ]:
def top_1_acc(full_OV_circuit: FactoredMatrix, batch_size: int = 1000) -> float:
    """
    Return the fraction of the time that the maximum value is on the circuit diagonal.
    """
    raise NotImplementedError()


print(f"Fraction of time that the best logit is on diagonal: {top_1_acc(full_OV_circuit):.4f}")

<details>
<summary>도움이 필요합니다 - 행(row)에 대해 argmax를 취해야 할지 열(column)에 대해 취해야 할지 잘 모르겠습니다.</summary>

OV circuit은 `W_E @ W_OV @ W_U` 로 정의됩니다. 우리는 i번째 행 `W_E[i] @ W_OV @ W_U` 을, OV matrix `W_OV` 를 가진 attention head를 통해 **`i`번째 token에 attention을 주는 모든 token에 더해지는 logit 벡터**를 나타내는 벡터로 볼 수 있습니다.

따라서 우리는 행(즉, `dim=1` )에 대해 argmax를 취하고자 합니다. 왜냐하면 `tok` 에 attention이 갈 때, 그것이 동시에 최상위 예측값이 되는 vocabulary 내의 token 수 `tok` 에 관심이 있기 때문입니다.

</details>

<details>
<summary>정답</summary>

```python
def top_1_acc(full_OV_circuit: FactoredMatrix, batch_size: int = 1000) -> float:
    """
    Return the fraction of the time that the maximum value is on the circuit diagonal.
    """
    total = 0

    for indices in t.split(t.arange(full_OV_circuit.shape[0], device=device), batch_size):
        AB_slice = full_OV_circuit[indices].AB
        total += (t.argmax(AB_slice, dim=1) == indices).float().sum().item()

    return total / full_OV_circuit.shape[0]
```

</details>

이 값은 약 30.79%를 반환하며, 상당히 실망스러운 결과입니다. top-5의 경우 47.73%까지 올라가지만, 여전히 좋지 않은 수준입니다. 왜 이런 결과가 나오는 것일까요?

### 연습 문제 - effective circuit 계산하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than 5-10 minutes on this exercise.
> This exercise should be very short; it only requires 2 lines of code. Understanding it conceptually is more important than the actual coding.
> ```

이제 왜 *두 개*의 induction head가 필요한지로 돌아가 보겠습니다. 만약 두 head가 동일한 attention pattern을 가진다면, 실질적인 OV circuit은 사실상 $W_E(W_V^{1.4}W_O^{1.4}+W_V^{1.10}W_O^{1.10})W_U$ 이며, 이것이 중요한 점입니다. 그러니 이 부분에 대해 분석을 다시 실행해 보겠습니다!

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/effective_ov_circuit.png" width="650">

<details>
<summary>질문 - 왜 모델이 circuit을 두 개의 head에 걸쳐 나누고 싶어 할까요?</summary>

$W_V W_O$이 rank 64 행렬이기 때문입니다. 두 개의 합은 rank 128 행렬이 됩니다. 이는 목표로 하는 50K x 50K 행렬에 대해 훨씬 더 나은 근사치가 될 수 있습니다!
</details>

In [ ]:
# YOUR CODE HERE - compute the effective OV circuit, and run `top_1_acc` on it

<details>
<summary>예상 출력</summary>

top-1에 대해 95.6%의 정확도를 얻어야 하며, 이는 훨씬 더 나은 결과입니다!

top 5 정확도를 시도해 볼 수도 있으며, 이 경우 결과가 98%까지 향상됩니다.

</details>


<details><summary>솔루션</summary>

```python
W_O_both = einops.rearrange(model.W_O[1, [4, 10]], "head d_head d_model -> (head d_head) d_model")
W_V_both = einops.rearrange(model.W_V[1, [4, 10]], "head d_model d_head -> d_model (head d_head)")

W_OV_eff = W_E @ FactoredMatrix(W_V_both, W_O_both) @ W_U

print(f"Fraction of the time that the best logit is on the diagonal: {top_1_acc(W_OV_eff):.4f}")
```
</details>

## [2] QK prev-token circuit

또 다른 쉬운 circuit은 L0H7의 QK-circuit입니다. 이것이 어떻게 이전 token circuit이라는 것을 알 수 있을까요?

우리는 positional embedding을 통해 전체 QK circuit을 곱해낼 수 있습니다:

$$
W_\text{pos} W_Q^{0.7} (W_K^{0.7})^T W_\text{pos}^T
$$

이를 통해 `[max_ctx, max_ctx]` 형태의 행렬 `pos_by_pos` 을 얻을 수 있습니다 (max ctx = max context length, 즉 우리가 허용하는 시퀀스의 최대 길이이며, 이는 $W_\text{pos}$ 에서 선택한 차원에 의해 설정됩니다).

이 경우, 우리의 max context window는 2048입니다 (이는 `model.cfg.n_ctx` 를 통해 확인할 수 있습니다). 이는 이전 섹션에서 다루었던 50k 크기의 행렬보다 훨씬 작으므로, 여기서는 factored matrix 클래스를 사용할 필요가 없습니다.

### 연습 문제 - `0.7`를 위한 전체 QK-circuit 해석하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

아래 코드는 head `0.7`에 대한 전체 QK circuit을 시각화합니다 (실제 attention layer에서 QK bilinear form이 어떻게 사용되는지를 반영하기 위해 scaling 및 softmax 단계가 포함되어 있습니다). 코드를 실행하고, induction circuit의 맥락에서 결과를 해석하시기 바랍니다.

In [ ]:
layer = 0
head_index = 7

# Compute full QK matrix (for positional embeddings)
W_pos = model.W_pos
W_QK = model.W_Q[layer, head_index] @ model.W_K[layer, head_index].T
pos_by_pos_scores = W_pos @ W_QK @ W_pos.T

# Mask, scale and softmax the scores
mask = t.tril(t.ones_like(pos_by_pos_scores)).bool()
pos_by_pos_pattern = t.where(mask, pos_by_pos_scores / model.cfg.d_head**0.5, -1.0e6).softmax(-1)

# Plot the results
print(f"Avg lower-diagonal value: {pos_by_pos_pattern.diag(-1).mean():.4f}")
imshow(
    utils.to_numpy(pos_by_pos_pattern[:200, :200]),
    labels={"x": "Key", "y": "Query"},
    title="Attention patterns for prev-token QK circuit, first 100 indices",
    width=700,
    height=600,
)

## [3] K-composition circuit

이제 circuit의 어려운 부분인 previous token head와 induction head 사이의 K-Composition을 증명해 보겠습니다.

#### activation 분리하기

logit attribution score에서 사용했던 기법을 반복해서 사용할 수 있습니다. layer 1의 QK-input은 14개 항(2+n_heads)의 합입니다. 즉, token embedding, positional embedding, 그리고 각 layer 0 head의 결과값들입니다. 따라서 layer 1의 각 head $\text{H}$ 에 대해, 시퀀스 위치 $i$ 에 해당하는 query tensor(key도 마찬가지입니다)는 다음과 같습니다:

$$
\begin{align*}
x W^\text{1.H}_Q &= (e + pe + \sum_{h=0}^{11} x^\text{0.h}) W^\text{1.H}_Q \\
&= e W^\text{1.H}_Q + pe W^\text{1.H}_Q + \sum_{h=0}^{11} x^\text{0.h} W^\text{1.H}_Q
\end{align*}
$$

여기서 $e$ 은 token embedding을, $pe$ 는 positional embedding을, 그리고 $x^\text{0.h}$ 는 layer 0의 head $h$ 의 출력을 나타냅니다 (그리고 이 tensor들의 합은 residual stream $x$ 과 같습니다). 이 모든 tensor들은 `[seq, d_model]` 의 shape을 가집니다. 따라서 위의 식을 matrix multiplication의 합 `[seq, d_model] @ [d_model, d_head] -> [seq, d_head]` 으로 취급할 수 있습니다.

표기법의 편의를 위해, 14개의 입력을 $(e, pe, x^\text{0.h}, ..., x^{h.11})$ 대신 $(y_0, y_1, ..., y_{13})$ 으로 부르겠습니다. 그러면 다음과 같습니다:

$$
x W^h_Q = \sum_{i=0}^{13} y_i W^h_Q
$$

여기서 각 $y_i$ 은 `[seq, d_model]` 의 shape을 가지며, $y_i$ 들의 합은 전체 residual stream $x$ 이 됩니다. 이를 설명하는 다이어그램은 다음과 같습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/components.png" width="520">

### 연습 문제 - 상대적 중요도 분석하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 15-25 minutes on these exercises.
> Most of these functions just involve indexing and einsums, but conceptual understanding / figuring out exactly what the question is asking for is the hard part!
> ```

이제 이 14개 항의 상대적 중요도를 분석할 수 있습니다! 매우 단순한 측정 방법은 각 항의 norm을 구하는 것입니다 (컴포넌트 및 위치별로).

이것이 상당히 불확실한 지표라는 점에 유의하십시오. q와 k는 본질적으로 interpretable하지 않습니다! 하지만 쉽고 빠르게 계산할 수 있는 좋은 대리 지표(proxy)가 될 수 있습니다.

<details>
<summary>질문 - 왜 Q와 K는 본질적으로 interpretable하지 않을까요? 그럼에도 불구하고 왜 norm이 좋은 지표가 될 수 있을까요?</summary>

이들이 본질적으로 interpretable하지 않은 이유는 **privileged basis**가 없는 residual stream 상에서 작동하기 때문입니다. 모든 $Q$, $K$, $V$ 가중치 뒤에 rotation matrix $R$를 추가하고 (그리고 residual stream에 쓰는 모든 것 앞에 rotation matrix를 추가하더라도), 모델은 여전히 정확히 동일하게 동작할 것입니다.

그럼에도 불구하고 norm을 구하는 것이 여전히 합리적인 이유는, 이 벡터들의 개별 요소들이 본질적으로 interpretable하지는 않더라도, 그 값이 더 크다면 residual stream에 더 큰 전반적 영향을 미칠 가능성이 높기 때문입니다. 따라서 norm을 살펴보는 것이 그것들이 어떻게 작동하는지를 알려주지는 않지만, 어떤 것이 더 중요한지는 나타내 줍니다.
</details>

아래 함수들을 완성하십시오:

In [ ]:
def decompose_qk_input(cache: ActivationCache) -> Float[Tensor, "n_heads+2 posn d_model"]:
    """
    Retrieves all the input tensors to the first attention layer, and concatenates them along the
    0th dim.

    The [i, :, :]th element is y_i (from notation above). The sum of these tensors along the 0th
    dim should be the input to the first attention layer.
    """
    raise NotImplementedError()


def decompose_q(
    decomposed_qk_input: Float[Tensor, "n_heads+2 posn d_model"],
    ind_head_index: int,
    model: HookedTransformer,
) -> Float[Tensor, "n_heads+2 posn d_head"]:
    """
    Computes the tensor of query vectors for each decomposed QK input.

    The [i, :, :]th element is y_i @ W_Q (so the sum along axis 0 is just the q-values).
    """
    raise NotImplementedError()


def decompose_k(
    decomposed_qk_input: Float[Tensor, "n_heads+2 posn d_model"],
    ind_head_index: int,
    model: HookedTransformer,
) -> Float[Tensor, "n_heads+2 posn d_head"]:
    """
    Computes the tensor of key vectors for each decomposed QK input.

    The [i, :, :]th element is y_i @ W_K(so the sum along axis 0 is just the k-values)
    """
    raise NotImplementedError()


# Recompute rep tokens/logits/cache, if we haven't already
seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_cache) = run_and_cache_model_repeated_tokens(model, seq_len, batch_size)
rep_cache.remove_batch_dim()

ind_head_index = 4

# First we get decomposed q and k input, and check they're what we expect
decomposed_qk_input = decompose_qk_input(rep_cache)
decomposed_q = decompose_q(decomposed_qk_input, ind_head_index, model)
decomposed_k = decompose_k(decomposed_qk_input, ind_head_index, model)
t.testing.assert_close(
    decomposed_qk_input.sum(0),
    rep_cache["resid_pre", 1] + rep_cache["pos_embed"],
    rtol=0.01,
    atol=1e-05,
)
t.testing.assert_close(decomposed_q.sum(0), rep_cache["q", 1][:, ind_head_index], rtol=0.01, atol=0.001)
t.testing.assert_close(decomposed_k.sum(0), rep_cache["k", 1][:, ind_head_index], rtol=0.01, atol=0.01)

# Second, we plot our results
component_labels = ["Embed", "PosEmbed"] + [f"0.{h}" for h in range(model.cfg.n_heads)]
for decomposed_input, name in [(decomposed_q, "query"), (decomposed_k, "key")]:
    imshow(
        utils.to_numpy(decomposed_input.pow(2).sum([-1])),
        labels={"x": "Position", "y": "Component"},
        title=f"Norms of components of {name}",
        y=component_labels,
        width=800,
        height=400,
    )

<details>
<summary>확인해야 할 사항</summary>


가장 중요한 query 구성 요소는 token 및 positional embedding이라는 것을 확인할 수 있습니다. 가장 중요한 key 구성 요소는 $y_9$에서 온 것이며, 이는 $x_7$, 즉 head `0.7`에서 온 것입니다.

</details>

<details>
<summary>positional embedding에 관한 기술적 참고 사항 - 선택 사항이므로 자유롭게 건너뛰셔도 됩니다.</summary>

왜 테스트에서 분해된 qk 합을 단순히 `resid_pre`이 아니라 `resid_pre + pos_embed`의 합과 비교하는지 궁금하실 수 있습니다. 그 답은 우리가 transformer를 정의한 방식, 특히 config의 다음 라인에 있습니다:

```python
positional_embedding_type="shortformer"
```

이로 인한 결과는 positional embedding이 residual stream에 더해지지 않는다는 것입니다. 대신, 이는 Q와 K 계산의 입력으로 더해지지만 (즉, `(resid_pre + pos_embed) @ W_Q`을 계산하며 `W_K`에 대해서도 마찬가지입니다), V 계산의 입력으로는 더해지지 않습니다 (즉, 단순히 `resid_pre @ W_V`만 계산합니다). 이는 일반적인 attention의 작동 방식은 아니지만, 우리의 목적상 positional embedding이 OV circuit을 방해하지 않으므로 induction heads 분석을 더 깔끔하게 만들어 줍니다.

**질문 - 이러한 유형의 embedding은 실제로 attention head가 Q-composition을 통해 형성되는 것을 불가능하게 만듭니다. 그 이유를 알 수 있을까요?**

</details>


<details><summary>정답</summary>

```python
def decompose_qk_input(cache: ActivationCache) -> Float[Tensor, "n_heads+2 posn d_model"]:
    """
    Retrieves all the input tensors to the first attention layer, and concatenates them along the
    0th dim.

    The [i, :, :]th element is y_i (from notation above). The sum of these tensors along the 0th
    dim should be the input to the first attention layer.
    """
    y0 = cache["embed"].unsqueeze(0)  # shape (1, seq, d_model)
    y1 = cache["pos_embed"].unsqueeze(0)  # shape (1, seq, d_model)
    y_rest = cache["result", 0].transpose(0, 1)  # shape (12, seq, d_model)

    return t.concat([y0, y1, y_rest], dim=0)


def decompose_q(
    decomposed_qk_input: Float[Tensor, "n_heads+2 posn d_model"],
    ind_head_index: int,
    model: HookedTransformer,
) -> Float[Tensor, "n_heads+2 posn d_head"]:
    """
    Computes the tensor of query vectors for each decomposed QK input.

    The [i, :, :]th element is y_i @ W_Q (so the sum along axis 0 is just the q-values).
    """
    W_Q = model.W_Q[1, ind_head_index]

    return einops.einsum(decomposed_qk_input, W_Q, "n seq d_model, d_model d_head -> n seq d_head")


def decompose_k(
    decomposed_qk_input: Float[Tensor, "n_heads+2 posn d_model"],
    ind_head_index: int,
    model: HookedTransformer,
) -> Float[Tensor, "n_heads+2 posn d_head"]:
    """
    Computes the tensor of key vectors for each decomposed QK input.

    The [i, :, :]th element is y_i @ W_K(so the sum along axis 0 is just the k-values)
    """
    W_K = model.W_K[1, ind_head_index]

    return einops.einsum(decomposed_qk_input, W_K, "n seq d_model, d_model d_head -> n seq d_head")


# Recompute rep tokens/logits/cache, if we haven't already
seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_cache) = run_and_cache_model_repeated_tokens(model, seq_len, batch_size)
rep_cache.remove_batch_dim()

ind_head_index = 4

# First we get decomposed q and k input, and check they're what we expect
decomposed_qk_input = decompose_qk_input(rep_cache)
decomposed_q = decompose_q(decomposed_qk_input, ind_head_index, model)
decomposed_k = decompose_k(decomposed_qk_input, ind_head_index, model)
t.testing.assert_close(
    decomposed_qk_input.sum(0),
    rep_cache["resid_pre", 1] + rep_cache["pos_embed"],
    rtol=0.01,
    atol=1e-05,
)
t.testing.assert_close(decomposed_q.sum(0), rep_cache["q", 1][:, ind_head_index], rtol=0.01, atol=0.001)
t.testing.assert_close(decomposed_k.sum(0), rep_cache["k", 1][:, ind_head_index], rtol=0.01, atol=0.01)

# Second, we plot our results
component_labels = ["Embed", "PosEmbed"] + [f"0.{h}" for h in range(model.cfg.n_heads)]
for decomposed_input, name in [(decomposed_q, "query"), (decomposed_k, "key")]:
    imshow(
        utils.to_numpy(decomposed_input.pow(2).sum([-1])),
        labels={"x": "Position", "y": "Component"},
        title=f"Norms of components of {name}",
        y=component_labels,
        width=800,
        height=400,
    )
```
</details>

이는 어떤 head가 중요할 가능성이 높은지 알려주지만, 우리는 이보다 더 나은 방법을 사용할 수 있습니다. query와 key 구성 요소를 개별적으로 보는 대신, 그것들이 어떻게 결합되는지 확인할 수 있습니다. 즉, 분해된 attention score를 취하는 것입니다.

이는 q와 k의 bilinear 함수이며, 따라서 우리는 `decomposed_scores` 텐서를 얻게 되며 그 shape는 `[query_component, key_component, query_pos, key_pos]` 입니다. 여기서 처음 두 축을 모두 따라 합산하면 원래의 attention score(mask 적용 전)를 얻을 수 있습니다.

### 연습 문제 - attention score 분해하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 5-10 minutes on this exercise.
> Having already done the previous exercises, this one should be easier.
> ```

분해된 score를 반환하는 함수를 구현하십시오 (`sqrt(d_head)`로 scale 하는 것을 잊지 마십시오!). 지금은 mask를 적용하지 마십시오.

<details>
<summary>질문 - 왜 attention pattern이 아니라 attention score에 집중하나요? (즉, softmax 이후가 아니라 이전 단계인 이유가 무엇인가요?)</summary>

분해 기법은 *오직* 선형적인 요소에만 작동하기 때문입니다. softmax는 선형이 아니므로, 더 이상 각 구성 요소를 독립적으로 고려할 수 없습니다.
</details>

<details>
<summary>도움말 - 우리가 무엇을 하고 있는지, 왜 하고 있는지 헷갈립니다.</summary>

우리의 각 구성 요소가 residual stream에 개별적으로 기록한다는 점을 기억하십시오. 따라서 layer 0 이후에 우리는 다음과 같은 상태가 됩니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/components.png" width="650">

우리는 특히 head `1.4`에서 계산된 attention score와, 그것이 해당 head의 입력값들에 어떻게 의존하는지에 관심이 있습니다. 우리는 이미 residual stream 값 $x$을 $x^{11}$(단순화를 위해 $y_0, ..., y_{13}$으로 표시함)를 통해 $e$, $pe$, $x^ 0$ 항들로 분해했으며, key와 query 항들에 대해서도 동일한 작업을 수행했습니다. 이러한 항들이 head `1.4`로 전달되는 모습은 다음과 같이 그려볼 수 있습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/components-2.png" width="650">

따라서 `attn_scores`를 완전히 전개하면, `(query_component, key_component)`의 각 조합에 대해 하나씩, 총 $14^2 = 196$개 항의 합이 됩니다.

---

#### 이 분해가 왜 유용한가요?

우리는 모델의 특정 circuit에 대한 가설을 가지고 있습니다. 우리는 head `1.4`가 induction head라고 생각하며, 이 head로 들어오는 가장 중요한 구성 요소는 prev token head `0.7` (key로서)와 token embedding (query로서)라고 생각합니다. 이는 이미 위에서 확인한 magnitude plot의 증거로 뒷받침됩니다 (`0.7`가 key이고 token embedding이 query일 때 값이 컸기 때문입니다). 하지만 우리는 여전히 이 특정 key와 query가 어떻게 **함께** 작동하는지 알지 못하며, 지금까지는 각각 따로 살펴봤을 뿐입니다.

`attn_scores`를 이렇게 분해함으로써, 조합 `(query=tok_emb, key=0.7)`의 기여도가 우리가 관찰한 특징적인 induction head pattern을 실제로 만들어내고 있는지(그리고 나머지 195개 항은 실제로 중요하지 않은지) 확인할 수 있습니다.
</details>

In [ ]:
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
    model: HookedTransformer,
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    """
    Output is decomposed_scores with shape [query_component, key_component, query_pos, key_pos]

    The [i, j, 0, 0]th element is y_i @ W_QK @ y_j^T (so the sum along both first axes are the
    attention scores)
    """
    raise NotImplementedError()


tests.test_decompose_attn_scores(decompose_attn_scores, decomposed_q, decomposed_k, model)

<details><summary>솔루션</summary>

```python
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
    model: HookedTransformer,
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    """
    Output is decomposed_scores with shape [query_component, key_component, query_pos, key_pos]

    The [i, j, 0, 0]th element is y_i @ W_QK @ y_j^T (so the sum along both first axes are the
    attention scores)
    """
    return einops.einsum(
        decomposed_q,
        decomposed_k,
        "q_comp q_pos d_head, k_comp k_pos d_head -> q_comp k_comp q_pos k_pos",
    ) / (model.cfg.d_head**0.5)
```
</details>

이 테스트들이 통과되면, 결과를 plot 할 수 있습니다:

In [ ]:
# First plot: attention score contribution from (query_component, key_component) = (Embed, L0H7), you can replace this
# with any other pair and see that the values are generally much smaller, i.e. this pair dominates the attention score
# calculation
decomposed_scores = decompose_attn_scores(decomposed_q, decomposed_k, model)

q_label = "Embed"
k_label = "0.7"
decomposed_scores_from_pair = decomposed_scores[component_labels.index(q_label), component_labels.index(k_label)]

imshow(
    utils.to_numpy(t.tril(decomposed_scores_from_pair)),
    title=f"Attention score contributions from query = {q_label}, key = {k_label}<br>(by query & key sequence positions)",
    width=700,
)


# Second plot: std dev over query and key positions, shown by component. This shows us that the other pairs of
# (query_component, key_component) are much less important, without us having to look at each one individually like we
# did in the first plot!
decomposed_stds = einops.reduce(
    decomposed_scores, "query_decomp key_decomp query_pos key_pos -> query_decomp key_decomp", t.std
)
imshow(
    utils.to_numpy(decomposed_stds),
    labels={"x": "Key Component", "y": "Query Component"},
    title="Std dev of attn score contributions across sequence positions<br>(by query & key comp)",
    x=component_labels,
    y=component_labels,
    width=700,
)

<details>
<summary>도움말 - 이 그래프들의 해석이 이해되지 않습니다.</summary>

첫 번째 그래프는 $e W_{QK}^{1.4} (x^{0.7})^T$ 항(즉, query는 token embedding에서 제공되고 key는 head `0.7`의 출력에서 제공되는 head `1.4`의 attention score 성분)이 induction head에서 볼 수 있는 특징적인 attention pattern인 강한 대각선 줄무늬를 생성한다는 것을 알려줍니다.

비록 이것이 이 성분이 induction 메커니즘을 구현하는 데 아마도 충분할 것이라는 점을 알려주지만, 이것이 전체 이야기는 아닙니다. 이상적으로는 나머지 195개의 항이 중요하지 않다는 것을 보여주고 싶을 것입니다. 특정 성분 쌍의 attention score에 대해 표준 편차를 구하는 것은 이 항이 전체 attention pattern에서 얼마나 중요한지를 나타내는 괜찮은 대리 지표가 됩니다. 두 번째 그래프는 다른 모든 성분에 대해 표준 편차가 매우 작다는 것을 보여주며, 따라서 다른 성분들이 중요하지 않다고 확신할 수 있습니다.

요약하자면 다음과 같습니다:

* 첫 번째 그래프는 쌍 `(q_component=tok_emb, k_component=0.7)`이 attention head `1.4`에서 보이는 특징적인 induction-head pattern을 생성한다는 것을 알려줍니다.
* 두 번째 그래프는 이 쌍이 `1.4`의 attention pattern에 영향을 주는 유일하게 중요한 쌍임을 확인시켜 줍니다. 다른 모든 쌍의 기여도는 매우 작습니다.
</details>

위와 같은 그래프들은 종종 중요한 정보의 요약을 제시하는 가장 간결한 방법이며, 무엇을 그래프로 그릴지 이해하는 것은 모든 모델 내부 분석 기반 작업에서 가치 있는 기술이라는 점에 유의하십시오. 하지만 위 두 그래프가 어떤 의미에서 단순화된 형태인 "전체 그래프"를 보고 싶다면, 아래 코드를 실행하여 모든 단일 성분 쌍이 attention score에 기여하는 행렬을 확인할 수 있습니다. 따라서 위의 첫 번째 그래프는 아래 전체 그래프의 한 단면(slice)일 뿐이며, 위의 두 번째 그래프는 아래 그래프의 각 단면을 표준 편차 연산으로 축소한 결과입니다.

(참고 - 아래에서 생성할 그래프는 크기가 상당히 크므로, 사용 후에는 이를 지우는 것이 좋습니다. 렌더링 중에 컴퓨터가 여전히 느리게 작동한다면, `fig.show(config={"staticPlot": True})`을 사용하여 비대화형 버전을 표시할 수 있습니다.)

In [ ]:
decomposed_scores_centered = t.tril(decomposed_scores - decomposed_scores.mean(dim=-1, keepdim=True))

decomposed_scores_reshaped = einops.rearrange(
    decomposed_scores_centered,
    "q_comp k_comp q_token k_token -> (q_comp q_token) (k_comp k_token)",
)

fig = imshow(
    decomposed_scores_reshaped,
    title="Attention score contributions from all pairs of (key, query) components",
    width=1200,
    height=1200,
    return_fig=True,
)
full_seq_len = seq_len * 2 + 1
for i in range(0, full_seq_len * len(component_labels), full_seq_len):
    fig.add_hline(y=i, line_color="black", line_width=1)
    fig.add_vline(x=i, line_color="black", line_width=1)

fig.show(config={"staticPlot": True})

### 전체 circuit 해석하기

이제 head `1.4`가 K composition을 통해 head `0.7`와 결합한다는 것을 알았으므로, 이를 곱하여 전체 circuit을 생성할 수 있습니다:

$$
W_E\, W_{QK}^{1.4}\, (W_{OV}^{0.7})^T\, W_E^T
$$

그리고 이것이 identity임을 확인합니다. (참고로, 여기서 identity라고 말할 때는 다시 한번 logit 상의 분포로 생각하는 것이므로, 이는 "높은 대각 성분 값"을 의미하는 것으로 해석해야 하며, 이전의 `top_1_acc` 지표를 사용할 것입니다.)

#### 질문 - 왜 이것이 identity여야 합니까?

<details>
<summary>답변</summary>

이 행렬은 bilinear form입니다. 이 행렬의 대각 성분 $(A, A)$은 다음과 같습니다:

$$
A^T \, W_E\, W_{QK}^{1.4}\, W_{OV}^{0.7}\, W_E^T \, A = \underbrace{(A^T W_E W_Q^{1.4})}_{\text{query}} \underbrace{(A^T W_E W_{OV}^{0.7} W_K^{1.4})^T}_{\text{key}}
$$

직관적으로, query는 **"나는 $A$ 뒤에 오는 token을 찾고 있습니다"**라고 말하고 있으며, key는 **"나는 $A$ 뒤에 오는 token *입니다*"**라고 말하고 있습니다 ($A^T W_E W_{OV}^{0.7}$이 prev token head `0.7`에 의해 한 위치 앞으로 이동되는 벡터임을 상기하십시오).

이제, ($X \neq A$에 대한) 비대각 성분 $(A, X)$을 고려해 보십시오. key가 query와 일치하지 않기 때문에 이 값들은 작을 것으로 예상합니다:

$$
A^T \, W_E\, W_{QK}^{1.4}\, W_{OV}^{0.7}\, W_E^T \, X = \underbrace{(\text{I'm looking for a token which followed A})}_\text{query} \boldsymbol{\cdot} \underbrace{(\text{I am a token which followed X})}_{\text{key}}
$$

따라서, 우리는 이것이 identity가 될 것으로 예상합니다.

설명 그림:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described-K-last.png" width="700">

<!-- ![kcomp_diagram_described-K.png](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described-K.png) -->
</details>

### 연습 문제 - K-comp circuit 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 10-20 minutes on this exercise.
> ```

위의 행렬을 `FactoredMatrix` 객체로 계산하십시오.

<details>
<summary>FactoredMatrix 객체 간의 곱셈에 관한 참고 사항입니다.</summary>

`M1 = A1 @ B1`와 `M2 = A2 @ B2`이 factored matrices라면, `M = M1 @ M2`은 새로운 factored matrix를 반환합니다. 이는 다음과 같을 수 있습니다:

```python
FactoredMatrix(M1.AB @ M2.A, M2.B)
```

또는 다음과 같을 수 있습니다:

```python
FactoredMatrix(M1.A, M1.B @ M2.AB)
```

이 두 객체는 각각 $M = (A_1 B_1 A_2) (B_2)$ 및 $M = (A_1) (B_1 A_2 B_2)$ 분해(factorisations)에 대응합니다.

어느 것이 반환되는지는 hidden dimension의 크기에 따라 달라집니다. 예를 들어 `M1.mdim < M2.mdim`인 경우, 사용되는 분해 방식은 $M = A_1 B_1 (A_2 B_2)$이 됩니다.

이 두 분해 방식 모두 유효하며, 정확히 동일한 SVD 결과를 제공한다는 점을 기억하십시오. 한 방식을 다른 방식보다 선호하는 유일한 이유는 계산 효율성 때문입니다 (SVD를 찾는 것과 같은 연산의 계산 복잡도를 결정하므로, 더 작은 bottleneck dimension을 선호합니다).
</details>

In [ ]:
def find_K_comp_full_circuit(
    model: HookedTransformer, prev_token_head_index: int, ind_head_index: int
) -> FactoredMatrix:
    """
    Returns a (vocab, vocab)-size FactoredMatrix, with the first dimension being the query side
    (direct from token embeddings) and the second dimension being the key side (going via the
    previous token head).
    """
    raise NotImplementedError()


prev_token_head_index = 7
ind_head_index = 4
K_comp_circuit = find_K_comp_full_circuit(model, prev_token_head_index, ind_head_index)

tests.test_find_K_comp_full_circuit(find_K_comp_full_circuit, model)

print(f"Token frac where max-activating key = same token: {top_1_acc(K_comp_circuit.T):.4f}")

<details><summary>솔루션</summary>

```python
def find_K_comp_full_circuit(
    model: HookedTransformer, prev_token_head_index: int, ind_head_index: int
) -> FactoredMatrix:
    """
    Returns a (vocab, vocab)-size FactoredMatrix, with the first dimension being the query side
    (direct from token embeddings) and the second dimension being the key side (going via the
    previous token head).
    """
    W_E = model.W_E
    W_Q = model.W_Q[1, ind_head_index]
    W_K = model.W_K[1, ind_head_index]
    W_O = model.W_O[0, prev_token_head_index]
    W_V = model.W_V[0, prev_token_head_index]

    Q = W_E @ W_Q
    K = W_E @ W_V @ W_O @ W_K
    return FactoredMatrix(Q, K.T)
```
</details>

다른 induction head `ind_head_index=10`에 대해서도 이를 시도해 볼 수 있으며, 이 경우에도 상대적으로 높은 결과가 나올 것입니다. head `1.4` 보다 더 높게 나오나요?

<details>
<summary>참고 - 지난번과 달리, head <code>1.4</code>와 <code>1.10</code>의 weight matrix들을 더해서 형성되는 "effective circuit"을 고려하는 것은 의미가 없습니다. 그 이유를 알 수 있나요?</summary>

여기서 다루고 있는 weight matrix들은 OV circuit이 아니라 QK circuit의 것이기 때문입니다. 이들은 선형적인 방식으로 결합되지 않으며, 대신 각 head의 QK-circuit 출력에 대해 개별적으로 softmax를 취합니다.
</details>

## Induction Circuit의 추가 탐색

이제 feature를 해석하는 방법과 weight로부터 circuit을 역공학하는 방법 모두를 통해 induction circuit을 완전히 역공학했다고 생각합니다. 하지만 네트워크에서 circuit을 찾기 위해 적용할 수 있는 더 많은 아이디어들이 있으며, 이를 induction head로 연습하는 것은 매우 흥미롭습니다. 따라서 몇 가지 보너스 내용을 준비했습니다. 뒤쪽의 보너스 아이디어로 바로 건너뛰셔도 좋습니다.

### Composition scores

논문에서 특히 멋진 아이디어는 [virtual weights](https://transformer-circuits.pub/2021/framework/index.html#residual-comms), 즉 compositional scores에 관한 아이디어입니다. (비록 제가 생각해낸 것이라 매우 편향되어 있긴 합니다!). 이는 [to identify induction heads](https://transformer-circuits.pub/2021/framework/index.html#analyzing-a-two-layer-model)에 사용됩니다.

compositional scores의 핵심 아이디어는 residual stream이 매우 큰 공간이며, 각 head는 작은 subspace에서 읽고 쓴다는 점입니다. 기본적으로 임의의 두 head는 그들의 subspace 사이에 겹치는 부분이 거의 없을 것입니다 (큰 벡터 공간에서 임의의 두 벡터의 dot product가 거의 0인 것과 같은 방식입니다). 하지만 두 head가 의도적으로 composition을 수행한다면, 정보 손실을 최소화하기 위해 유사한 subspace에서 쓰고 읽으려 할 가능성이 높습니다. 결과적으로, 이전 head의 출력 공간과 이후 head의 K, Q 또는 V 입력 공간 사이에 "얼마나 많은 겹침이 있는지"를 직접 살펴볼 수 있습니다.

우리는 **출력 공간(output space)**을 $W_{OV}=W_V W_O$로 표현합니다. 이러한 행렬들을 $W_A$라고 부릅니다.

우리는 **입력 공간(input space)**을 (이후 head의) $W_{QK}=W_Q W_K^T$ (Q-composition의 경우), $W_{QK}^T=W_K  W_Q^T$ (K-composition의 경우), 또는 $W_{OV}=W_V W_O$ (V-composition의 경우)로 표현합니다. 이러한 행렬들을 $W_B$라고 부릅니다 ($W_B$는 이후 head를, $W_A$는 이전 head를 가리키도록 이 표기법을 사용했습니다).

<details>
<summary>도움말 - 이러한 정의들이 왜 필요한지 이해가 가지 않습니다.</summary>

각 head가 세 개의 입력 와이어(keys, queries, values)와 하나의 출력 와이어(outputs)를 가지고 있다고 볼 수 있음을 기억하십시오. composition의 다양한 형태는 keys, queries, values 모두가 다른 head의 출력으로부터 공급될 수 있다는 사실에서 기인합니다.

다음은 세 가지 서로 다른 경우를 보여주는 그림이며, 왜 이러한 용어를 사용하는지도 설명해 줄 것입니다. 명확하게 보려면 새 탭에서 열어야 할 수도 있습니다.

![composition](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/composition_new.png)

</details>

겹침(overlap)을 어떻게 공식화할까요? 이는 기본적으로 열린 문제이지만, 놀랍게도 좋은 지표는 $\frac{\|W_AW_B\|_F}{\|W_B\|_F\|W_A\|_F}$이며, 여기서 $\|W\|_F=\sqrt{\sum_{i,j}W_{i,j}^2}$는 모든 요소의 제곱 합의 제곱근인 Frobenius norm입니다. (이것이 왜 좋은 지표인지 너무 궁금하시다면, 아래 연습 문제 바로 다음 섹션으로 건너뛰셔도 됩니다.)

### 연습 문제 - composition score 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than 15-25 minutes on these exercises.
> Writing a composition score function should be fairly easy. The harder part is getting the right weight matrices in the exercises that come after.
> ```

K, Q, V composition 각각에 대해 layer 0과 layer 1의 모든 head 쌍에 대한 이 metric을 계산하고 이를 plot 해보겠습니다.

먼저 일반적인 tensor를 사용하여 이를 구현해 보겠습니다 (나중에 `FactoredMatrix` 클래스를 사용하여 이를 어떻게 가속화할 수 있는지 살펴보겠습니다). 또한 아직은 계산의 batching에 대해 걱정하지 않고, 한 번에 하나의 행렬씩 처리하겠습니다.

Q, K, V composition 각각의 composition score를 저장하기 위해 `q_comp_scores` 등의 tensor를 제공했습니다 (즉, `q_comp_scores`의 `[i, j]`번째 요소는 layer 0의 `i`번째 head의 출력과 layer 1의 `j`번째 head의 입력 사이의 Q-composition score입니다). `get_comp_score` 함수를 완성한 다음, 이 tensor들을 각각 채워 넣으시면 됩니다.

In [ ]:
def get_comp_score(W_A: Float[Tensor, "in_A out_A"], W_B: Float[Tensor, "out_A out_B"]) -> float:
    """
    Return the composition score between W_A and W_B.
    """
    raise NotImplementedError()


tests.test_get_comp_score(get_comp_score)

<details><summary>솔루션</summary>

```python
def get_comp_score(W_A: Float[Tensor, "in_A out_A"], W_B: Float[Tensor, "out_A out_B"]) -> float:
    """
    Return the composition score between W_A and W_B.
    """
    W_A_norm = W_A.pow(2).sum().sqrt()
    W_B_norm = W_B.pow(2).sum().sqrt()
    W_AB_norm = (W_A @ W_B).pow(2).sum().sqrt()

    return (W_AB_norm / (W_A_norm * W_B_norm)).item()
```
</details>

테스트를 통과했다면, 모든 composition 점수를 채울 수 있습니다. 여기서는 각 composition 유형에 대해 layer 0의 모든 가능한 `W_A` 쌍과 layer 1의 `W_B` 쌍을 반복하는 for 루프를 사용하면 됩니다. 나중에는 이 계산을 배치(batch)로 처리하는 방법을 살펴보겠습니다.

In [ ]:
# Get all QK and OV matrices
W_QK = model.W_Q @ model.W_K.transpose(-1, -2)
W_OV = model.W_V @ model.W_O

# Define tensors to hold the composition scores
composition_scores = {
    "Q": t.zeros(model.cfg.n_heads, model.cfg.n_heads).to(device),
    "K": t.zeros(model.cfg.n_heads, model.cfg.n_heads).to(device),
    "V": t.zeros(model.cfg.n_heads, model.cfg.n_heads).to(device),
}

# YOUR CODE HERE - fill in values of the `composition_scores` dict, using `get_comp_score`

# Plot the composition scores
for comp_type in ["Q", "K", "V"]:
    plot_comp_scores(model, composition_scores[comp_type], f"{comp_type} Composition Scores")

<details><summary>솔루션</summary>

```python
for i in tqdm(range(model.cfg.n_heads)):
    for j in range(model.cfg.n_heads):
        composition_scores["Q"][i, j] = get_comp_score(W_OV[0, i], W_QK[1, j])
        composition_scores["K"][i, j] = get_comp_score(W_OV[0, i], W_QK[1, j].T)
        composition_scores["V"][i, j] = get_comp_score(W_OV[0, i], W_OV[1, j])
```
</details>

### 연습 문제 - 베이스라인 설정하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than ~10 minutes on this exercise.
> ```

위의 그래프들을 해석하기 위해서는 베이스라인이 필요합니다! 좋은 베이스라인은 초기화 시점의 score가 어떤 모습인지 확인하는 것입니다. composition score를 200번 무작위로 생성하여 이를 시도하는 함수를 만들어 보십시오. 2개의 `[d_model, d_model]` 행렬이 아니라, 4개의 `[d_head, d_model]` 행렬을 생성해야 함을 기억하십시오! 이 모델은 **Kaiming Uniform Initialisation**으로 초기화되었습니다:

```python
W = t.empty(shape)
nn.init.kaiming_uniform_(W, a=np.sqrt(5))
```

(이상적으로는 batching을 포함하여 더 효율적인 생성 방식과 더 많은 샘플을 사용하겠지만, 지금은 그 부분에 대해 걱정하지 않겠습니다.)

In [ ]:
def generate_single_random_comp_score() -> float:
    """
    Write a function which generates a single composition score for random matrices
    """
    raise NotImplementedError()


n_samples = 300
comp_scores_baseline = np.zeros(n_samples)
for i in tqdm(range(n_samples)):
    comp_scores_baseline[i] = generate_single_random_comp_score()

print("\nMean:", comp_scores_baseline.mean())
print("Std:", comp_scores_baseline.std())

hist(
    comp_scores_baseline,
    nbins=50,
    width=800,
    labels={"x": "Composition score"},
    title="Random composition scores",
)

<details><summary>솔루션</summary>

```python
def generate_single_random_comp_score() -> float:
    """
    Write a function which generates a single composition score for random matrices
    """
    W_A_left = t.empty(model.cfg.d_model, model.cfg.d_head)
    W_B_left = t.empty(model.cfg.d_model, model.cfg.d_head)
    W_A_right = t.empty(model.cfg.d_model, model.cfg.d_head)
    W_B_right = t.empty(model.cfg.d_model, model.cfg.d_head)

    for W in [W_A_left, W_B_left, W_A_right, W_B_right]:
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))

    W_A = W_A_left @ W_A_right.T
    W_B = W_B_left @ W_B_right.T

    return get_comp_score(W_A, W_B)
```
</details>

위의 그래프들을 이 baseline을 흰색으로 설정하여 다시 그릴 수 있습니다. 이 그래프에서 흥미로운 점들을 찾아보십시오!

In [ ]:
baseline = comp_scores_baseline.mean()
for comp_type, comp_scores in composition_scores.items():
    plot_comp_scores(model, comp_scores, f"{comp_type} Composition Scores", baseline=baseline)

<details>
<summary>관찰할 만한 몇 가지 흥미로운 점들입니다:</summary>

(지금까지 수행한 모든 분석의 맥락에서 고려했을 때) 가장 눈에 띄는 점은 K-composition 점수입니다. `0.7` (prev token head)는 `1.4` 및 `1.10` (두 개의 attention head)와 강하게 compose하고 있습니다. 이는 우리가 예상한 결과이며, composition 점수가 의도한 대로 작동하고 있다는 좋은 신호입니다.

또 다른 흥미로운 점은 layer 0의 다른 모든 head와 head `1.4` 및 `1.10` 사이의 V-composition 점수가 매우 낮다는 것입니다. induction circuit의 맥락에서 이는 긍정적인 현상입니다. induction head의 OV circuit은 layer-0 head의 출력이 아니라 **embedding**에서 작동해야 하기 때문입니다. (만약 반복되는 시퀀스가 `A B ... A B`이라면, 두 번째 `A`가 첫 번째 `B`에 attend하도록 만드는 것이 QK circuit의 역할이며, 해당 위치의 residual vector를 **embedding space**로 투영하여 `B` 정보를 추출하는 것이 OV circuit의 역할입니다. 이때 layer 0의 head들이 해당 위치에 기록한 다른 정보들은 무시되기를 기대합니다). 따라서 이 또한 composition 점수가 잘 작동하고 있다는 좋은 징후입니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/small_comp_diagram_last.png" width="900">

</details>

#### 이론 + 효율적인 구현

그렇다면, 이 metric에는 어떤 의미가 있을까요? 핵심은 squared Frobenius norm이 singular value들의 제곱 합과 같다는 간단한 선형 대수학 결과입니다.

<details>
<summary>증명</summary>

세 가지 서로 다른 증명을 제시하겠습니다:

---

##### 증명의 짧은 스케치

$M$가 diagonal일 때 $\|M\|_F^2$이 squared singular values의 합과 같음은 명확합니다. $M$에 orthogonal matrix를 곱해도 singular values는 변하지 않으므로 ($U$과 $V$ 행렬만 변하고 $S$은 변하지 않습니다), $M$에 orthogonal matrix를 곱했을 때 Frobenius norm 또한 변하지 않는다는 것만 보여주면 됩니다. 이는 Frobenius norm이 $M$의 열 벡터들의 squared $l_2$ norms의 합이라는 사실과, orthogonal matrices가 $l_2$ norms를 보존한다는 사실로부터 도출됩니다. (만약 $M$에 orthogonal matrix를 우측 곱셈한다면, 이를 $M$의 행 벡터들에 orthogonal operations를 수행하는 것으로 볼 수 있으며, 동일한 논리가 적용됩니다.)

---

##### 긴 증명

$$
\begin{aligned}
\|M\|_F^2 &= \sum_{ij}M_{ij}^2 \\
&= \sum_{ij}((USV^T)_{ij})^2 \\
&= \sum_{ij}\bigg(\sum_k U_{ik}S_{kk}V_{jk}\bigg)^2 \\
&= \sum_{ijk_1 k_2}S_{k_1 k_1} S_{k_2 k_2} U_{i k_1} U_{i k_2} V_{j k_2} V_{j k_2} \\
&= \sum_{k_1 k_2}S_{k_1 k_1} S_{k_2 k_2} \bigg(\sum_i U_{i k_1} U_{i k_2}\bigg)\bigg(\sum_j V_{j k_2} V_{j k_2}\bigg) \\
\end{aligned}
$$

큰 괄호 안의 각 항은 실제로는 각각 $U$과 $V$ 열들의 dot product입니다. 이들은 orthogonal matrices이므로, 이 항들은 $k_1=k_2$일 때 1이 되고 그 외에는 0이 됩니다. 따라서 다음과 같은 결과가 남습니다:

$$
\|M\|_F^2 = \sum_{k}S_{k k}^2
$$

---

##### squared Frobenius norm $|M|^2$이 $MM^T$의 trace와 같다는 사실을 이용한 간단한 증명

$$
\|M\|_F^2 = \text{Tr}(MM^T) = \text{Tr}(USV^TVSU^T) = \text{Tr}(US^2U^T) = \text{Tr}(S^2 U^T U) = \text{Tr}(S^2) = \|S\|_F^2
$$

여기서 우리는 trace의 cyclicity와, $U$이 orthogonal이므로 $U^TU=I$라는 사실($V$에 대해서도 동일함)을 사용했습니다. 마지막으로 $\|S\|_F^2$이 정확히 squared singular values의 합임을 관찰함으로써 증명을 마칩니다.
</details>

따라서 $W_A=U_AS_AV_A^T$, $W_B=U_BS_BV_B^T$ 이라면 $\|W_A\|_F=\|S_A\|_F$, $\|W_B\|_F=\|S_B\|_F$ 그리고 $\|W_AW_B\|_F=\|S_AV_A^TU_BS_B\|_F$ 입니다. 어떤 의미에서 $V_A^TU_B$는 쓰여진 subspace와 읽어온 subspace가 얼마나 정렬되어 있는지를 나타내며, $S_A$ 및 $S_B$ 항은 해당 subspace들의 중요도에 따라 가중치를 부여합니다.

<details>
<summary>이 설명이 여전히 혼란스럽다면 여기를 클릭하십시오.</summary>

$U_B$은 `[d_model, d_head]` 모양의 행렬입니다. 이는 **읽어오는 subspace**를 나타냅니다. 즉, 이후의 head는 이 행렬의 `d_head`개 열(column)에 residual stream을 투영함으로써 정보를 읽어옵니다.

$V_A$는 `[d_model, d_head]` 모양의 행렬입니다. 이는 **쓰여지는 subspace**를 나타냅니다. 즉, 이전 head가 residual stream에 쓰는 내용은 $V_A$의 `d_head`개 열 벡터들의 선형 결합입니다.

$V_A^T U_B$은 `[d_head, d_head]` 모양의 행렬입니다. 이 행렬의 각 원소는 길이가 `d_model`인 두 벡터의 내적(dot product)으로 형성됩니다:

* $V_A$의 열인 $v_i^A$ (이전 head가 residual stream으로 embedding하는 벡터 중 하나)
* $U_B$의 열인 $u_j^B$ (이후 head가 residual stream을 투영하는 벡터 중 하나)

$S_A$의 singular value를 $\sigma_1^A, ..., \sigma_k^A$라 하고, $S_B$에 대해서도 마찬가지라고 합시다. 그러면 다음과 같습니다:

$$
\|S_A V_A^T U_B S_B\|_F^2 = \sum_{i,j=1}^k (\sigma_i^A \sigma_j^B)^2 \|v^A_i \cdot u^B_j\|_F^2
$$

이는 $V_A$와 $U_B$의 열들(즉, 이전 head의 output 방향과 이후 head의 input 방향) 사이의 squared cosine similarity의 가중 합입니다. 이 합의 가중치는 $S_A$과 $S_B$ 모두의 singular value에 의해 결정됩니다. 즉, $v^A_i$이 중요한 output 방향이고, **동시에** $u_B^i$가 중요한 input 방향이라면, 이 두 방향이 서로 정렬되었을 때 composition score가 훨씬 더 높아집니다.

---

직관을 키우기 위해, 몇 가지 극단적인 예시를 생각해보겠습니다.

* 쓰여지는 공간과 읽어오는 공간 사이에 겹치는 부분이 없다면, 모든 $v_i^A \cdot u_j^B$이 0이 되므로 $V_A^T U_B$는 0으로 채워진 행렬이 됩니다. 이는 composition score가 0이 됨을 의미합니다.
* 완벽하게 겹치는 경우, 즉 $v_i^A$ 벡터들과 $u_j^B$ 벡터들의 span이 같다면, composition score는 커집니다. 가장 중요한 input 방향과 가장 중요한 output 방향이 일치할 때(즉, singular value $\sigma_i^A$와 $\sigma_j^B$의 순서가 같을 때) 이 값은 최대가 됩니다.
* 만약 행렬 $W_A$과 $W_B$가 단순히 rank 1이라면 (즉, $W_A = \sigma_A u_A v_A^T$ 및 $W_B = \sigma_B u_B v_B^T$), composition score는 $|v_A^T u_B|$가 됩니다. 다시 말해, 이는 $W_A$의 단일 output 방향과 $W_B$의 단일 input 방향 사이의 cosine similarity와 같습니다.
</details>

### 연습 문제 - batching 및 `FactoredMatrix` 클래스 사용하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> This exercise is optional, and not a super important part of this section conceptually.
> It's also quite messy to rearrange our tensors in the right way! You are invited to skip it if you want.
> ```

우리는 또한 이 통찰을 사용하여 composition score를 계산하는 더 효율적인 방법을 작성할 수 있습니다. 이는 분석을 대규모로 수행하려는 경우 매우 유용합니다! 핵심은 우리의 행렬이 low rank factorisation을 가지고 있다는 점이며, 두 차원이 모두 큰 행렬보다 좁은 행렬의 SVD를 계산하는 것이 훨씬 저렴하다는 것입니다. [algorithm described at the end of the paper](https://transformer-circuits.pub/2021/framework/index.html#induction-heads:~:text=Working%20with%20Low%2DRank%20Matrices) (SVD 검색)를 참고하십시오.

따라서 우리는 `FactoredMatrix` 클래스를 사용할 수 있습니다. 이 클래스는 Frobenium norm을 반환하는 `.norm()` 메서드도 제공합니다. 또한 지금이 batching을 다시 도입하기 좋은 기회입니다. 이는 분석 중에 때때로 유용하게 사용될 것입니다. 아래 함수에서 `W_As`과 `W_Bs`는 모두 2D 초과 factored matrices입니다 (예를 들어, 특정 레이어의 모든 head에 대한 OV circuit 또는 여러 레이어에 걸친 circuit을 나타낼 수 있습니다). 그리고 함수의 출력은 `(W_As, W_Bs)` 2D 초과 tensor 내의 각 행렬 쌍 `(W_A, W_B)`에 대한 composition score tensor여야 합니다.

In [ ]:
def get_batched_comp_scores(W_As: FactoredMatrix, W_Bs: FactoredMatrix) -> Tensor:
    """
    Computes the compositional scores from indexed factored matrices W_As and W_Bs.

    Each of W_As and W_Bs is a FactoredMatrix object which is indexed by all but its last 2
    dimensions, i.e.:
        W_As.shape == (*A_idx, A_in, A_out)
        W_Bs.shape == (*B_idx, B_in, B_out)
        A_out == B_in

    Return: tensor of shape (*A_idx, *B_idx) where the [*a_idx, *b_idx]th element is the
    compositional score from W_As[*a_idx] to W_Bs[*b_idx].
    """
    raise NotImplementedError()


W_QK = FactoredMatrix(model.W_Q, model.W_K.transpose(-1, -2))
W_OV = FactoredMatrix(model.W_V, model.W_O)

composition_scores_batched = dict()
composition_scores_batched["Q"] = get_batched_comp_scores(W_OV[0], W_QK[1])
composition_scores_batched["K"] = get_batched_comp_scores(
    W_OV[0], W_QK[1].T
)  # Factored matrix: .T is interpreted as transpose of the last two axes
composition_scores_batched["V"] = get_batched_comp_scores(W_OV[0], W_OV[1])

t.testing.assert_close(composition_scores_batched["Q"], composition_scores["Q"])
t.testing.assert_close(composition_scores_batched["K"], composition_scores["K"])
t.testing.assert_close(composition_scores_batched["V"], composition_scores["V"])
print("Tests passed - your `get_batched_comp_scores` function is working!")

<details>
<summary>힌트</summary>

`W_As`의 shape이 `(A1, A2, ..., Am, A_in, A_out)`이고 `W_Bs`의 shape이 `(B1, B2, ..., Bn, B_in, B_out)`(여기서 `A_out == B_in`)이라고 가정해 보겠습니다.

다음과 같이 이 두 tensor를 reshape하는 것이 도움이 될 것입니다:

```python
W_As.shape == (A1*A2*...*Am, 1, A_in, A_out)
W_Bs.shape == (1, B1*B2*...*Bn, B_in, B_out)
```

그렇게 하면 `W_As @ W_Bs`로 두 tensor를 곱할 수 있기 때문입니다 (broadcasting이 이 과정을 처리해 줍니다!).

reshape를 수행하는 가장 쉬운 방법은 `W_As.A`과 `W_As.B`을 reshape하고, 이 reshape된 tensor들로부터 새로운 `FactoredMatrix`를 정의하는 것입니다 (`W_Bs`에 대해서도 동일하게 적용합니다).
</details>


<details><summary>솔루션</summary>

```python
def get_batched_comp_scores(W_As: FactoredMatrix, W_Bs: FactoredMatrix) -> Tensor:
    """
    Computes the compositional scores from indexed factored matrices W_As and W_Bs.

    Each of W_As and W_Bs is a FactoredMatrix object which is indexed by all but its last 2
    dimensions, i.e.:
        W_As.shape == (*A_idx, A_in, A_out)
        W_Bs.shape == (*B_idx, B_in, B_out)
        A_out == B_in

    Return: tensor of shape (*A_idx, *B_idx) where the [*a_idx, *b_idx]th element is the
    compositional score from W_As[*a_idx] to W_Bs[*b_idx].
    """
    # Flatten W_As into (single_A_idx, 1, A_in, A_out)
    W_As = FactoredMatrix(
        W_As.A.reshape(-1, 1, *W_As.A.shape[-2:]),
        W_As.B.reshape(-1, 1, *W_As.B.shape[-2:]),
    )
    # Flatten W_Bs into (1, single_B_idx, B_in(=A_out), B_out)
    W_Bs = FactoredMatrix(
        W_Bs.A.reshape(1, -1, *W_Bs.A.shape[-2:]),
        W_Bs.B.reshape(1, -1, *W_Bs.B.shape[-2:]),
    )

    # Compute the product, with shape (single_A_idx, single_B_idx, A_in, B_out)
    W_ABs = W_As @ W_Bs

    # Compute the norms, and return the metric
    return W_ABs.norm() / (W_As.norm() * W_Bs.norm())
```
</details>

### Targeted Ablations

우리는 loss 대신 induction head의 attention pattern에 ablation이 미치는 영향을 살펴봄으로써, composition을 감지하기 위해 ablation 기법을 개선할 수 있습니다. 이를 구현해 보겠습니다!

주의하세요 - 기본적으로 `run_with_hooks`은 실행 시 기존의 모든 hook을 제거합니다. caching을 사용하고 싶다면 `reset_hooks_start` 플래그를 False로 설정하십시오.

In [ ]:
seq_len = 50


def ablation_induction_score(prev_head_index: int | None, ind_head_index: int) -> float:
    """
    Takes as input the index of the L0 head and the index of the L1 head, and then runs with the
    previous token head ablated and returns the induction score for the ind_head_index now.
    """

    def ablation_hook(v, hook):
        if prev_head_index is not None:
            v[:, :, prev_head_index] = 0.0
        return v

    def induction_pattern_hook(attn, hook):
        hook.ctx[prev_head_index] = attn[0, ind_head_index].diag(-(seq_len - 1)).mean()

    model.run_with_hooks(
        rep_tokens,
        fwd_hooks=[
            (utils.get_act_name("v", 0), ablation_hook),
            (utils.get_act_name("pattern", 1), induction_pattern_hook),
        ],
    )
    return model.blocks[1].attn.hook_pattern.ctx[prev_head_index].item()


baseline_induction_score = ablation_induction_score(None, 4)
print(f"Induction score for no ablations: {baseline_induction_score:.5f}\n")
for i in range(model.cfg.n_heads):
    new_induction_score = ablation_induction_score(i, 4)
    induction_score_change = new_induction_score - baseline_induction_score
    print(f"Ablation score change for head {i:02}: {induction_score_change:+.5f}")

<details>
<summary>질문 - 현재 얻고 있는 결과의 해석은 무엇입니까?</summary>

아무런 ablation을 수행하지 않았을 때의 induction score는 약 0.68이며, head 7을 제외한 대부분의 다른 head들은 ablation 시 induction score를 크게 변화시키지 않는다는 것을 발견하셨을 것입니다. head 7의 경우 induction score를 거의 0으로 감소시킵니다.

이는 head `0.7`이 이 induction circuit에서 prev token head라는 강력한 또 다른 증거입니다.
</details>

## 보너스

### 실제 LLM에서 Circuit 찾기

이러한 기술들의 특히 멋진 응용 사례는 거대 언어 모델에서 실제 circuit의 예시를 찾는 것입니다. 다행히도 `TransformerLens` 라이브러리에서 직접 다뤄볼 수 있는 많은 오픈 소스 모델들이 있습니다! 우리가 2L transformer에서 사용해 온 많은 기술들이 더 많은 레이어를 가진 모델들에도 그대로 적용됩니다.

이 라이브러리를 사용하면 이러한 모델들을 비교적 쉽게 다룰 수 있습니다. 과감하게 도전하여 흥미로운 circuit들을 찾아보시는 것을 추천합니다!

시도해 볼 만한 몇 가지 재미있는 활동들입니다:

- induction head 찾기 - 위에서 수행한 모든 단계를 반복해 보십시오. 동일한 알고리즘을 따르고 있습니까?
- 정보를 지우는 neuron 찾기
    - 즉, 입력 가중치와 출력 가중치 사이에 높은 음수 cosine similarity를 가진 경우입니다.
- position embedding 해석해 보기.

<details>
<summary>Positional Embedding 힌트</summary>

singular value decomposition `t.svd`을 살펴보고 position 공간에 대해 principal components를 플롯해 보십시오. 값이 높은 것들은 서로 다른 주파수의 sine 및 cosine 파형인 경향이 있습니다.
</details>

- 해석 가능한 attention pattern을 가진 head 찾기: 예를 들어, 서로 다른 언어로 된 텍스트가 주어졌을 때 동일한 단어(또는 다음 단어)에 attention을 주는 head, 가장 최근의 고유 명사, 가장 최근의 마침표, 또는 문장의 주어 등에 attention을 주는 head 등이 있습니다.
    - 특정 head를 선택해 ablate하고, 해당 head가 있을 때와 없을 때 많은 양의 텍스트로 모델을 실행해 보십시오. loss 차이가 가장 큰 token들을 찾아 해당 head가 무엇을 하고 있는지 해석해 보십시오.
- 간접 목적어 식별(indirect object identification)에 관한 Kevin의 연구 일부를 재현해 보십시오.
- [ROME paper](https://rome.baulab.info/)에서 영감을 받아, residual stream에 patching을 하는 causal tracing 기술을 사용해 보십시오 - 네트워크가 서로 다른 사실들에 어떻게 답하는지 분석할 수 있습니까?

참고: 저는 결과 transformer에 몇 가지 단순화를 적용했습니다. 이는 모델을 수학적으로 동일하게 유지하며 출력 log probs를 변경하지 않지만, 모델의 구조를 다소 변경하며 그중 하나의 변경 사항은 출력 logits를 상수로 이동시킵니다.

<details>
<summary>모델 단순화</summary>

#### Centering $W_U$

$W_U$의 출력은 softmax로 들어가는 $d_{vocab}$ 벡터(또는 이를 마지막 차원으로 갖는 tensor)입니다.

#### LayerNorm Folding

LayerNorm은 residual stream에서 읽어오는 linear layer의 시작 부분(예: query, key, value, mlp_in 또는 unembed 계산)에만 적용됩니다.

각 LayerNorm은 $LN:\mathbb{R}^n\to\mathbb{R}^n$의 함수 형태를 가집니다.
$LN(x)=s(x) * w_{ln} + b_{ln}$ 여기서 $*$는 element-wise 곱셈이며, $s(x)=\frac{x-\bar{x}}{|x-\bar{x}|}$과 $w_{ln},b_{ln}$은 모두 $\mathbb{R}^n$의 벡터입니다.

linear layer는 $l:\mathbb{R}^n\to\mathbb{R}^m$, $l(y)=Wy+b$의 형태를 가지며 여기서 $W\in \mathbb{R}^{m\times n},b\in \mathbb{R}^m,y\in\mathbb{R}^n$ 입니다.

따라서 $f(LN(x))=W(w_{ln} * s(x)+b_{ln})+b=(W * w_{ln})s(x)+(Wb_{ln}+b)=W_{eff}s(x)+b_{eff}$가 되며, 여기서 $W_{eff}$은 $W$와 $w_{ln}$의 elementwise product(elementwise 곱셈이 이와 같이 교환 가능하다는 것을 보여주는 것은 연습 문제로 남겨둡니다)와 $b_{eff}=Wb_{ln}+b\in \mathbb{R}^m$입니다.

interpretability 관점에서는 folding된 layer $W_{eff},b_{eff}$을 해석하는 것이 훨씬 더 좋습니다. 근본적으로 이것이 실제로 수행되는 계산이며, $W$이나 $w_{ln}$가 단독으로 의미가 있을 것이라고 기대할 이유는 없습니다.
</details>

### 자신만의 Toy 모델 학습시키기

재미있는 연습 중 하나는 induction head를 생성하는 최소한의 태스크, 즉 반복되는 부분 시퀀스가 포함된 랜덤 token 시퀀스에서 다음 token을 예측하는 모델을 학습시키는 것입니다. 작은 2L Attention-Only 모델로도 이를 구현할 수 있습니다.

<details>
<summary>팁</summary>

* 반복되는 위치를 반드시 랜덤하게 설정하십시오! 그렇지 않으면 모델이 단순히 고정된 위치에 attention을 주는 지루한 알고리즘을 학습할 수 있습니다.
* 반복되는 token에 대해서만 loss를 평가하면 태스크의 노이즈가 줄어들어 더 잘 작동합니다.
* 동일한 시퀀스가 한 번만 반복되는 것보다 여러 번 반복될 때 가장 효과적입니다.
* 올바르게 설정하고 유한한 데이터와 weight decay를 제공한다면, 모델이 grok하게 만들 수 있을 것입니다. 다만, 이를 위해서는 하이퍼파라미터 튜닝이 필요할 수 있습니다.
* 제가 이 작업을 수행했을 때, 각 head가 induction stripe의 1/3씩을 가지고 있으며 이들이 함께 모든 token을 커버하는 기이한 franken-induction head가 생성되었습니다.
* query와 key만 positional embedding에 접근할 수 있게 하면 더 잘 작동하겠지만, 어느 쪽으로든 작동은 할 것입니다.
</details>

### 학습 중 Induction Head 해석하기

induction head에 관한 특히 놀라운 결과는 이들이 일관되게 [form very abruptly in training as a phase change](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html#argument-phase-change) 하며, 매우 중요한 기능이기 때문에 [visible non-convex bump in the loss curve](https://wandb.ai/mechanistic-interpretability/attn-only/reports/loss_ewma-22-08-24-22-08-00---VmlldzoyNTI2MDM0?accessToken=r6v951q0e1l4q4o70wb2q67wopdyo3v69kz54siuw7lwb4jz6u732vo56h6dr7c2) 가 존재한다는 점입니다 (이 모델의 경우, 약 2B에서 4B token 사이입니다). 저는 이 모델의 여러 체크포인트를 가지고 있습니다. 중간 체크포인트들에 대해 induction head 탐지 기법들을 다시 실행해 보고 어떤 일이 일어나는지 확인해 보시기 바랍니다. (Wandb에서 수많은 300MB 체크포인트들을 효율적으로 전송할 좋은 아이디어가 있다면 보너스 점수를 드리겠습니다 ㅎㅎ)

### 추가 논의 / 조사

Anthropic은 induction heads에 대해 훨씬 더 깊이 있게 다룬 [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) 포스트를 작성했습니다. 이 포스트는 **induction heads가 대형 모델을 포함한 transformer 모델에서 in-context learning의 주요 원천이다**라는 가설에 대한 여섯 가지 증거를 중심으로 구성되어 있습니다. 간단히 요약하면 다음과 같습니다:

1. Transformers는 갑자기 in-context learning 능력이 훨씬 좋아지는 "상전이(phase change)"를 겪으며, 이는 induction heads가 나타나는 시점과 거의 일치합니다.
2. induction heads가 더 쉽게 형성되도록 transformer의 architecture를 변경하면, 그에 상응하는 in-context learning의 성능 향상이 나타납니다.
3. 런타임에 induction heads를 ablate하면, in-context learning 성능이 저하됩니다.
4. induction heads가 더 복잡한 in-context learning 알고리즘을 수행하는 구체적인 사례들이 있습니다 (나중에 이 중 하나인 **indirect object identification**을 조사할 기회가 있을 것입니다).
5. induction heads에 대한 mechanistic한 설명이 있으며, 이는 더 일반적인 형태의 in-context learning으로의 자연스러운 확장을 시사합니다.
6. in-context learning 관련 동작은 일반적으로 소형 모델과 대형 모델 사이에서 매끄럽게 연속적이며, 이는 기저 메커니즘 또한 동일함을 시사합니다.

여러분께 드리는 몇 가지 질문입니다:

* 이 증거들이 얼마나 설득력 있다고 생각하십니까? 파트너와 논의해 보십시오.
    * 어떤 점이 가장 설득력 있다고 생각하십니까?
    * 어떤 점이 가장 설득력이 부족하다고 생각하십니까?
    * 다른 증거들이 없더라도, 가설을 확신시키기에 충분한 부분 집합이 있습니까?
* 3번 항목에서, 논문은 induction heads를 ablate했을 때 in-context learning 성능이 저하됨을 관찰했습니다. 우리는 중복된 랜덤 시퀀스를 복사하는 모델의 능력을 테스트하여 이를 측정했지만, 논문에서는 **in-context learning score** (컨텍스트 내 500번째 token의 loss에서 50번째 token의 loss를 뺀 값)를 사용했습니다.
    * 이것이 왜 합리적인 metric인지 이해하시겠습니까?
    * 이 결과들을 재현할 수 있습니까 (우리가 사용해 온 2-layer 모델보다 더 큰 모델에서 시도해 보십시오)?
* 4번 항목(더 복잡한 형태의 in-context learning)에서, 논문은 `[A][B]...[A][B]` 보다는 `[A*][B*]...[A][B]`과 같은 패턴을 매칭하는 "fuzzy induction heads"로의 자연스러운 확장을 제안합니다 (여기서 `*`은 반드시 동일한 token일 필요는 없는, 일종의 언어적 유사성을 나타냅니다).
    * 이것이 어떤 형태를 띨 수 있을지, 즉 induction heads가 포착할 수 있는 유사성의 종류에는 무엇이 있을지 생각할 수 있습니까? 예시를 생성해 보시겠습니까?